# Amazon ML Challenge 2026 — Entity Resolution: PREDICT & SUBMIT

Uses the models trained by `01_train.ipynb` (same `WORK_DIR` and `RUN_NAME`) to produce `candidate_pairs.tsv` and `matching_results.tsv`, validates them, and builds the submission archive.


**How to run:** run the cells top to bottom. Section 1 writes the pipeline code into `WORK_DIR/src/`, one module
per cell. Section 2 is the only place you need to edit (data paths, column names, run name, mode).

**If the kernel dies or runs out of memory:** every stage saves its results to
`outputs/<RUN_NAME>/.../checkpoints/`. Restart the kernel, re-run the Setup and Config cells, and continue from
the first stage that did not finish. Nothing before it has to be recomputed. Each stage also prints the current
and peak memory (`RSS ... (peak ...)`), so you can see which one is tight.

**Memory:** `LOW_MEMORY = True` uses `configs/low_memory.yaml`, which sets a smaller candidate cap, smaller chunks,
fewer worker processes and negative down-sampling for LightGBM. The pipeline streams the pairwise features to a
disk-backed matrix, so they never need to fit in RAM all at once.

## 0. Setup

In [ ]:
import os, sys

# Everything (code, checkpoints, models, outputs) lives in WORK_DIR.
# On Colab, put it on Google Drive so a crashed / restarted runtime can resume:
#   from google.colab import drive; drive.mount("/content/drive")
#   WORK_DIR = "/content/drive/MyDrive/amazon_er"
if os.path.isdir("/kaggle/working"):
    WORK_DIR = "/kaggle/working/amazon_er"
elif os.path.isdir("/content"):
    WORK_DIR = "/content/amazon_er"
else:
    WORK_DIR = os.path.abspath("amazon_er_work")
WORK_DIR = os.environ.get("ER_WORK_DIR", WORK_DIR)   # (override used by the automated notebook test)

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
for d in ("src", "configs", "scripts"):
    os.makedirs(d, exist_ok=True)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print("working directory:", os.getcwd())

In [ ]:
import importlib.util, subprocess, sys

INSTALL_FULL_MODE_DEPS = False   # True for mode="full" (torch, transformers, faiss; GPU recommended)

core = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
        "lightgbm": "lightgbm", "rapidfuzz": "rapidfuzz>=3.6", "jellyfish": "jellyfish",
        "networkx": "networkx", "yaml": "pyyaml", "unidecode": "unidecode"}
full = {"torch": "torch", "transformers": "transformers", "tokenizers": "tokenizers",
        "sentencepiece": "sentencepiece", "google.protobuf": "protobuf", "faiss": "faiss-cpu"}
wanted = {**core, **(full if INSTALL_FULL_MODE_DEPS else {})}
missing = [pkg for mod, pkg in wanted.items() if importlib.util.find_spec(mod.split(".")[0]) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("all required packages are installed")

## 1. Pipeline code

Each cell below writes one module of the pipeline to `src/` (identical to the repository's `src/`). Run them all. If you edit one, restart the kernel before re-running the later cells.

In [ ]:
%%writefile src/__init__.py
"""Business entity resolution pipeline for the Amazon ML Challenge 2026.

Source 1 (clean reference) entities are matched against the noisy Source 2 and
Source 3 vendor records using only business names and addresses.
"""

__version__ = "1.0.0"


In [ ]:
%%writefile src/config.py
"""YAML configuration loading with `base:` inheritance and dotted overrides."""
from __future__ import annotations

import copy
import os
from typing import Any

import yaml


def deep_merge(base: dict, override: dict) -> dict:
    """Recursively merge ``override`` into a copy of ``base``."""
    out = copy.deepcopy(base)
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = deep_merge(out[key], value)
        else:
            out[key] = copy.deepcopy(value)
    return out


def _load_yaml(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as fh:
        data = yaml.safe_load(fh) or {}
    base = data.pop("base", None)
    if base:
        base_path = base if os.path.isabs(base) else os.path.join(os.path.dirname(path), base)
        data = deep_merge(_load_yaml(os.path.normpath(base_path)), data)
    return data


def _coerce(value: str) -> Any:
    return yaml.safe_load(value)


def apply_overrides(cfg: dict, overrides: list[str] | None) -> dict:
    """Apply ``key.sub=value`` overrides (value parsed as YAML)."""
    cfg = copy.deepcopy(cfg)
    for item in overrides or []:
        if "=" not in item:
            raise ValueError(f"Override '{item}' must look like key.subkey=value")
        key, value = item.split("=", 1)
        node = cfg
        parts = key.split(".")
        for part in parts[:-1]:
            node = node.setdefault(part, {})
        node[parts[-1]] = _coerce(value)
    return cfg


def load_config(path: str, overrides: list[str] | None = None) -> dict:
    cfg = _load_yaml(path)
    cfg = apply_overrides(cfg, overrides)
    cfg["_config_path"] = os.path.abspath(path)
    mode = cfg.get("mode", "baseline")
    if mode not in ("baseline", "full"):
        raise ValueError(f"mode must be 'baseline' or 'full', got {mode!r}")
    return cfg


def get(cfg: dict, dotted: str, default: Any = None) -> Any:
    node: Any = cfg
    for part in dotted.split("."):
        if not isinstance(node, dict) or part not in node:
            return default
        node = node[part]
    return node


def is_full(cfg: dict) -> bool:
    return cfg.get("mode", "baseline") == "full"


def semantic_enabled(cfg: dict) -> bool:
    flag = get(cfg, "blocking.methods.semantic.enabled", "auto")
    if flag == "auto":
        return is_full(cfg)
    return bool(flag)


def model_dir(cfg: dict) -> str:
    return os.path.join(get(cfg, "paths.models_dir", "models"), cfg.get("run_name", "default"))


def output_dir(cfg: dict) -> str:
    return os.path.join(get(cfg, "paths.outputs_dir", "outputs"), cfg.get("run_name", "default"))


In [ ]:
%%writefile src/utils.py
"""Logging, timing, seeding and serialisation helpers."""
from __future__ import annotations

import json
import logging
import os
import pickle
import random
import sys
import time
import zlib
from contextlib import contextmanager
from typing import Any

import numpy as np

LOGGER_NAME = "er"


def get_logger() -> logging.Logger:
    return logging.getLogger(LOGGER_NAME)


def setup_logging(log_file: str | None = None, level: int = logging.INFO) -> logging.Logger:
    logger = get_logger()
    logger.setLevel(level)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    if log_file:
        ensure_dir(os.path.dirname(log_file))
        fh = logging.FileHandler(log_file, mode="a", encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    logger.propagate = False
    return logger


def rss_mb() -> float:
    """Current resident memory of this process in MB (Linux /proc; else peak)."""
    try:
        with open("/proc/self/statm") as fh:
            return int(fh.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
    except (OSError, ValueError, AttributeError):
        return peak_rss_mb()


def peak_rss_mb() -> float:
    try:
        import resource
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak / 1024 if sys.platform != "darwin" else peak / 2**20
    except (ImportError, OSError):
        return float("nan")


def release_memory() -> None:
    """Garbage-collect and hand freed heap pages back to the OS (glibc
    malloc_trim). Without the trim a long-lived notebook kernel keeps the
    peak memory of earlier stages reserved."""
    import gc
    gc.collect()
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except (OSError, AttributeError):
        pass


def _proc_status_mb(field: str) -> float | None:
    try:
        with open("/proc/self/status") as fh:
            for line in fh:
                if line.startswith(field + ":"):
                    return int(line.split()[1]) / 1024
    except OSError:
        pass
    return None


def mem_str() -> str:
    """Memory report for the logs. On Linux, real process memory (anonymous)
    is shown separately from pages of memory-mapped files (the disk-backed
    feature matrix): the latter is page cache the OS can drop at any time and
    is not what causes out-of-memory kills."""
    anon, mapped = _proc_status_mb("RssAnon"), _proc_status_mb("RssFile")
    if anon is None:
        return f"RSS {rss_mb():,.0f} MB (peak {peak_rss_mb():,.0f} MB)"
    return f"mem {anon:,.0f} MB (+{mapped:,.0f} MB mapped files; peak total {peak_rss_mb():,.0f} MB)"


@contextmanager
def timer(name: str, store: dict | None = None):
    log = get_logger()
    log.info(">> %s ...  [%s]", name, mem_str())
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        log.info("<< %s done in %.1fs  [%s]", name, dt, mem_str())
        if store is not None:
            store[name] = round(dt, 3)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def ensure_dir(path: str) -> str:
    if path:
        os.makedirs(path, exist_ok=True)
    return path


def _json_default(obj: Any):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, set):
        return sorted(obj)
    return str(obj)


def save_json(obj: Any, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=_json_default, ensure_ascii=False)


def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


def save_pickle(obj: Any, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "wb") as fh:
        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)


def load_pickle(path: str) -> Any:
    with open(path, "rb") as fh:
        return pickle.load(fh)


def stable_hash(text: str) -> int:
    """Deterministic 32-bit hash (Python's hash() is salted per process)."""
    return zlib.crc32(text.encode("utf-8")) & 0xFFFFFFFF


def resolve_n_jobs(n_jobs: int | None) -> int:
    if n_jobs is None or n_jobs == 0:
        return 1
    if n_jobs < 0:
        return max(1, (os.cpu_count() or 1) + 1 + n_jobs)
    return int(n_jobs)


def describe_counts(values) -> dict:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return {"count": 0}
    return {
        "count": int(arr.size),
        "mean": float(arr.mean()),
        "min": float(arr.min()),
        "p25": float(np.percentile(arr, 25)),
        "median": float(np.median(arr)),
        "p75": float(np.percentile(arr, 75)),
        "p90": float(np.percentile(arr, 90)),
        "p99": float(np.percentile(arr, 99)),
        "max": float(arr.max()),
    }


In [ ]:
%%writefile src/schema.py
"""Column schema definitions and validation for the challenge TSV files.

The official transcript does not name files or columns, so every column name
is taken from the `schema:` section of the config. Validation errors list the
columns that ARE present so the config can be fixed quickly
(`python inspect_data.py` prints them as well).
"""
from __future__ import annotations

from dataclasses import dataclass, field

import pandas as pd

from .config import get


class SchemaError(ValueError):
    pass


@dataclass
class SourceSchema:
    source: int
    id_col: str
    name_col: str
    address_cols: list[str]
    address_join: str = ", "

    @property
    def required(self) -> list[str]:
        return [self.id_col, self.name_col, *self.address_cols]


@dataclass
class LabelSchema:
    source1_id: str
    match_columns: list[str]
    column_sources: dict[str, int] | None = None
    list_separator: str = ","
    missing_source1_policy: str = "singleton"
    ambiguous_id_policy: str = "skip"
    unknown_id_policy: str = "warn"

    @property
    def required(self) -> list[str]:
        return [self.source1_id, *self.match_columns]


@dataclass
class SubmissionSchema:
    id_column: str
    match_columns: list[str]
    column_sources: dict[str, int] | None
    list_separator: str = ","
    list_format: str = "plain"
    candidate_columns: dict = field(default_factory=dict)


def source_schema(cfg: dict, source: int) -> SourceSchema:
    node = get(cfg, f"schema.source{source}")
    if node is None:
        raise SchemaError(f"schema.source{source} missing from config")
    address = node.get("address", [])
    if isinstance(address, str):
        address = [address]
    return SourceSchema(
        source=source,
        id_col=node["id"],
        name_col=node["name"],
        address_cols=list(address),
        address_join=get(cfg, "schema.address_join", ", "),
    )


def label_schema(cfg: dict) -> LabelSchema:
    node = get(cfg, "schema.labels", {})
    cols = node.get("match_columns", ["matches"])
    if isinstance(cols, str):
        cols = [cols]
    col_sources = node.get("column_sources")
    if col_sources:
        col_sources = {str(k): int(v) for k, v in col_sources.items()}
    return LabelSchema(
        source1_id=node.get("source1_id", "source1_id"),
        match_columns=list(cols),
        column_sources=col_sources,
        list_separator=node.get("list_separator", ","),
        missing_source1_policy=node.get("missing_source1_policy", "singleton"),
        ambiguous_id_policy=node.get("ambiguous_id_policy", "skip"),
        unknown_id_policy=node.get("unknown_id_policy", "warn"),
    )


def submission_schema(cfg: dict, labels: LabelSchema, detected_list_format: str = "plain",
                      sample_header: list[str] | None = None) -> SubmissionSchema:
    """Output schema. Defaults mirror the label file ("This mirrors exactly
    what you'll submit" - transcript). A sample submission header wins."""
    node = get(cfg, "submission", {})
    id_col = node.get("id_column") or labels.source1_id
    match_cols = node.get("match_columns") or list(labels.match_columns)
    col_sources = labels.column_sources
    if sample_header:
        id_col = sample_header[0]
        new_cols = sample_header[1:]
        if len(new_cols) == len(match_cols) and col_sources:
            col_sources = {new: col_sources.get(old) for new, old in zip(new_cols, match_cols)}
        elif len(new_cols) != len(match_cols):
            col_sources = None
        match_cols = new_cols
    if isinstance(match_cols, str):
        match_cols = [match_cols]
    if len(match_cols) > 1 and not col_sources:
        # Heuristic only used when the per-source mapping is not configured.
        guess = {}
        for c in match_cols:
            if "2" in c and "3" not in c:
                guess[c] = 2
            elif "3" in c and "2" not in c:
                guess[c] = 3
        if len(guess) == len(match_cols) and sorted(guess.values()) == [2, 3]:
            col_sources = guess
        else:
            raise SchemaError(
                f"Several output match columns {match_cols} but no source mapping; "
                "set schema.labels.column_sources, e.g. {col_a: 2, col_b: 3}."
            )
    fmt = node.get("list_format", "auto")
    if fmt == "auto":
        fmt = detected_list_format
    return SubmissionSchema(
        id_column=id_col,
        match_columns=list(match_cols),
        column_sources=col_sources,
        list_separator=node.get("list_separator", labels.list_separator),
        list_format=fmt,
        candidate_columns=node.get("candidate_columns", {
            "source1_id": "source1_id", "candidate_id": "candidate_id",
            "candidate_source": "candidate_source"}),
    )


def validate_columns(df: pd.DataFrame, required: list[str], what: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise SchemaError(
            f"{what}: missing column(s) {missing}. Columns present: {list(df.columns)}. "
            "Fix the `schema:` section of the config (see `python inspect_data.py`)."
        )


In [ ]:
%%writefile src/data_io.py
"""TSV ingestion: loading, schema validation, null handling, duplicate
detection, Source 1/2/3 separation and label parsing.

Named ``data_io`` (not ``io``) so it can never shadow Python's stdlib ``io``.
Every read uses an explicit TAB separator, as the transcript requires.
"""
from __future__ import annotations

import csv
import os
import re

import pandas as pd

from .config import get
from .schema import (LabelSchema, SchemaError, SourceSchema, label_schema, source_schema,
                     validate_columns)
from .utils import get_logger

SEP = "\t"
_EMPTY_TOKENS = {"", "[]", "nan", "none", "null", "na", "n/a", "{}", "()"}


def read_tsv(path: str, cfg: dict | None = None) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"TSV file not found: {path}")
    cfg = cfg or {}
    encoding = get(cfg, "io.encoding", "utf-8-sig")
    quoting = csv.QUOTE_NONE if get(cfg, "io.quoting", "minimal") == "none" else csv.QUOTE_MINIMAL
    df = pd.read_csv(
        path,
        sep=SEP,
        dtype=str,
        keep_default_na=False,   # never turn "NA"/"null" business names into NaN
        na_values=[],
        quoting=quoting,
        encoding=encoding,
        engine="c",
    )
    df.columns = [str(c).strip() for c in df.columns]
    # Sanity check: parsed rows vs physical lines (detects quoting problems).
    with open(path, "rb") as fh:
        n_lines = sum(1 for line in fh if line.strip())
    if n_lines - 1 != len(df):
        get_logger().warning(
            "%s: %d data lines on disk but %d parsed rows. If names contain quote characters, "
            "try io.quoting: none (or minimal).", path, n_lines - 1, len(df))
    return df


def _clean_str(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()


def load_source(path: str, source: int, cfg: dict) -> tuple[pd.DataFrame, dict]:
    """Load one source file into columns [id, name, address] (+ original row order)."""
    log = get_logger()
    schema: SourceSchema = source_schema(cfg, source)
    raw = read_tsv(path, cfg)
    validate_columns(raw, schema.required, f"Source {source} ({path})")
    df = pd.DataFrame({
        "id": _clean_str(raw[schema.id_col]),
        "name": _clean_str(raw[schema.name_col]),
    })
    parts = [_clean_str(raw[c]) for c in schema.address_cols]
    if parts:
        joined = parts[0]
        for p in parts[1:]:
            joined = joined.str.cat(p, sep=schema.address_join)
        # remove empty components produced by missing columns: "a, , b" -> "a, b"
        sep_re = re.escape(schema.address_join.strip() or " ")
        joined = (joined.str.replace(rf"(\s*{sep_re}\s*)+", schema.address_join, regex=True)
                  .str.strip().str.strip(schema.address_join.strip() or " ").str.strip())
        df["address"] = joined
    else:
        df["address"] = ""
    df["row_order"] = range(len(df))

    stats = {"file": path, "rows": int(len(df))}
    empty_id = (df["id"] == "").sum()
    if empty_id:
        log.warning("Source %d: %d rows with empty id are dropped", source, empty_id)
        df = df[df["id"] != ""]
    stats["empty_id_dropped"] = int(empty_id)
    stats["empty_name"] = int((df["name"] == "").sum())
    stats["empty_address"] = int((df["address"] == "").sum())
    stats["empty_name_and_address"] = int(((df["name"] == "") & (df["address"] == "")).sum())

    dup_mask = df["id"].duplicated(keep="first")
    stats["duplicate_ids"] = int(dup_mask.sum())
    if dup_mask.any():
        policy = get(cfg, "schema.duplicate_ids", "warn_keep_first")
        msg = f"Source {source}: {int(dup_mask.sum())} duplicated ids (e.g. {df.loc[dup_mask, 'id'].head(3).tolist()})"
        if policy == "error":
            raise SchemaError(msg)
        log.warning("%s -> keeping first occurrence", msg)
        df = df[~dup_mask]
    key = df["name"].str.lower() + "\x1f" + df["address"].str.lower()
    stats["duplicate_name_address_rows"] = int(key.duplicated().sum())
    df = df.reset_index(drop=True)
    df["source"] = source
    log.info("Source %d: %d records (%s)", source, len(df), path)
    return df, stats


def load_sources(cfg: dict, split: str) -> tuple[dict[int, pd.DataFrame], dict]:
    sources, stats = {}, {}
    for s in (1, 2, 3):
        path = get(cfg, f"data.{split}.source{s}")
        if not path:
            raise SchemaError(f"data.{split}.source{s} is not configured")
        sources[s], stats[f"source{s}"] = load_source(path, s, cfg)
    overlap = set(sources[2]["id"]) & set(sources[3]["id"])
    stats["ids_shared_by_source2_and_source3"] = len(overlap)
    if overlap:
        get_logger().warning(
            "%d ids occur in BOTH Source 2 and Source 3 (e.g. %s). Label ids are then ambiguous "
            "unless the label file has one column per source.", len(overlap), sorted(overlap)[:3])
    return sources, stats


def parse_id_list(value, sep: str = ",") -> list[str]:
    """Parse "a,b", "a, b", "[a, b]", "['a','b']", "" or "[]" into a list."""
    if value is None:
        return []
    text = str(value).strip()
    if text.lower() in _EMPTY_TOKENS:
        return []
    if (text.startswith("[") and text.endswith("]")) or (text.startswith("(") and text.endswith(")")) \
            or (text.startswith("{") and text.endswith("}")):
        text = text[1:-1]
    out, seen = [], set()
    for tok in text.split(sep):
        tok = tok.strip().strip("'\"").strip()
        if tok and tok.lower() not in _EMPTY_TOKENS and tok not in seen:
            seen.add(tok)
            out.append(tok)
    return out


def detect_list_format(values: pd.Series) -> str:
    vals = [str(v).strip() for v in values if str(v).strip()]
    if not vals:
        return "plain"
    bracketed = sum(v.startswith("[") and v.endswith("]") for v in vals)
    return "bracketed" if bracketed > len(vals) / 2 else "plain"


def load_labels(path: str, cfg: dict) -> tuple[pd.DataFrame, dict]:
    """Return a frame with columns [s1_id, <match col>..., all_ids(list)]."""
    log = get_logger()
    schema: LabelSchema = label_schema(cfg)
    raw = read_tsv(path, cfg)
    validate_columns(raw, schema.required, f"Labels ({path})")
    out = pd.DataFrame({"s1_id": _clean_str(raw[schema.source1_id])})
    fmt_votes = []
    for col in schema.match_columns:
        out[col] = [parse_id_list(v, schema.list_separator) for v in raw[col]]
        fmt_votes.append(detect_list_format(raw[col]))
    out["all_ids"] = [
        list(dict.fromkeys(i for col in schema.match_columns for i in row[col]))
        for _, row in out.iterrows()
    ]
    stats = {"file": path, "rows": int(len(out)), "list_format": max(set(fmt_votes), key=fmt_votes.count)}
    dups = out["s1_id"].duplicated(keep=False)
    if dups.any():
        log.warning("Labels: %d rows share a Source-1 id -> their lists are unioned", int(dups.sum()))
        agg = {col: (lambda s: list(dict.fromkeys(x for lst in s for x in lst))) for col in
               [*schema.match_columns, "all_ids"]}
        out = out.groupby("s1_id", as_index=False, sort=False).agg(agg)
    stats["duplicate_source1_rows"] = int(dups.sum())
    sizes = out["all_ids"].map(len)
    stats["entities"] = int(len(out))
    stats["singletons"] = int((sizes == 0).sum())
    stats["singleton_rate"] = float((sizes == 0).mean()) if len(out) else 0.0
    stats["match_list_size_distribution"] = {int(k): int(v) for k, v in sizes.value_counts().sort_index().items()}
    stats["total_label_ids"] = int(sizes.sum())
    log.info("Labels: %d Source-1 entities, %d singletons (%.1f%%), %d matching ids",
             len(out), stats["singletons"], 100 * stats["singleton_rate"], stats["total_label_ids"])
    return out, stats


def read_sample_submission_header(cfg: dict) -> list[str] | None:
    path = get(cfg, "data.sample_submission")
    if not path:
        return None
    if not os.path.exists(path):
        get_logger().warning("sample_submission %s not found - mirroring the label schema", path)
        return None
    return list(read_tsv(path, cfg).columns)


def write_tsv(df: pd.DataFrame, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    df.to_csv(path, sep=SEP, index=False, lineterminator="\n", quoting=csv.QUOTE_MINIMAL)


In [ ]:
%%writefile src/normalization.py
"""Business-name normalisation (PDF: "Lexical Standardization and Legal Suffix
Abstraction").

Pipeline for every name:
  1. Unicode NFKD decomposition, removal of Latin combining marks
     (optionally `unidecode` transliteration, an offline table).
  2. Aggressive case folding (str.casefold: "Straße" -> "strasse").
  3. Punctuation handling: dotted acronyms are collapsed ("S.A." -> "sa",
     "L.L.C." -> "llc"), letter/letter slashes joined ("M/s" -> "ms"),
     apostrophes removed, "&" -> "and", every other non-alphanumeric -> space.
  4. Legal suffix extraction with a LONGEST-FIRST match at the end of the
     token list, repeated (so "co ltd" or "gmbh and co kg" are peeled as a
     unit). The suffix is NOT discarded: it is kept in its own column
     (canonical form + family) and used as a matching feature, as the PDF
     prescribes ("Acme GmbH" vs "Acme LLC").
  5. Conservative abbreviation canonicalisation ("manufacturing"/"mfg" ->
     "mfg") so both vendor conventions map to one token.
  6. Derived keys: token list, no-space string, phonetic codes, prefix,
     initials, digits.
"""
from __future__ import annotations

import re
import unicodedata
from functools import lru_cache

try:  # optional, offline transliteration table
    from unidecode import unidecode as _unidecode
except ImportError:  # pragma: no cover
    _unidecode = None

try:
    import jellyfish
except ImportError:  # pragma: no cover
    jellyfish = None

# ---------------------------------------------------------------------------
# Legal suffixes: canonical -> variants (variants written in NORMALISED form,
# i.e. after steps 1-3, so "S.A. de C.V." is "sa de cv").
# ---------------------------------------------------------------------------
LEGAL_SUFFIXES: dict[str, list[str]] = {
    # North America
    "inc": ["inc", "incorporated", "incorporation", "incorp"],
    "corp": ["corp", "corporation", "corpn", "corpo"],
    "co": ["co", "company", "cmpny", "compny"],
    "llc": ["llc", "l l c", "limited liability company", "limited liability co"],
    "pllc": ["pllc", "professional limited liability company"],
    "lp": ["lp", "limited partnership"],
    "llp": ["llp", "limited liability partnership"],
    "pc": ["pc", "professional corporation"],
    "ltee": ["ltee", "limitee"],
    # UK / Commonwealth
    "ltd": ["ltd", "limited", "ltd co", "limited co"],
    "plc": ["plc", "public limited company"],
    "pty ltd": ["pty ltd", "pty limited", "proprietary limited"],
    "pty": ["pty"],
    # India
    "pvt ltd": ["pvt ltd", "private limited", "pvt limited", "private ltd", "p ltd", "pvt"],
    "opc pvt ltd": ["opc pvt ltd", "opc private limited"],
    # Company-limited compounds (East Asia / generic)
    "co ltd": ["co ltd", "company limited", "co limited", "company ltd"],
    "co inc": ["co inc", "company inc"],
    # German-speaking
    "gmbh": ["gmbh", "gesellschaft mit beschrankter haftung"],
    "gmbh co kg": ["gmbh and co kg", "gmbh co kg", "gmbh cokg"],
    "ag": ["ag", "aktiengesellschaft"],
    "kg": ["kg", "kommanditgesellschaft"],
    "ug": ["ug", "ug haftungsbeschrankt"],
    "ohg": ["ohg"],
    "ev": ["ev", "eingetragener verein"],
    # Romance languages / Latin America
    "sa": ["sa", "s a", "sociedad anonima", "societe anonyme"],
    "sa de cv": ["sa de cv", "s a de c v", "sociedad anonima de capital variable"],
    "sapi de cv": ["sapi de cv"],
    "s de rl de cv": ["s de rl de cv", "s de r l de c v", "srl de cv"],
    "s de rl": ["s de rl", "s de r l"],
    "sl": ["sl", "sociedad limitada"],
    "sas": ["sas"],
    "sarl": ["sarl"],
    "eurl": ["eurl"],
    "srl": ["srl", "societa a responsabilita limitata"],
    "spa": ["spa", "societa per azioni"],
    "ltda": ["ltda", "limitada"],
    "eireli": ["eireli"],
    # Benelux / Nordics / others
    "bv": ["bv"],
    "nv": ["nv"],
    "ab": ["ab"],
    "aps": ["aps"],
    "as": ["as", "asa"],
    "oy": ["oy", "oyj"],
    "sp z oo": ["sp z oo", "sp z o o"],
    "kk": ["kk", "kabushiki kaisha"],
    "bhd": ["bhd", "sdn bhd", "berhad", "sendirian berhad"],
}

# Legal forms that appear as a PREFIX (Indonesia, CIS, ...).
LEGAL_PREFIXES: dict[str, list[str]] = {
    "pt": ["pt"],
    "cv": ["cv"],
    "ud": ["ud"],
    "ooo": ["ooo"],
    "zao": ["zao"],
    "oao": ["oao"],
}

# Coarse families so that interchangeable suffixes ("inc" vs "corp") are not
# treated as a contradiction while "llc" vs "gmbh" is.
SUFFIX_FAMILY: dict[str, str] = {
    "inc": "corporation", "corp": "corporation", "co": "corporation", "co inc": "corporation",
    "pc": "corporation", "llc": "llc", "pllc": "llc",
    "lp": "partnership", "llp": "partnership", "kg": "partnership", "ohg": "partnership",
    "ltd": "limited", "plc": "limited", "pty ltd": "limited", "pty": "limited",
    "pvt ltd": "limited", "opc pvt ltd": "limited", "co ltd": "limited", "ltee": "limited",
    "gmbh": "gmbh", "gmbh co kg": "gmbh", "ug": "gmbh", "ag": "ag", "ev": "association",
    "sa": "sa", "sa de cv": "sa", "sapi de cv": "sa", "spa": "sa", "sas": "sa", "nv": "sa",
    "s de rl de cv": "srl", "s de rl": "srl", "sl": "srl", "srl": "srl", "sarl": "srl",
    "eurl": "srl", "ltda": "srl", "eireli": "srl", "bv": "srl", "sp z oo": "srl",
    "ab": "ab", "aps": "ab", "as": "ab", "oy": "ab", "kk": "kk", "bhd": "limited",
}

# Conservative canonicalisation of frequent business-name abbreviations.
# Both the long and short forms map to one canonical token.
NAME_ABBREVIATIONS: dict[str, str] = {
    "manufacturing": "mfg", "manufacturers": "mfg", "manufacturer": "mfg", "mfrs": "mfg",
    "mfr": "mfg", "mfg": "mfg", "manuf": "mfg",
    "international": "intl", "intl": "intl", "intnl": "intl", "int'l": "intl",
    "technologies": "tech", "technology": "tech", "techn": "tech", "tech": "tech", "technol": "tech",
    "holdings": "hldg", "holding": "hldg", "hldgs": "hldg", "hldg": "hldg",
    "services": "svc", "service": "svc", "svcs": "svc", "svc": "svc", "srvcs": "svc", "servs": "svc",
    "solutions": "soln", "solution": "soln", "solns": "soln", "soln": "soln",
    "industries": "ind", "industry": "ind", "inds": "ind", "indus": "ind",
    "enterprises": "ent", "enterprise": "ent", "entp": "ent", "entps": "ent", "ent": "ent",
    "associates": "assoc", "associated": "assoc", "assoc": "assoc", "assocs": "assoc",
    "association": "assn", "assn": "assn",
    "brothers": "bros", "brother": "bros", "bros": "bros",
    "department": "dept", "dept": "dept",
    "distributors": "dist", "distribution": "dist", "distributor": "dist", "distr": "dist",
    "engineering": "eng", "engineers": "eng", "engg": "eng", "engrs": "eng",
    "products": "prod", "product": "prod", "prods": "prod",
    "systems": "sys", "system": "sys", "sys": "sys",
    "management": "mgmt", "mgmt": "mgmt", "mgt": "mgmt",
    "laboratories": "labs", "laboratory": "labs", "lab": "labs", "labs": "labs",
    "pharmaceuticals": "pharma", "pharmaceutical": "pharma", "pharma": "pharma",
    "construction": "constr", "constructions": "constr", "constr": "constr",
    "communications": "comm", "communication": "comm", "comms": "comm",
    "consulting": "consult", "consultants": "consult", "consultancy": "consult", "consultant": "consult",
    "center": "ctr", "centre": "ctr", "ctr": "ctr",
    "national": "natl", "natl": "natl",
    "university": "univ", "univ": "univ",
    "institute": "inst", "inst": "inst",
    "mount": "mt",
    "private": "pvt",
    "and": "and", "n": "and",
}

STOPWORDS = frozenset({"the", "and", "of", "a", "an", "de", "la", "le", "du", "des", "der", "die", "das"})

_ACRONYM_RE = re.compile(r"(?<![\w])([^\W\d_])(?:\.([^\W\d_]))+\.?(?![\w])")
_SLASH_RE = re.compile(r"(?<![\w])([^\W\d_])\s*/\s*([^\W\d_])(?![\w])")
_APOS_RE = re.compile(r"['’‘`´]")
# Combining marks of non-Latin scripts (Indic vowel signs, Arabic/Hebrew
# points, Thai marks) are not matched by \w but are part of the word.
MARK_RANGES = ("\u0300-\u036F\u0483-\u0489\u0591-\u05BD\u05BF-\u05C7\u0610-\u061A"
               "\u064B-\u065F\u0670\u06D6-\u06ED\u0900-\u0DFF\u0E31-\u0E4E\u0EB1-\u0ECD"
               "\u1AB0-\u1AFF\u1DC0-\u1DFF\u20D0-\u20FF\uFE20-\uFE2F")
_NONWORD_RE = re.compile(rf"[^\w{MARK_RANGES}]+|_+")
_MS_PREFIX_RE = re.compile(r"^\s*m\s*/\s*s\b\.?\s*", re.IGNORECASE)
_DIGITS_RE = re.compile(r"\d+")


def _build_lookup(table: dict[str, list[str]]) -> dict[tuple[str, ...], str]:
    lookup: dict[tuple[str, ...], str] = {}
    for canonical, variants in table.items():
        for v in [canonical, *variants]:
            lookup[tuple(v.split())] = canonical
    return lookup


def strip_latin_marks(text: str) -> str:
    """NFKD + drop combining marks that follow a Latin base character.

    Marks after non-Latin letters (e.g. Devanagari vowel signs) are kept, so
    non-Latin names are not destroyed.
    """
    decomposed = unicodedata.normalize("NFKD", text)
    out = []
    prev_latin = False
    for ch in decomposed:
        if unicodedata.combining(ch):
            if prev_latin:
                continue
            out.append(ch)
            continue
        out.append(ch)
        prev_latin = ord(ch) < 0x0250
    return "".join(out)


def clean_text(text: str, transliterate: bool = False) -> str:
    """Steps 1-3: unicode fold, case fold, punctuation handling."""
    if not text:
        return ""
    s = strip_latin_marks(str(text))
    if transliterate and _unidecode is not None:
        s = _unidecode(s)
    s = s.casefold()
    s = _APOS_RE.sub("", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = _ACRONYM_RE.sub(lambda m: m.group(0).replace(".", ""), s)
    s = _SLASH_RE.sub(r"\1\2", s)
    s = _NONWORD_RE.sub(" ", s)
    return " ".join(s.split())


class NameNormalizer:
    """Stateful normaliser (dictionaries can be extended from the config)."""

    def __init__(self, cfg: dict | None = None):
        cfg = cfg or {}
        norm_cfg = cfg.get("normalization", {}) if isinstance(cfg, dict) else {}
        suffixes = {k: list(v) for k, v in LEGAL_SUFFIXES.items()}
        for canon, variants in (norm_cfg.get("extra_legal_suffixes") or {}).items():
            suffixes.setdefault(canon, []).extend(variants)
        self.suffix_lookup = _build_lookup(suffixes)
        self.max_suffix_len = max(len(k) for k in self.suffix_lookup)
        self.prefix_lookup = _build_lookup(LEGAL_PREFIXES)
        self.abbrev = dict(NAME_ABBREVIATIONS)
        self.abbrev.update(norm_cfg.get("extra_name_abbreviations") or {})
        self.transliterate = bool(norm_cfg.get("transliterate", False))
        self.strip_the = bool(norm_cfg.get("strip_leading_the", True))
        self._cached = lru_cache(maxsize=500_000)(self._normalize)

    def extract_suffix(self, tokens: list[str]) -> tuple[list[str], list[str]]:
        """Longest-first legal suffix peeling at the end of the token list.

        Returns (core_tokens, canonical_suffixes_in_original_order). A suffix
        is never peeled if that would leave an empty core ("The Company").
        """
        tokens = list(tokens)
        found: list[str] = []
        for _ in range(3):
            matched = False
            for length in range(min(self.max_suffix_len, len(tokens) - 1), 0, -1):
                key = tuple(tokens[-length:])
                if key in self.suffix_lookup:
                    found.insert(0, self.suffix_lookup[key])
                    tokens = tokens[:-length]
                    matched = True
                    break
            if not matched:
                break
        return tokens, found

    def extract_prefix(self, tokens: list[str]) -> tuple[list[str], str]:
        if len(tokens) > 1 and (tokens[0],) in self.prefix_lookup:
            return tokens[1:], self.prefix_lookup[(tokens[0],)]
        return tokens, ""

    def _normalize(self, raw: str) -> dict:
        raw = raw or ""
        title = ""
        if _MS_PREFIX_RE.match(raw):
            title = "ms"
            raw = _MS_PREFIX_RE.sub("", raw)
        clean = clean_text(raw, self.transliterate)
        tokens = clean.split()
        if self.strip_the and len(tokens) > 1 and tokens[0] == "the":
            tokens = tokens[1:]
        core_tokens, suffixes = self.extract_suffix(tokens)
        core_tokens, legal_prefix = self.extract_prefix(core_tokens)
        canon_tokens = [self.abbrev.get(t, t) for t in core_tokens]
        core = " ".join(canon_tokens)
        suffix = " ".join(suffixes)
        families = sorted({SUFFIX_FAMILY.get(s, s) for s in suffixes})
        content = [t for t in canon_tokens if t not in STOPWORDS] or canon_tokens
        return {
            "name_clean": clean,
            "name_core": core,
            "name_tokens": " ".join(content),
            "name_nospace": "".join(content),
            "name_suffix": suffix,
            "name_suffix_family": "|".join(families),
            "name_legal_prefix": legal_prefix,
            "name_title": title,
            "name_phonetic": phonetic_key(" ".join(content)),
            "name_first_phonetic": phonetic_key(content[0]) if content else "",
            "name_initials": "".join(t[0] for t in content if t),
            "name_digits": " ".join(_DIGITS_RE.findall(core)),
        }

    def __call__(self, raw: str) -> dict:
        return self._cached(raw or "")


def phonetic_key(text: str) -> str:
    """Metaphone code of the ASCII part of `text` (spaces removed)."""
    if not text:
        return ""
    ascii_text = text if text.isascii() else (
        _unidecode(text) if _unidecode is not None else "".join(c for c in text if c.isascii()))
    ascii_text = re.sub(r"[^a-z ]", "", ascii_text.lower()).strip()
    if not ascii_text:
        return ""
    if jellyfish is not None:
        try:
            return jellyfish.metaphone(ascii_text).replace(" ", "").upper()
        except Exception:  # pragma: no cover - defensive
            pass
    return soundex(ascii_text.replace(" ", ""))


def soundex(word: str) -> str:
    """Fallback phonetic code when jellyfish is unavailable."""
    codes = {**dict.fromkeys("bfpv", "1"), **dict.fromkeys("cgjkqsxz", "2"),
             **dict.fromkeys("dt", "3"), "l": "4", **dict.fromkeys("mn", "5"), "r": "6"}
    word = "".join(c for c in word.lower() if c.isalpha())
    if not word:
        return ""
    out, last = word[0].upper(), codes.get(word[0], "")
    for c in word[1:]:
        code = codes.get(c, "")
        if code and code != last:
            out += code
        if c not in "hw":
            last = code
    return (out + "000")[:4]


NAME_FIELDS = ["name_clean", "name_core", "name_tokens", "name_nospace", "name_suffix",
               "name_suffix_family", "name_legal_prefix", "name_title", "name_phonetic",
               "name_first_phonetic", "name_initials", "name_digits"]


In [ ]:
%%writefile src/address_parser.py
"""Offline address normalisation and parsing (PDF: "Statistical Address Parsing
in an Offline Context").

Two back-ends, both 100% OFFLINE:

* ``libpostal`` (optional, via the `postal` Python bindings). libpostal is a
  local C library + model files; parsing makes no network call. It is used
  when installed and `address.parser` is `auto` or `libpostal`.
* A rule-based multi-region fallback (always available) that extracts the
  same components: house_number, road, unit, city, state, postcode, country,
  landmark ("house" in libpostal terms, e.g. "Nr City Hall") and suburb.

NEVER used: geocoders, map APIs, OpenStreetMap queries or any other online
lookup - the challenge forbids external databases, APIs and lookups.
"""
from __future__ import annotations

import re
from functools import lru_cache

from .normalization import MARK_RANGES, strip_latin_marks
from .utils import get_logger

try:  # optional offline statistical parser
    from postal.parser import parse_address as _lp_parse  # type: ignore
except Exception:  # pragma: no cover - libpostal usually not installed
    _lp_parse = None

ADDRESS_FIELDS = ["addr_clean", "addr_house_number", "addr_road", "addr_unit", "addr_city",
                  "addr_state", "addr_postcode", "addr_country", "addr_landmark", "addr_suburb",
                  "addr_numbers", "addr_parse_method"]

ADDRESS_ABBREVIATIONS: dict[str, str] = {
    "street": "st", "st": "st", "strt": "st", "stree": "st",
    "avenue": "ave", "ave": "ave", "av": "ave", "aven": "ave", "avn": "ave", "avenu": "ave",
    "road": "rd", "rd": "rd", "raod": "rd",
    "boulevard": "blvd", "blvd": "blvd", "boul": "blvd", "blv": "blvd",
    "drive": "dr", "dr": "dr", "drv": "dr",
    "lane": "ln", "ln": "ln",
    "court": "ct", "ct": "ct", "crt": "ct",
    "place": "pl", "pl": "pl",
    "square": "sq", "sq": "sq",
    "highway": "hwy", "hwy": "hwy", "hiway": "hwy",
    "parkway": "pkwy", "pkwy": "pkwy", "pky": "pkwy",
    "expressway": "expy", "expy": "expy", "freeway": "fwy", "fwy": "fwy",
    "terrace": "ter", "ter": "ter", "terr": "ter",
    "circle": "cir", "cir": "cir", "crescent": "cres", "cres": "cres",
    "trail": "trl", "trl": "trl", "alley": "aly", "aly": "aly",
    "strasse": "str", "str": "str", "gasse": "g", "weg": "weg", "platz": "pl",
    "suite": "ste", "ste": "ste", "suit": "ste",
    "apartment": "apt", "apartments": "apt", "apt": "apt",
    "building": "bldg", "bldg": "bldg", "bld": "bldg",
    "floor": "fl", "fl": "fl", "flr": "fl",
    "room": "rm", "rm": "rm", "office": "ofc", "ofc": "ofc",
    "number": "no", "no": "no", "num": "no", "nbr": "no", "nr.": "no",
    "north": "n", "south": "s", "east": "e", "west": "w",
    "northeast": "ne", "northwest": "nw", "southeast": "se", "southwest": "sw",
    "near": "nr", "nr": "nr", "nearby": "nr",
    "opposite": "opp", "opp": "opp", "opposit": "opp",
    "behind": "behind", "beside": "beside", "adjacent": "adj", "adj": "adj",
    "junction": "jn", "jn": "jn", "jct": "jn",
    "extension": "extn", "extn": "extn", "ext": "extn",
    "industrial": "indl", "indl": "indl",
    "market": "mkt", "mkt": "mkt",
    "block": "blk", "blk": "blk", "sector": "sec", "sec": "sec",
    "mount": "mt", "mt": "mt", "fort": "ft", "heights": "hts", "hts": "hts",
    "center": "ctr", "centre": "ctr", "ctr": "ctr",
    "first": "1st", "second": "2nd", "third": "3rd", "fourth": "4th", "fifth": "5th",
    "sixth": "6th", "seventh": "7th", "eighth": "8th", "ninth": "9th", "tenth": "10th",
}
PHRASE_ABBREVIATIONS = [
    ("post office box", "pobox"), ("p o box", "pobox"), ("po box", "pobox"),
    ("next to", "nr"), ("close to", "nr"), ("in front of", "opp"), ("near by", "nr"),
]
STREET_TYPES = frozenset({"st", "ave", "rd", "blvd", "dr", "ln", "ct", "pl", "sq", "hwy", "pkwy",
                          "expy", "fwy", "ter", "cir", "cres", "trl", "aly", "str", "weg", "way",
                          "marg", "path", "gali", "chowk", "main", "cross", "row", "walk", "close",
                          "mews", "gate", "view", "rise", "loop", "run", "pike", "plaza", "calle",
                          "rua", "via", "avenida", "carrera"})
LANDMARK_MARKERS = frozenset({"nr", "opp", "behind", "beside", "adj", "above", "below", "off",
                              "landmark", "via"})
UNIT_MARKERS = ("ste", "apt", "unit", "rm", "fl", "ofc", "shop", "flat", "bldg", "blk", "door",
                "office", "room", "suite", "gala", "stall", "kiosk", "pobox")
DIRECTIONALS = frozenset({"n", "s", "e", "w", "ne", "nw", "se", "sw"})

US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia",
    "kansas": "ks", "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms",
    "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv", "new hampshire": "nh",
    "new jersey": "nj", "new mexico": "nm", "new york": "ny", "north carolina": "nc",
    "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or", "pennsylvania": "pa",
    "rhode island": "ri", "south carolina": "sc", "south dakota": "sd", "tennessee": "tn",
    "texas": "tx", "utah": "ut", "vermont": "vt", "virginia": "va", "washington": "wa",
    "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy", "district of columbia": "dc",
    "puerto rico": "pr", "guam": "gu",
}
CA_PROVINCES = {
    "alberta": "ab", "british columbia": "bc", "manitoba": "mb", "new brunswick": "nb",
    "newfoundland and labrador": "nl", "nova scotia": "ns", "ontario": "on",
    "prince edward island": "pe", "quebec": "qc", "saskatchewan": "sk",
    "northwest territories": "nt", "nunavut": "nu", "yukon": "yt",
}
AU_STATES = {
    "new south wales": "nsw", "victoria": "vic", "queensland": "qld", "western australia": "wa",
    "south australia": "sa", "tasmania": "tas", "australian capital territory": "act",
    "northern territory": "nt",
}
# Indian states map full names -> codes; 2-letter Indian codes are NOT matched
# as bare tokens because they collide with common words ("up", "as", "ga").
IN_STATES = {
    "andhra pradesh": "in-ap", "arunachal pradesh": "in-ar", "assam": "in-as", "bihar": "in-br",
    "chhattisgarh": "in-cg", "goa": "in-ga", "gujarat": "in-gj", "haryana": "in-hr",
    "himachal pradesh": "in-hp", "jharkhand": "in-jh", "karnataka": "in-ka", "kerala": "in-kl",
    "madhya pradesh": "in-mp", "maharashtra": "in-mh", "manipur": "in-mn", "meghalaya": "in-ml",
    "mizoram": "in-mz", "nagaland": "in-nl", "odisha": "in-od", "orissa": "in-od",
    "punjab": "in-pb", "rajasthan": "in-rj", "sikkim": "in-sk", "tamil nadu": "in-tn",
    "tamilnadu": "in-tn", "telangana": "in-ts", "tripura": "in-tr", "uttar pradesh": "in-up",
    "uttarakhand": "in-uk", "uttaranchal": "in-uk", "west bengal": "in-wb",
    "jammu and kashmir": "in-jk", "ladakh": "in-la", "puducherry": "in-py",
    "pondicherry": "in-py", "chandigarh": "in-ch", "national capital territory of delhi": "in-dl",
    "nct of delhi": "in-dl", "andaman and nicobar islands": "in-an", "lakshadweep": "in-ld",
    "dadra and nagar haveli and daman and diu": "in-dn",
}
COUNTRIES = {
    "usa": "us", "us": "us", "united states": "us", "united states of america": "us",
    "india": "in", "bharat": "in", "uk": "gb", "united kingdom": "gb", "great britain": "gb",
    "england": "gb", "scotland": "gb", "wales": "gb", "germany": "de", "deutschland": "de",
    "canada": "ca", "australia": "au", "mexico": "mx", "france": "fr", "spain": "es",
    "espana": "es", "italy": "it", "italia": "it", "netherlands": "nl", "singapore": "sg",
    "uae": "ae", "united arab emirates": "ae", "china": "cn", "japan": "jp", "brazil": "br",
    "brasil": "br", "ireland": "ie", "new zealand": "nz", "south africa": "za",
}

DEFAULT_POSTCODE_PATTERNS = [
    # (name, regex, numeric_only). Order = priority on ties.
    ("us_zip4", r"(?<![\w-])(\d{5})-\d{4}(?![\w-])", True),
    ("ca", r"(?<![\w-])([a-z]\d[a-z] ?\d[a-z]\d)(?![\w-])", False),
    ("uk", r"(?<![\w-])([a-z]{1,2}\d[a-z\d]? ?\d[a-z]{2})(?![\w-])", False),
    ("in_pin", r"(?<![\w-])(\d{3} ?\d{3})(?![\w-])", True),
    ("five_digit", r"(?<![\w-])(\d{5})(?![\w-])", True),
    ("four_digit", r"(?<![\w-])(\d{4})(?![\w-])", True),
]

_SEG_SPLIT_RE = re.compile(r"[,;|\n\r]+")
_APOS_RE = re.compile(r"['’‘`´]")
_ACRONYM_RE = re.compile(r"(?<![\w])([^\W\d_])(?:\.([^\W\d_]))+\.?(?![\w])")
_LETTER_SEP_RE = re.compile(r"(?<=[^\W\d_])[-/](?=[^\W\d_])")
_KEEP_RE = re.compile(rf"[^\w{MARK_RANGES}\-/ ]+|_+")
_STRAY_SEP_RE = re.compile(r"(?<![\w])[-/]+|[-/]+(?![\w])")
_HN_RE = re.compile(r"^(?:\d+[a-z]?|[a-z]?\d+[a-z]?|\d+[-/][0-9a-z]+(?:[-/][0-9a-z]+)*)$")
_NUMERIC_TOKEN_RE = re.compile(r"\d")
_ORDINAL_FLOOR_RE = re.compile(r"\b(\d+)(?:st|nd|rd|th)? fl\b")


def _unit_regex() -> re.Pattern:
    markers = "|".join(sorted(set(UNIT_MARKERS), key=len, reverse=True))
    return re.compile(rf"\b({markers})\s*(?:no\s*)?([0-9][0-9a-z\-/]*|[a-z][0-9][0-9a-z\-/]*|[a-z])\b")


_UNIT_RE = _unit_regex()


class AddressParser:
    """Parse raw address strings into normalised components (offline)."""

    def __init__(self, cfg: dict | None = None):
        cfg = cfg or {}
        acfg = cfg.get("address", {}) if isinstance(cfg, dict) else {}
        ncfg = cfg.get("normalization", {}) if isinstance(cfg, dict) else {}
        self.abbrev = dict(ADDRESS_ABBREVIATIONS)
        self.abbrev.update(ncfg.get("extra_address_abbreviations") or {})
        self.state_names: dict[str, str] = {**US_STATES, **CA_PROVINCES, **AU_STATES, **IN_STATES}
        self.state_names.update(acfg.get("extra_states") or {})
        self.state_codes = set(US_STATES.values()) | set(CA_PROVINCES.values()) | set(AU_STATES.values())
        self.countries = dict(COUNTRIES)
        for c in acfg.get("extra_countries") or []:
            self.countries[str(c).lower()] = str(c).lower()
        patterns = acfg.get("postcode_patterns") or DEFAULT_POSTCODE_PATTERNS
        self.postcode_patterns = [(n, re.compile(p), bool(num)) for n, p, num in patterns]
        backend = acfg.get("parser", "auto")
        if backend == "libpostal" and _lp_parse is None:
            get_logger().warning("address.parser=libpostal but the `postal` package is not "
                                 "installed -> using the offline rule-based parser")
        self.use_libpostal = backend in ("auto", "libpostal") and _lp_parse is not None
        self._max_state_len = max(len(k.split()) for k in self.state_names)
        self._max_country_len = max(len(k.split()) for k in self.countries)
        self._cached = lru_cache(maxsize=500_000)(self._parse)

    # ------------------------------------------------------------------ text
    def canon_tokens(self, text: str) -> list[str]:
        for phrase, repl in PHRASE_ABBREVIATIONS:
            text = re.sub(rf"\b{phrase}\b", repl, text)
        out = []
        for tok in text.split():
            tok = self.abbrev.get(tok, tok)
            if len(tok) > 7 and tok.endswith("strasse"):
                tok = tok[:-7] + "str"   # "hauptstrasse" -> "hauptstr"
            out.append(tok)
        return out

    def clean_segments(self, raw: str) -> list[list[str]]:
        s = strip_latin_marks(raw or "").casefold()
        s = _APOS_RE.sub("", s).replace("&", " and ").replace("#", " no ")
        s = _ACRONYM_RE.sub(lambda m: m.group(0).replace(".", ""), s)
        segments = []
        for seg in _SEG_SPLIT_RE.split(s):
            seg = _LETTER_SEP_RE.sub(" ", seg)
            seg = _KEEP_RE.sub(" ", seg)
            seg = _STRAY_SEP_RE.sub(" ", seg)
            toks = self.canon_tokens(" ".join(seg.split()))
            if toks:
                segments.append(toks)
        return segments

    # ----------------------------------------------------------------- parse
    def __call__(self, raw: str) -> dict:
        return self._cached(raw or "")

    def _parse(self, raw: str) -> dict:
        segments = self.clean_segments(raw)
        clean = " ".join(" ".join(seg) for seg in segments)
        if self.use_libpostal and clean:
            try:
                parsed = self._parse_libpostal(raw)
                parsed["addr_clean"] = clean
                return parsed
            except Exception as exc:  # pragma: no cover - defensive
                get_logger().debug("libpostal failed on %r: %s", raw, exc)
        parsed = self._parse_rules(segments)
        parsed["addr_clean"] = clean
        return parsed

    def _parse_libpostal(self, raw: str) -> dict:
        comps: dict[str, list[str]] = {}
        for value, label in _lp_parse(raw):
            comps.setdefault(label, []).append(value)

        def norm(labels):
            vals = [v for lab in labels for v in comps.get(lab, [])]
            return " ".join(self.canon_tokens(" ".join(" ".join(t) for t in self.clean_segments(" ".join(vals)))))

        state = norm(["state"])
        state = self.state_names.get(state, state)
        postcode = norm(["postcode"]).replace(" ", "")
        numbers = sorted({t for seg in self.clean_segments(raw) for t in seg
                          if _NUMERIC_TOKEN_RE.search(t) and t != postcode})
        return {
            "addr_house_number": norm(["house_number"]),
            "addr_road": norm(["road"]),
            "addr_unit": norm(["unit", "level", "staircase", "entrance", "po_box"]),
            "addr_city": norm(["city", "city_district"]),
            "addr_state": state,
            "addr_postcode": postcode,
            "addr_country": self.countries.get(norm(["country"]), norm(["country"])),
            "addr_landmark": norm(["house", "near", "category"]),
            "addr_suburb": norm(["suburb", "state_district", "island"]),
            "addr_numbers": " ".join(numbers),
            "addr_parse_method": "libpostal",
        }

    def _strip_trailing_lookup(self, tokens: list[str], table: dict[str, str], max_len: int):
        for n in range(min(max_len, len(tokens)), 0, -1):
            key = " ".join(tokens[-n:])
            if key in table:
                return tokens[:-n], table[key]
        return tokens, ""

    def _find_postcode(self, segments: list[list[str]]):
        best = None  # (seg_idx, end_pos, priority, value, start, end)
        for si, seg in enumerate(segments):
            text = " ".join(seg)
            for pri, (name, rx, numeric_only) in enumerate(self.postcode_patterns):
                for m in rx.finditer(text):
                    start, end = m.span(0)
                    if numeric_only and si == 0 and start == 0 and (len(segments) > 1 or len(seg) > 1):
                        continue  # leading number of the first segment = house number
                    if name == "four_digit" and si != len(segments) - 1:
                        continue
                    if numeric_only and end < len(text):
                        nxt = text[end:].split()
                        if nxt and nxt[0] in STREET_TYPES:
                            continue
                    key = (si, end, -pri)
                    if best is None or key > best[0]:
                        best = (key, si, m.group(1).replace(" ", ""), start, end, name)
        if best is None:
            return segments, ""
        _, si, value, start, end, name = best
        text = " ".join(segments[si])
        remainder = (text[:start] + " " + text[end:]).split()
        segments = [s for s in segments]
        segments[si] = remainder
        return segments, value

    def _parse_rules(self, segments: list[list[str]]) -> dict:
        segments = [list(s) for s in segments]
        n_raw_segments = len(segments)
        result = {k: "" for k in ADDRESS_FIELDS}
        result["addr_parse_method"] = "rules"
        if not segments:
            return result

        # 1. country (trailing tokens of the last segment)
        segments[-1], country = self._strip_trailing_lookup(segments[-1], self.countries,
                                                            self._max_country_len)
        segments = [s for s in segments if s]
        # 2. postcode (rightmost plausible match)
        segments, postcode = self._find_postcode(segments)
        segments = [s for s in segments if s]
        numbers = sorted({t for s in segments for t in s if _NUMERIC_TOKEN_RE.search(t)})
        # 3. state in the last two segments
        state, state_seg_rest = "", None
        for si in range(len(segments) - 1, max(-1, len(segments) - 3), -1):
            seg = segments[si]
            rest, st = self._strip_trailing_lookup(seg, self.state_names, self._max_state_len)
            if not st and seg and seg[-1] in self.state_codes and (len(seg) > 1 or si > 0):
                rest, st = seg[:-1], seg[-1]
            if st:
                state = st
                segments[si] = rest
                state_seg_rest = (si, rest)
                break
        # 4. units (anywhere)
        units = []
        new_segments = []
        for seg in segments:
            text = " ".join(seg)
            text = _ORDINAL_FLOOR_RE.sub(lambda m: f"fl {m.group(1)}", text)
            for m in _UNIT_RE.finditer(text):
                units.append(m.group(2))
            text = _UNIT_RE.sub(" ", text)
            new_segments.append(text.split())
        segments = new_segments
        # 5. landmarks: segment starting with a marker, or tail after a marker
        landmarks, kept = [], []
        for si, seg in enumerate(segments):
            if not seg:
                kept.append((si, seg))
                continue
            idx = next((i for i, t in enumerate(seg) if t in LANDMARK_MARKERS), None)
            if idx is not None:
                landmarks.append(" ".join(seg[idx:]))
                seg = seg[:idx]
            kept.append((si, seg))
        seg_map = {si: seg for si, seg in kept}
        order = [si for si, seg in kept if seg]

        city, road, house_number, suburbs = "", "", "", []
        used: set[int] = set()
        # 6. city: residue of the state segment, else last remaining segment
        if state_seg_rest is not None and seg_map.get(state_seg_rest[0]):
            si = state_seg_rest[0]
            seg = seg_map[si]
            if si != (order[0] if order else -1) or len(order) == 1:
                split_at = max((i for i, t in enumerate(seg) if t in STREET_TYPES), default=-1)
                if split_at >= 0 and si == order[0]:
                    city = " ".join(seg[split_at + 1:])
                    seg_map[si] = seg[:split_at + 1]
                else:
                    city = " ".join(seg)
                    used.add(si)
        if not city and (len(order) >= 2 or (len(order) == 1 and n_raw_segments >= 2 and landmarks)):
            last = order[-1]
            cand = seg_map[last]
            if not any(_NUMERIC_TOKEN_RE.search(t) for t in cand) and not any(t in STREET_TYPES for t in cand):
                city = " ".join(cand)
                used.add(last)
        # 7. house number + road in the first unused segments
        remaining = [si for si in order if si not in used and seg_map[si]]
        if remaining:
            first = remaining[0]
            seg = seg_map[first]
            hn_len = 0
            if len(seg) >= 2 and seg[0] in ("no", "plot", "door", "h", "house", "bldg") :
                j = 1
                if seg[0] != "no" and j < len(seg) and seg[j] == "no":
                    j += 1
                if j < len(seg) and _HN_RE.match(seg[j]):
                    house_number, hn_len = seg[j], j + 1
            elif _HN_RE.match(seg[0]) and _NUMERIC_TOKEN_RE.search(seg[0]):
                house_number, hn_len = seg[0], 1
            elif len(seg) >= 2 and _HN_RE.match(seg[-1]) and _NUMERIC_TOKEN_RE.search(seg[-1]) \
                    and (seg[-2] in STREET_TYPES or seg[-2].endswith("str")):
                house_number = seg[-1]
                seg = seg[:-1]
            rest = seg[hn_len:]
            if rest:
                if not city and len(remaining) == 1:
                    split_at = max((i for i, t in enumerate(rest) if t in STREET_TYPES), default=-1)
                    if 0 <= split_at < len(rest) - 1:
                        city = " ".join(rest[split_at + 1:])
                        rest = rest[:split_at + 1]
                road = " ".join(rest)
                used.add(first)
            else:
                used.add(first)
                nxt = [si for si in remaining[1:]]
                pick = next((si for si in nxt if any(t in STREET_TYPES for t in seg_map[si])),
                            nxt[0] if nxt else None)
                if pick is not None:
                    road = " ".join(seg_map[pick])
                    used.add(pick)
            if not road:
                pick = next((si for si in remaining if si not in used
                             and any(t in STREET_TYPES for t in seg_map[si])), None)
                if pick is not None:
                    road = " ".join(seg_map[pick])
                    used.add(pick)
        for si in order:
            if si not in used and seg_map[si]:
                suburbs.append(" ".join(seg_map[si]))

        result.update({
            "addr_house_number": house_number,
            "addr_road": road,
            "addr_unit": " ".join(units),
            "addr_city": city,
            "addr_state": state,
            "addr_postcode": postcode,
            "addr_country": country,
            "addr_landmark": " | ".join(landmarks),
            "addr_suburb": " ".join(suburbs),
            "addr_numbers": " ".join(numbers),
        })
        return result


def libpostal_available() -> bool:
    return _lp_parse is not None


In [ ]:
%%writefile src/preprocess.py
"""Apply name normalisation and offline address parsing to all records.

Produces two tables per dataset:
  * ``s1``: Source-1 records (row index = ``s1_idx``)
  * ``v``:  Source-2 and Source-3 records stacked (row index = ``v_idx``),
            with a ``source`` column (2 or 3).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .address_parser import ADDRESS_FIELDS, AddressParser
from .normalization import NAME_FIELDS, NameNormalizer
from .utils import get_logger


def _apply_unique(values: pd.Series, fn, fields: list[str]) -> pd.DataFrame:
    """Run `fn` once per unique string (vendor data is highly repetitive) and
    broadcast the parsed columns back with integer codes (no per-row dicts)."""
    codes, uniques = pd.factorize(values, sort=False)
    parsed = [fn(u) for u in uniques]
    out = {}
    for f in fields:
        col = np.array([p[f] for p in parsed], dtype=object)
        out[f] = col[codes] if len(codes) else col[:0]
    del parsed
    return pd.DataFrame(out, index=values.index)


def normalize_records(df: pd.DataFrame, normalizer: NameNormalizer, parser: AddressParser) -> pd.DataFrame:
    # the batch path is already de-duplicated -> bypass the per-call LRU caches
    names = _apply_unique(df["name"], getattr(normalizer, "_normalize", normalizer), NAME_FIELDS)
    addrs = _apply_unique(df["address"], getattr(parser, "_parse", parser), ADDRESS_FIELDS)
    out = pd.concat([df.reset_index(drop=True), names.reset_index(drop=True),
                     addrs.reset_index(drop=True)], axis=1)
    out["full_text"] = (out["name_core"] + " " + out["addr_clean"]).str.strip()
    out["addr_street_key"] = [
        f"{hn}|{_first_significant(road)}" if hn and road else ""
        for hn, road in zip(out["addr_house_number"], out["addr_road"])
    ]
    out["city_state_key"] = [
        f"{c}|{s}" if c else "" for c, s in zip(out["addr_city"], out["addr_state"])
    ]
    return out


def _first_significant(road: str) -> str:
    for tok in road.split():
        if tok not in {"n", "s", "e", "w", "ne", "nw", "se", "sw", "the"}:
            return tok
    return road.split()[0] if road else ""


def preprocess_dataset(sources: dict[int, pd.DataFrame], cfg: dict) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    log = get_logger()
    normalizer = NameNormalizer(cfg)
    parser = AddressParser(cfg)
    log.info("Address parser backend: %s", "libpostal (offline)" if parser.use_libpostal else "offline rules")
    s1 = normalize_records(sources[1], normalizer, parser)
    s1["s1_idx"] = range(len(s1))
    v = pd.concat([normalize_records(sources[2], normalizer, parser),
                   normalize_records(sources[3], normalizer, parser)], ignore_index=True)
    v["v_idx"] = range(len(v))
    v["key"] = v["source"].astype(str) + ":" + v["id"]
    stats = {}
    for name, tab in (("source1", s1), ("vendors", v)):
        stats[name] = {
            "records": int(len(tab)),
            "empty_core_name": int((tab["name_core"] == "").sum()),
            "with_legal_suffix": int((tab["name_suffix"] != "").sum()),
            "with_postcode": int((tab["addr_postcode"] != "").sum()),
            "with_house_number": int((tab["addr_house_number"] != "").sum()),
            "with_city": int((tab["addr_city"] != "").sum()),
            "with_state": int((tab["addr_state"] != "").sum()),
            "with_unit": int((tab["addr_unit"] != "").sum()),
            "with_landmark": int((tab["addr_landmark"] != "").sum()),
            "parse_method": tab["addr_parse_method"].value_counts().to_dict(),
            "top_suffixes": tab["name_suffix"].replace("", pd.NA).dropna().value_counts().head(15).to_dict(),
        }
    log.info("Normalised %d S1 and %d vendor records (%d S2 / %d S3)", len(s1), len(v),
             int((v["source"] == 2).sum()), int((v["source"] == 3).sum()))
    return s1, v, stats


In [ ]:
%%writefile src/labels.py
"""Ground-truth handling: resolve label ids to records, build positive pairs,
attach labels to candidate pairs, tag hard negatives.

Label format (transcript): one row per Source-1 entity; its id maps to a
comma-separated list of ALL its matching Source-2/3 ids; empty list = no match.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from .schema import SchemaError, label_schema
from .utils import get_logger


@dataclass
class LabelIndex:
    key_mode: str                         # "id" (combined column) or "source_id" (per-source columns)
    true_keys: list[set]                  # per s1_idx: set of metric keys (strings)
    labeled: np.ndarray                   # per s1_idx: bool, entity usable for training/eval
    true_pairs: pd.DataFrame              # [s1_idx, v_idx] resolved positives
    stats: dict = field(default_factory=dict)

    @property
    def n_true(self) -> np.ndarray:
        return np.array([len(s) for s in self.true_keys], dtype=np.int64)

    def is_singleton(self) -> np.ndarray:
        return self.n_true == 0


def metric_key_series(v: pd.DataFrame, key_mode: str) -> pd.Series:
    if key_mode == "id":
        return v["id"]
    return v["source"].astype(str) + ":" + v["id"]


def build_label_index(labels_df: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame, cfg: dict) -> LabelIndex:
    log = get_logger()
    schema = label_schema(cfg)
    key_mode = "source_id" if schema.column_sources else "id"
    id2s1 = dict(zip(s1["id"], s1["s1_idx"]))
    ids_by_source = {s: dict(zip(v.loc[v["source"] == s, "id"], v.loc[v["source"] == s, "v_idx"]))
                     for s in (2, 3)}
    true_keys: list[set] = [set() for _ in range(len(s1))]
    labeled = np.zeros(len(s1), dtype=bool)
    pos_rows = []
    unknown_s1, unknown_ids, ambiguous_ids = [], [], []
    for _, row in labels_df.iterrows():
        s1_idx = id2s1.get(row["s1_id"])
        if s1_idx is None:
            unknown_s1.append(row["s1_id"])
            continue
        labeled[s1_idx] = True
        for col in schema.match_columns:
            for mid in row[col]:
                if schema.column_sources:
                    src = schema.column_sources.get(col)
                    if src not in (2, 3):
                        raise SchemaError(f"column_sources must map {col} to 2 or 3")
                    true_keys[s1_idx].add(f"{src}:{mid}")
                    v_idx = ids_by_source[src].get(mid)
                    if v_idx is None:
                        unknown_ids.append(mid)
                    else:
                        pos_rows.append((s1_idx, v_idx))
                    continue
                true_keys[s1_idx].add(mid)
                in2, in3 = mid in ids_by_source[2], mid in ids_by_source[3]
                if in2 and in3:
                    ambiguous_ids.append(mid)
                    if schema.ambiguous_id_policy == "error":
                        raise SchemaError(f"Label id {mid!r} exists in both Source 2 and Source 3")
                    if schema.ambiguous_id_policy == "both":
                        pos_rows.append((s1_idx, ids_by_source[2][mid]))
                        pos_rows.append((s1_idx, ids_by_source[3][mid]))
                elif in2:
                    pos_rows.append((s1_idx, ids_by_source[2][mid]))
                elif in3:
                    pos_rows.append((s1_idx, ids_by_source[3][mid]))
                else:
                    unknown_ids.append(mid)
    if unknown_ids and schema.unknown_id_policy == "error":
        raise SchemaError(f"{len(unknown_ids)} label ids not found in Source 2/3, e.g. {unknown_ids[:5]}")
    missing = int((~labeled).sum())
    if missing:
        policy = schema.missing_source1_policy
        msg = f"{missing} Source-1 entities have no row in the label file"
        if policy == "error":
            raise SchemaError(msg)
        if policy == "singleton":
            log.warning("%s -> treated as singletons (empty list)", msg)
            labeled[:] = True
        else:
            log.warning("%s -> excluded from training/evaluation", msg)
    if unknown_s1:
        log.warning("%d label rows reference unknown Source-1 ids (e.g. %s)", len(unknown_s1), unknown_s1[:3])
    if unknown_ids:
        log.warning("%d label ids not found in Source 2/3 (e.g. %s); they still count in recall",
                    len(unknown_ids), unknown_ids[:3])
    if ambiguous_ids:
        log.warning("%d label ids exist in both Source 2 and 3 (policy=%s)", len(ambiguous_ids),
                    schema.ambiguous_id_policy)
    true_pairs = pd.DataFrame(pos_rows, columns=["s1_idx", "v_idx"]).drop_duplicates()
    true_pairs = true_pairs.merge(v[["v_idx", "source"]], on="v_idx", how="left")

    n_true = np.array([len(s) for s in true_keys])
    vendor_multi = true_pairs.groupby("v_idx")["s1_idx"].nunique()
    stats = {
        "key_mode": key_mode,
        "labeled_entities": int(labeled.sum()),
        "singletons": int(((n_true == 0) & labeled).sum()),
        "singleton_rate": float(((n_true == 0) & labeled).sum() / max(1, labeled.sum())),
        "positive_pairs_resolved": int(len(true_pairs)),
        "positive_pairs_source2": int((true_pairs["source"] == 2).sum()),
        "positive_pairs_source3": int((true_pairs["source"] == 3).sum()),
        "label_ids_total": int(n_true.sum()),
        "unknown_label_ids": len(unknown_ids),
        "ambiguous_label_ids": len(ambiguous_ids),
        "unknown_source1_rows": len(unknown_s1),
        # S1 is de-duplicated, so a vendor record should belong to at most one
        # S1 entity. This counts violations in the ground truth itself.
        "vendor_records_matched_to_multiple_s1": int((vendor_multi > 1).sum()),
        "matches_per_entity": {int(k): int(val) for k, val in
                               pd.Series(n_true[labeled]).value_counts().sort_index().items()},
    }
    log.info("Resolved %d positive pairs (%d S2 / %d S3); %d vendor records labelled to >1 S1 entity",
             stats["positive_pairs_resolved"], stats["positive_pairs_source2"],
             stats["positive_pairs_source3"], stats["vendor_records_matched_to_multiple_s1"])
    return LabelIndex(key_mode=key_mode, true_keys=true_keys, labeled=labeled,
                      true_pairs=true_pairs[["s1_idx", "v_idx"]], stats=stats)


def attach_labels(pairs: pd.DataFrame, li: LabelIndex, v: pd.DataFrame) -> pd.Series:
    """1 = true match, 0 = non-match, -1 = uncertain (excluded from training).

    A candidate whose id appears in the entity's label list but could not be
    resolved unambiguously is marked -1, so a true match can NEVER be used as
    a negative. Fully vectorised (no Python loop over the candidate pairs).
    """
    nv = max(1, len(v))
    s1_idx = pairs["s1_idx"].to_numpy(np.int64)
    v_idx = pairs["v_idx"].to_numpy(np.int64)
    pid = s1_idx * nv + v_idx
    tp_pid = np.sort(li.true_pairs["s1_idx"].to_numpy(np.int64) * nv + li.true_pairs["v_idx"].to_numpy(np.int64))
    out = np.zeros(len(pairs), dtype=np.int8)
    if len(tp_pid):
        pos = np.minimum(np.searchsorted(tp_pid, pid), len(tp_pid) - 1)
        out[tp_pid[pos] == pid] = 1
    # id listed in the entity's truth but not resolved to this record -> uncertain
    truth = pd.DataFrame([(e, k) for e, keys in enumerate(li.true_keys) for k in keys],
                         columns=["s1_idx", "mkey"])
    if len(truth):
        # only non-positive candidates of entities that HAVE true ids can be uncertain
        ent_has = np.zeros(len(li.true_keys), dtype=bool)
        ent_has[truth["s1_idx"].to_numpy()] = True
        rows = np.flatnonzero((out == 0) & ent_has[s1_idx])
        keys = metric_key_series(v, li.key_mode).to_numpy()
        cand = pd.DataFrame({"s1_idx": s1_idx[rows], "mkey": keys[v_idx[rows]], "row": rows})
        hit = cand.merge(truth, on=["s1_idx", "mkey"])["row"].to_numpy()
        listed = np.zeros(len(pairs), dtype=bool)
        listed[hit] = True
        out[listed & (out == 0)] = -1
    return pd.Series(out, index=pairs.index, name="label")


def missed_positive_pairs(pairs: pd.DataFrame, li: LabelIndex) -> pd.DataFrame:
    """True pairs absent from the candidate set (lost in blocking)."""
    merged = li.true_pairs.merge(pairs[["s1_idx", "v_idx"]], on=["s1_idx", "v_idx"],
                                 how="left", indicator=True)
    return merged.loc[merged["_merge"] == "left_only", ["s1_idx", "v_idx"]].reset_index(drop=True)


def tag_negative_types(feats: pd.DataFrame) -> pd.Series:
    """Categorise non-matches (for reporting and cross-encoder sampling)."""
    same_addr = (feats.get("addr_char_cos", 0) >= 0.8) | (
        (feats.get("hn_eq", 0) == 1) & (feats.get("pc_eq", 0) == 1))
    sim_name = (feats.get("name_jw", 0) >= 0.9) | (feats.get("name_char_cos", 0) >= 0.7)
    out = np.where(same_addr & sim_name, "similar_name_same_address",
                   np.where(same_addr, "shared_address",
                            np.where(sim_name, "similar_name", "other")))
    return pd.Series(out, index=feats.index, name="neg_type")


In [ ]:
%%writefile src/splits.py
"""Leakage-safe entity-level splits.

Unit of splitting: ONE Source-1 entity together with every labelled
Source-2/3 record of that entity. Because Source 1 is de-duplicated, a
business appears in exactly one split, and all of its positive pairs,
candidate pairs and features go with it.

  fit      -> trains LightGBM / bi-encoder / cross-encoder
              (fit_val = subset of fit used only for early stopping)
  calib    -> isotonic calibration, blend weights, decision search,
              singleton model
  holdout  -> never used for any fitting; unbiased final estimate

Stratified on (#S2 matches, #S3 matches) capped at 2, so each split has the
same singleton / multi-match mix.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .labels import LabelIndex


def make_splits(li: LabelIndex, v_sources: pd.Series, fractions: dict, inner_val_fraction: float,
                seed: int) -> np.ndarray:
    n = len(li.true_keys)
    tp = li.true_pairs.copy()
    tp["source"] = v_sources.to_numpy()[tp["v_idx"].to_numpy()] if len(tp) else []
    n2 = np.bincount(tp.loc[tp["source"] == 2, "s1_idx"], minlength=n) if len(tp) else np.zeros(n, int)
    n3 = np.bincount(tp.loc[tp["source"] == 3, "s1_idx"], minlength=n) if len(tp) else np.zeros(n, int)
    unresolved = li.n_true - (n2 + n3)
    strata = np.minimum(n2, 2) * 3 + np.minimum(n3, 2) + 9 * (unresolved > 0)
    names = list(fractions.keys())
    probs = np.array([fractions[k] for k in names], dtype=float)
    probs = probs / probs.sum()
    rng = np.random.default_rng(seed)
    split = np.array(["unlabeled"] * n, dtype=object)
    for s in np.unique(strata):
        idx = np.where((strata == s) & li.labeled)[0]
        rng.shuffle(idx)
        cuts = np.round(np.cumsum(probs) * len(idx)).astype(int)
        start = 0
        for name, end in zip(names, cuts):
            split[idx[start:end]] = name
            start = end
    fit_idx = np.where(split == "fit")[0]
    rng.shuffle(fit_idx)
    n_val = int(round(inner_val_fraction * len(fit_idx)))
    split[fit_idx[:n_val]] = "fit_val"
    return split


def split_summary(split: np.ndarray, li: LabelIndex) -> dict:
    out = {}
    single = li.is_singleton()
    for name in pd.unique(split):
        m = split == name
        out[str(name)] = {"entities": int(m.sum()), "singletons": int((single & m).sum()),
                          "true_matches": int(li.n_true[m].sum())}
    return out


In [ ]:
%%writefile src/metrics.py
"""Evaluation metrics.

Challenge metric (transcript): Macro F0.5 over Source-1 entities. For each
Source-1 entity the predicted SET of Source-2/3 ids is compared with the true
set; F-beta (beta = 0.5, precision weighted twice as heavily as recall) is
computed per entity and averaged with equal weight.

    F_beta = (1 + b^2) TP / ((1 + b^2) TP + b^2 FN + FP)

Special case (transcript): a singleton (true set empty) scores 1.0 when the
prediction is empty and 0.0 otherwise. A non-singleton predicted empty scores 0.
The formula above already gives these values except the empty/empty case,
which is set to 1 explicitly.
"""
from __future__ import annotations

import numpy as np


def entity_fbeta(tp, n_pred, n_true, beta: float = 0.5) -> np.ndarray:
    tp = np.asarray(tp, dtype=float)
    n_pred = np.asarray(n_pred, dtype=float)
    n_true = np.asarray(n_true, dtype=float)
    fp = n_pred - tp
    fn = n_true - tp
    b2 = beta * beta
    denom = (1 + b2) * tp + b2 * fn + fp
    with np.errstate(divide="ignore", invalid="ignore"):
        f = np.where(denom > 0, (1 + b2) * tp / np.where(denom > 0, denom, 1), 1.0)
    return f


def macro_fbeta_sets(pred: dict, true: dict, entities, beta: float = 0.5) -> dict:
    """Reference implementation on Python sets (used by the scorer and tests)."""
    tp, npred, ntrue = [], [], []
    for e in entities:
        p = set(pred.get(e, ()))
        t = set(true.get(e, ()))
        tp.append(len(p & t))
        npred.append(len(p))
        ntrue.append(len(t))
    return summarize(np.array(tp), np.array(npred), np.array(ntrue), beta)


def summarize(tp: np.ndarray, n_pred: np.ndarray, n_true: np.ndarray, beta: float = 0.5) -> dict:
    tp = np.asarray(tp, dtype=float)
    n_pred = np.asarray(n_pred, dtype=float)
    n_true = np.asarray(n_true, dtype=float)
    f = entity_fbeta(tp, n_pred, n_true, beta)
    n = len(f)
    single = n_true == 0
    pred_empty = n_pred == 0
    micro_p = tp.sum() / n_pred.sum() if n_pred.sum() else 1.0
    micro_r = tp.sum() / n_true.sum() if n_true.sum() else 1.0
    b2 = beta * beta
    micro_f = ((1 + b2) * micro_p * micro_r / (b2 * micro_p + micro_r)) if (micro_p + micro_r) else 0.0
    with np.errstate(divide="ignore", invalid="ignore"):
        ent_p = np.where(n_pred > 0, tp / np.maximum(n_pred, 1), np.nan)
        ent_r = np.where(n_true > 0, tp / np.maximum(n_true, 1), np.nan)
    return {
        "macro_f0.5" if beta == 0.5 else f"macro_f{beta}": float(f.mean()) if n else float("nan"),
        "entities": int(n),
        "entity_precision_mean(pred_nonempty)": float(np.nanmean(ent_p)) if np.any(n_pred > 0) else float("nan"),
        "entity_recall_mean(true_nonempty)": float(np.nanmean(ent_r)) if np.any(n_true > 0) else float("nan"),
        "pair_precision": float(micro_p),
        "pair_recall": float(micro_r),
        "pair_f0.5": float(micro_f),
        "true_singletons": int(single.sum()),
        "predicted_singletons": int(pred_empty.sum()),
        "singleton_correct(empty_pred)": int((single & pred_empty).sum()),
        "singleton_false_merges": int((single & ~pred_empty).sum()),
        "singleton_accuracy": float((single & pred_empty).sum() / single.sum()) if single.any() else float("nan"),
        "nonsingleton_predicted_empty": int((~single & pred_empty).sum()),
        "nonsingleton_exact_set": int((~single & (tp == n_true) & (n_pred == n_true)).sum()),
        "macro_f_singletons": float(f[single].mean()) if single.any() else float("nan"),
        "macro_f_nonsingletons": float(f[~single].mean()) if (~single).any() else float("nan"),
        "predicted_matches": int(n_pred.sum()),
        "true_matches": int(n_true.sum()),
    }


def macro_from_arrays(ent: np.ndarray, selected: np.ndarray, is_true: np.ndarray,
                      n_true: np.ndarray, beta: float = 0.5) -> float:
    """Fast Macro F-beta. `ent` = dense entity codes 0..E-1 for each pair,
    `n_true` = true-set sizes for ALL E entities (including entities that have
    no candidate at all and true matches lost in blocking)."""
    E = len(n_true)
    sel = selected.astype(bool)
    tp = np.bincount(ent[sel & is_true], minlength=E)
    npred = np.bincount(ent[sel], minlength=E)
    return float(entity_fbeta(tp, npred, n_true, beta).mean())


def pairwise_prf(selected: np.ndarray, y: np.ndarray, beta: float = 0.5) -> dict:
    sel = selected.astype(bool)
    y = y.astype(bool)
    tp = int((sel & y).sum())
    fp = int((sel & ~y).sum())
    fn = int((~sel & y).sum())
    p = tp / (tp + fp) if tp + fp else 1.0
    r = tp / (tp + fn) if tp + fn else 1.0
    b2 = beta * beta
    f = (1 + b2) * p * r / (b2 * p + r) if (p + r) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, f"f{beta}": f}


def reduction_ratio(n_candidates: int, n_possible: int) -> float:
    return 1.0 - n_candidates / n_possible if n_possible else float("nan")


In [ ]:
%%writefile src/blocking.py
"""Hybrid candidate generation (blocking).

PDF "Hybrid Candidate Generation": a syntactic path (exact keys, phonetic
hashing, LSH/MinHash, TF-IDF) and a semantic path (SC-Block bi-encoder +
FAISS, see train_biencoder.py). The UNION of all paths is the candidate set
written to candidate_pairs.tsv. Transcript: "records can group either through
a similar name or through a shared address" - so both name-driven and
address-driven blocks are used.

Every method retrieves Source-2 and Source-3 candidates SEPARATELY for each
Source-1 entity (the task is Source-1-centric). Methods:

  name_prefix        first `prefix_length` chars of the normalised core name
  postcode           exact normalised postcode
  postcode_phonetic  postcode + phonetic code of the first name token (PDF)
  city_state         parsed city + state
  phonetic_name      metaphone code of the whole core name
  address_key        house number + first significant road token ("shared address")
  tfidf_name_char    char n-gram TF-IDF cosine top-k on the name
  tfidf_name_word    word TF-IDF cosine top-k on the name
  tfidf_full_char    char n-gram TF-IDF top-k on name + address
  tfidf_address      char n-gram TF-IDF top-k on the address only
  minhash_lsh        MinHash signatures of name shingles, banded LSH buckets
  semantic           bi-encoder embeddings + FAISS top-k (full mode)

Key blocks larger than `top_k` are ranked inside the block by TF-IDF cosine
(so a big postcode block keeps the k most name-similar records); blocks larger
than `max_block_size` are purged (standard block purging). Finally every
(entity, source) keeps at most `max_candidates_per_source` pairs ranked by a
cheap similarity. TF-IDF statistics are fitted on the unlabelled text of the
dataset being blocked; no label is ever used for blocking.
"""
from __future__ import annotations

import gc
import zlib

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

from .config import get, semantic_enabled
from .metrics import entity_fbeta, reduction_ratio
from .utils import describe_counts, get_logger, mem_str

METHODS = ["name_prefix", "postcode", "postcode_phonetic", "city_state", "phonetic_name",
           "address_key", "tfidf_name_char", "tfidf_name_word", "tfidf_full_char",
           "tfidf_address", "minhash_lsh", "semantic"]
METHOD_BIT = {m: 1 << i for i, m in enumerate(METHODS)}


# ---------------------------------------------------------------- TF-IDF
def _fit_tfidf(texts_a: pd.Series, texts_b: pd.Series, **kwargs):
    corpus = pd.concat([texts_a, texts_b], ignore_index=True).fillna("")
    vec = TfidfVectorizer(dtype=np.float32, sublinear_tf=True, lowercase=False, **kwargs)
    try:
        vec.fit(corpus)
    except ValueError:  # vocabulary pruned away (tiny corpus) -> no max_df
        kwargs = {k: val for k, val in kwargs.items() if k != "max_df"}
        vec = TfidfVectorizer(dtype=np.float32, sublinear_tf=True, lowercase=False, **kwargs)
        try:
            vec.fit(corpus)
        except ValueError:
            return None, None, None
    return vec, vec.transform(texts_a.fillna("")).tocsr(), vec.transform(texts_b.fillna("")).tocsr()


class TextRepresentations:
    """Sparse, L2-normalised TF-IDF matrices for S1 and vendor records."""

    def __init__(self, s1: pd.DataFrame, v: pd.DataFrame, ngram_range=(2, 4), char_max_df=1.0,
                 word_max_df=1.0):
        ngram_range = tuple(ngram_range)
        self.mats: dict[str, tuple] = {}
        specs = {
            "name_char": ("name_tokens", dict(analyzer="char_wb", ngram_range=ngram_range, max_df=char_max_df)),
            "name_word": ("name_tokens", dict(analyzer="word", token_pattern=r"\S+", max_df=word_max_df)),
            "full_char": ("full_text", dict(analyzer="char_wb", ngram_range=ngram_range, max_df=char_max_df)),
            "addr_char": ("addr_clean", dict(analyzer="char_wb", ngram_range=ngram_range, max_df=char_max_df)),
        }
        for name, (col, kw) in specs.items():
            _, a, b = _fit_tfidf(s1[col], v[col], **kw)
            if a is None:
                a = sparse.csr_matrix((len(s1), 1), dtype=np.float32)
                b = sparse.csr_matrix((len(v), 1), dtype=np.float32)
            self.mats[name] = (a, b)

    def s1(self, name):
        return self.mats[name][0]

    def v(self, name):
        return self.mats[name][1]


def rowwise_cosine(A: sparse.csr_matrix, B: sparse.csr_matrix, ia: np.ndarray, ib: np.ndarray,
                   chunk: int = 200_000) -> np.ndarray:
    out = np.zeros(len(ia), dtype=np.float32)
    for start in range(0, len(ia), chunk):
        sl = slice(start, start + chunk)
        prod = A[ia[sl]].multiply(B[ib[sl]])
        out[sl] = np.asarray(prod.sum(axis=1)).ravel()
    return out


def sparse_topk(A: sparse.csr_matrix, B: sparse.csr_matrix, k: int, chunk: int = 2000,
                min_sim: float = 1e-6) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Top-k rows of B for every row of A by cosine (both L2-normalised)."""
    rows, cols, sims = [], [], []
    BT = B.T.tocsr()
    for start in range(0, A.shape[0], chunk):
        S = (A[start:start + chunk] @ BT).tocsr()
        indptr, indices, data = S.indptr, S.indices, S.data
        for r in range(S.shape[0]):
            lo, hi = indptr[r], indptr[r + 1]
            if hi == lo:
                continue
            d = data[lo:hi]
            if hi - lo > k:
                sel = np.argpartition(-d, k - 1)[:k]
            else:
                sel = np.arange(hi - lo)
            sel = sel[d[sel] >= min_sim]
            rows.append(np.full(len(sel), start + r, dtype=np.int64))
            cols.append(indices[lo:hi][sel].astype(np.int64))
            sims.append(d[sel])
    if not rows:
        return np.zeros(0, np.int64), np.zeros(0, np.int64), np.zeros(0, np.float32)
    return np.concatenate(rows), np.concatenate(cols), np.concatenate(sims)


# ------------------------------------------------------------- key blocks
def key_block(s1_keys: np.ndarray, v_keys: np.ndarray, v_global: np.ndarray,
              rank_s1: sparse.csr_matrix, rank_v: sparse.csr_matrix, top_k: int,
              max_block: int) -> tuple[pd.DataFrame, dict]:
    """Candidates sharing an exact key; oversized blocks ranked / purged.

    `v_keys`, `rank_v` and `v_global` are aligned (vendor rows of ONE source).
    """
    s1df = pd.DataFrame({"s1_idx": np.arange(len(s1_keys)), "key": s1_keys})
    s1df = s1df[s1df["key"] != ""]
    vdf = pd.DataFrame({"vpos": np.arange(len(v_keys)), "key": v_keys})
    vdf = vdf[vdf["key"] != ""]
    counts = vdf["key"].value_counts()
    counts = counts[counts.index.isin(set(s1df["key"]))]
    small = set(counts.index[counts <= top_k])
    mid = set(counts.index[(counts > top_k) & (counts <= max_block)])
    purged = counts[counts > max_block]
    parts = []
    if small:
        m = s1df[s1df["key"].isin(small)].merge(vdf[vdf["key"].isin(small)], on="key")
        parts.append(m[["s1_idx", "vpos"]].astype(np.int32))
    if mid:
        s1_groups = s1df[s1df["key"].isin(mid)].groupby("key")["s1_idx"].apply(np.asarray)
        v_groups = vdf[vdf["key"].isin(mid)].groupby("key")["vpos"].apply(np.asarray)
        for key, s1_rows in s1_groups.items():
            v_rows = v_groups[key]
            v_block_T = rank_v[v_rows].T.tocsr()
            kk = min(top_k, len(v_rows))
            # bound the dense (S1 rows x block) score matrix to ~2M cells
            row_chunk = max(1, int(2_000_000 // max(1, len(v_rows))))
            for c in range(0, len(s1_rows), row_chunk):
                rows = s1_rows[c:c + row_chunk]
                S = (rank_s1[rows] @ v_block_T).toarray()
                top = np.argpartition(-S, kk - 1, axis=1)[:, :kk]
                parts.append(pd.DataFrame({"s1_idx": np.repeat(rows, kk).astype(np.int32),
                                           "vpos": v_rows[top].ravel().astype(np.int32)}))
    stats = {"blocks": int(len(counts)), "blocks_ranked": len(mid), "blocks_purged": int(len(purged)),
             "vendor_rows_in_purged_blocks": int(purged.sum())}
    if not parts:
        return pd.DataFrame({"s1_idx": [], "v_idx": []}, dtype=np.int64), stats
    out = pd.concat(parts, ignore_index=True)
    return pd.DataFrame({"s1_idx": out["s1_idx"].to_numpy(np.int32),
                         "v_idx": v_global[out["vpos"].to_numpy(np.int64)].astype(np.int32)}), stats


# ---------------------------------------------------------------- MinHash
_MERSENNE = (1 << 31) - 1


def _shingles(texts, k: int) -> tuple[np.ndarray, np.ndarray]:
    flat, lens = [], []
    for t in texts:
        if not t:
            lens.append(0)
            continue
        sh = {t[i:i + k] for i in range(max(1, len(t) - k + 1))}
        flat.extend(zlib.crc32(x.encode("utf-8")) for x in sh)
        lens.append(len(sh))
    return np.asarray(flat, dtype=np.int64), np.asarray(lens, dtype=np.int64)


def minhash_signatures(texts, num_perm: int = 64, k: int = 3, seed: int = 1) -> tuple[np.ndarray, np.ndarray]:
    """Vectorised MinHash (universal hashing a*x+b mod p). Returns (sig, valid)."""
    flat, lens = _shingles(texts, k)
    n = len(lens)
    rng = np.random.default_rng(seed)
    a = rng.integers(1, _MERSENNE, num_perm, dtype=np.int64)
    b = rng.integers(0, _MERSENNE, num_perm, dtype=np.int64)
    sig = np.full((n, num_perm), _MERSENNE, dtype=np.int64)
    valid = lens > 0
    if flat.size == 0:
        return sig, valid
    x = flat % _MERSENNE
    offsets = np.concatenate([[0], np.cumsum(lens)])[:-1][valid]
    step = max(1, int(2e7 // max(1, x.size)))
    for s in range(0, num_perm, step):
        cols = slice(s, min(num_perm, s + step))
        h = (x[:, None] * a[None, cols] + b[None, cols]) % _MERSENNE
        sig[valid, cols] = np.minimum.reduceat(h, offsets, axis=0)
    return sig, valid


def _band_keys(sig: np.ndarray, bands: int) -> np.ndarray:
    rows = sig.shape[1] // bands
    rng = np.random.default_rng(12345)
    mult = rng.integers(1, 2**62, rows, dtype=np.int64).astype(np.uint64) | np.uint64(1)
    keys = np.empty((sig.shape[0], bands), dtype=np.uint64)
    u = sig.astype(np.uint64)
    with np.errstate(over="ignore"):
        for bi in range(bands):
            keys[:, bi] = (u[:, bi * rows:(bi + 1) * rows] * mult).sum(axis=1)
    return keys


def minhash_block(s1_texts, v_texts, v_global: np.ndarray, top_k: int, num_perm: int, bands: int,
                  k: int, max_bucket: int, s1_chunk: int = 20_000) -> tuple[pd.DataFrame, dict]:
    """Banded MinHash-LSH. Memory-bounded: Source-1 records are processed in
    chunks; for each chunk the bucket collisions of ALL bands are gathered,
    de-duplicated, scored by estimated Jaccard and cut to top-k before the
    next chunk starts (so pairs never pile up across bands)."""
    sig1, ok1 = minhash_signatures(list(s1_texts), num_perm, k)
    sig2, ok2 = minhash_signatures(list(v_texts), num_perm, k)
    keys1, keys2 = _band_keys(sig1, bands), _band_keys(sig2, bands)
    idx1, idx2 = np.where(ok1)[0], np.where(ok2)[0]
    purged = 0
    vendor_bands = []
    for bi in range(bands):
        d2 = pd.DataFrame({"vpos": idx2.astype(np.int32), "key": keys2[idx2, bi]})
        cnt = d2["key"].value_counts()
        big = cnt.index[cnt > max_bucket]
        purged += len(big)
        vendor_bands.append(d2[~d2["key"].isin(big)])
    out = []
    for c in range(0, len(idx1), s1_chunk):
        rows = idx1[c:c + s1_chunk]
        parts = []
        for bi in range(bands):
            d1 = pd.DataFrame({"s1_idx": rows.astype(np.int32), "key": keys1[rows, bi]})
            parts.append(d1.merge(vendor_bands[bi], on="key")[["s1_idx", "vpos"]])
        pairs = pd.concat(parts, ignore_index=True).drop_duplicates()
        del parts
        if pairs.empty:
            continue
        a = pairs["s1_idx"].to_numpy()
        bpos = pairs["vpos"].to_numpy()
        est = np.empty(len(pairs), dtype=np.float32)
        for s in range(0, len(pairs), 200_000):
            est[s:s + 200_000] = (sig1[a[s:s + 200_000]] == sig2[bpos[s:s + 200_000]]).mean(axis=1)
        pairs = pairs.assign(sim=est).sort_values(["s1_idx", "sim"], ascending=[True, False])
        out.append(pairs.groupby("s1_idx", sort=False).head(top_k)[["s1_idx", "vpos"]])
    if not out:
        return pd.DataFrame({"s1_idx": np.zeros(0, np.int32), "v_idx": np.zeros(0, np.int32)}), \
            {"buckets_purged": int(purged)}
    res = pd.concat(out, ignore_index=True)
    return (pd.DataFrame({"s1_idx": res["s1_idx"].to_numpy(np.int32),
                          "v_idx": v_global[res["vpos"].to_numpy(np.int64)].astype(np.int32)}),
            {"buckets_purged": int(purged)})


# ---------------------------------------------------------------- blocker
class Blocker:
    def __init__(self, cfg: dict):
        self.cfg = cfg
        self.bcfg = get(cfg, "blocking", {})
        self.methods_cfg = self.bcfg.get("methods", {})

    def _enabled(self, method: str) -> bool:
        if method == "semantic":
            return semantic_enabled(self.cfg)
        return bool(self.methods_cfg.get(method, {}).get("enabled", False))

    def _k(self, method: str, default: int = 20) -> int:
        return int(self.methods_cfg.get(method, {}).get("top_k", default))

    def generate(self, s1: pd.DataFrame, v: pd.DataFrame, semantic_retriever=None):
        """Return (candidates, precap_pairs, stats).

        candidates: [s1_idx, v_idx, source, blk_<method>..., n_methods, blk_score]
        """
        log = get_logger()
        b = self.bcfg
        reps = TextRepresentations(s1, v, b.get("char_ngram_range", [2, 4]),
                                   b.get("char_max_df", 0.05), b.get("word_max_df", 0.10))
        prefix_len = int(b.get("prefix_length", 4))
        max_block = int(b.get("max_block_size", 2000))
        chunk = int(b.get("chunk_size", 2000))
        stats: dict = {"methods": {}}

        s1_keys = {
            "name_prefix": np.where(s1["name_nospace"].str.len() >= 2,
                                    s1["name_nospace"].str[:prefix_len], "").astype(object),
            "postcode": s1["addr_postcode"].to_numpy(object),
            "postcode_phonetic": np.where((s1["addr_postcode"] != "") & (s1["name_first_phonetic"] != ""),
                                          s1["addr_postcode"] + "|" + s1["name_first_phonetic"], "").astype(object),
            "city_state": s1["city_state_key"].to_numpy(object),
            "phonetic_name": s1["name_phonetic"].str[:10].to_numpy(object),
            "address_key": s1["addr_street_key"].to_numpy(object),
        }
        v_keys_all = {
            "name_prefix": np.where(v["name_nospace"].str.len() >= 2,
                                    v["name_nospace"].str[:prefix_len], "").astype(object),
            "postcode": v["addr_postcode"].to_numpy(object),
            "postcode_phonetic": np.where((v["addr_postcode"] != "") & (v["name_first_phonetic"] != ""),
                                          v["addr_postcode"] + "|" + v["name_first_phonetic"], "").astype(object),
            "city_state": v["city_state_key"].to_numpy(object),
            "phonetic_name": v["name_phonetic"].str[:10].to_numpy(object),
            "address_key": v["addr_street_key"].to_numpy(object),
        }
        # ranking representation inside oversized key blocks
        rank_rep = {"name_prefix": "full_char", "postcode": "name_char", "postcode_phonetic": "name_char",
                    "city_state": "name_char", "phonetic_name": "full_char", "address_key": "addr_char"}
        tfidf_rep = {"tfidf_name_char": "name_char", "tfidf_name_word": "name_word",
                     "tfidf_full_char": "full_char", "tfidf_address": "addr_char"}

        cap = int(b.get("max_candidates_per_source", 60))
        bonus = float(b.get("cap_method_bonus", 0.02))
        mh_chunk = int(self.methods_cfg.get("minhash_lsh", {}).get("s1_chunk", 20_000))
        capped_parts, precap_parts = [], []
        for source in (2, 3):
            vmask = (v["source"] == source).to_numpy()
            v_global = np.where(vmask)[0]
            if len(v_global) == 0:
                continue
            frames = []
            for method in METHODS:
                if not self._enabled(method):
                    continue
                k = self._k(method)
                mstats = {}
                if method in s1_keys:
                    rep = rank_rep[method]
                    df, mstats = key_block(s1_keys[method], v_keys_all[method][vmask], v_global,
                                           reps.s1(rep), reps.v(rep)[v_global], k, max_block)
                elif method in tfidf_rep:
                    rep = tfidf_rep[method]
                    r, c, _ = sparse_topk(reps.s1(rep), reps.v(rep)[v_global], k, chunk)
                    df = pd.DataFrame({"s1_idx": r.astype(np.int32), "v_idx": v_global[c].astype(np.int32)})
                elif method == "minhash_lsh":
                    mc = self.methods_cfg.get("minhash_lsh", {})
                    df, mstats = minhash_block(s1["name_nospace"], v.loc[vmask, "name_nospace"], v_global, k,
                                               int(mc.get("num_perm", 64)), int(mc.get("bands", 16)),
                                               int(mc.get("shingle_size", 3)),
                                               int(mc.get("max_bucket_size", 300)), mh_chunk)
                elif method == "semantic":
                    if semantic_retriever is None:
                        log.warning("semantic blocking enabled but no retriever supplied - skipped")
                        continue
                    df = semantic_retriever.search(s1, v, source, k)[["s1_idx", "v_idx"]].astype(np.int32)
                else:  # pragma: no cover
                    continue
                df = df.astype(np.int32)
                df["bit"] = np.int32(METHOD_BIT[method])
                frames.append(df)
                stats["methods"][f"{method}_s{source}"] = {"pairs": int(len(df)), **mstats}
                log.info("  blocking %-18s source %d -> %9d pairs  [%s]", method, source, len(df), mem_str())
            if not frames:
                continue
            # union of this source's methods -> one row per pair with a bit mask
            allp = pd.concat(frames, ignore_index=True)
            del frames
            allp = allp.drop_duplicates(["s1_idx", "v_idx", "bit"])
            cs = allp.groupby(["s1_idx", "v_idx"], sort=False)["bit"].sum().reset_index()
            del allp
            gc.collect()
            cs.rename(columns={"bit": "blk_flags"}, inplace=True)
            cs["source"] = np.int8(source)
            ia, ib = cs["s1_idx"].to_numpy(np.int64), cs["v_idx"].to_numpy(np.int64)
            # Cap ranking: joint name+address similarity plus a small bonus per
            # independent method that retrieved the pair. (Ranking by name
            # similarity alone lets many same-name branches of a common business
            # push the true, address-matching record out of the cap.)
            flags = cs["blk_flags"].to_numpy()
            n_methods = sum(((flags & bit) > 0).astype(np.float32) for bit in METHOD_BIT.values())
            cs["blk_score"] = (rowwise_cosine(reps.s1("full_char"), reps.v("full_char"), ia, ib)
                               + bonus * n_methods).astype(np.float32)
            precap_parts.append(cs[["s1_idx", "v_idx", "source", "blk_flags"]].copy())
            cs = cs.sort_values(["s1_idx", "blk_score"], ascending=[True, False])
            capped_parts.append(cs.groupby("s1_idx", sort=False).head(cap).reset_index(drop=True))
            log.info("  source %d union: %d pairs before cap, %d after cap  [%s]", source,
                     len(cs), len(capped_parts[-1]), mem_str())
            del cs
            gc.collect()

        if not capped_parts:
            raise RuntimeError("No blocking method produced candidates - check blocking.methods")
        del reps
        cand = pd.concat(capped_parts, ignore_index=True)
        precap = pd.concat(precap_parts, ignore_index=True)
        del capped_parts, precap_parts
        cand["s1_idx"] = cand["s1_idx"].astype(np.int64)
        cand["v_idx"] = cand["v_idx"].astype(np.int64)
        for m in METHODS:
            cand[f"blk_{m}"] = ((cand["blk_flags"].to_numpy() & METHOD_BIT[m]) > 0).astype(np.int8)
        cand["n_methods"] = cand[[f"blk_{m}" for m in METHODS]].sum(axis=1).astype(np.int16)
        cand = cand.sort_values(["s1_idx", "source", "v_idx"]).reset_index(drop=True)
        stats["pairs_before_cap"] = int(len(precap))
        stats["pairs_after_cap"] = int(len(cand))
        log.info("Blocking union: %d pairs before cap, %d after cap (%d per source)  [%s]",
                 len(precap), len(cand), cap, mem_str())
        return cand, precap, stats


# ------------------------------------------------------------ evaluation
def _in_sorted(values: np.ndarray, sorted_ref: np.ndarray) -> np.ndarray:
    """Vectorised membership test of int64 pair ids against a sorted array."""
    if len(sorted_ref) == 0:
        return np.zeros(len(values), dtype=bool)
    pos = np.minimum(np.searchsorted(sorted_ref, values), len(sorted_ref) - 1)
    return sorted_ref[pos] == values


def blocking_report(cand: pd.DataFrame, precap: pd.DataFrame | None, s1: pd.DataFrame, v: pd.DataFrame,
                    li=None, split: np.ndarray | None = None, beta: float = 0.5) -> dict:
    """Pair completeness, reduction ratio, candidate distribution, lost matches,
    per-method contribution and the Macro-F0.5 ceiling imposed by blocking."""
    n1 = len(s1)
    n2 = int((v["source"] == 2).sum())
    n3 = int((v["source"] == 3).sum())
    possible = n1 * (n2 + n3)
    rep: dict = {
        "source1_records": n1, "source2_records": n2, "source3_records": n3,
        "possible_pairs": possible, "possible_pairs_s2": n1 * n2, "possible_pairs_s3": n1 * n3,
        "candidate_pairs": int(len(cand)),
        "candidate_pairs_s2": int((cand["source"] == 2).sum()),
        "candidate_pairs_s3": int((cand["source"] == 3).sum()),
    }
    rep["reduction_ratio"] = reduction_ratio(len(cand), possible)
    rep["reduction_ratio_s2"] = reduction_ratio(rep["candidate_pairs_s2"], n1 * n2)
    rep["reduction_ratio_s3"] = reduction_ratio(rep["candidate_pairs_s3"], n1 * n3)
    per_ent = cand.groupby("s1_idx").size().reindex(range(n1), fill_value=0)
    rep["candidates_per_entity"] = describe_counts(per_ent.to_numpy())
    rep["entities_without_candidates"] = int((per_ent == 0).sum())
    for s in (2, 3):
        pe = cand[cand["source"] == s].groupby("s1_idx").size().reindex(range(n1), fill_value=0)
        rep[f"candidates_per_entity_s{s}"] = describe_counts(pe.to_numpy())
    rep["candidate_count_histogram"] = {
        str(k): int(val) for k, val in pd.cut(per_ent, [-1, 0, 5, 10, 20, 50, 100, 200, 10**9]).value_counts(sort=False).items()}
    if li is None:
        return rep

    tp = li.true_pairs.merge(v[["v_idx", "source"]], on="v_idx")
    nv = max(1, len(v))
    tp_pid = tp["s1_idx"].to_numpy(np.int64) * nv + tp["v_idx"].to_numpy(np.int64)
    cand_pid = np.sort(cand["s1_idx"].to_numpy(np.int64) * nv + cand["v_idx"].to_numpy(np.int64))
    found = _in_sorted(tp_pid, cand_pid)
    del cand_pid
    tp = tp.assign(found=found)
    rep["true_pairs_resolved"] = int(len(tp))
    rep["pair_completeness"] = float(found.mean()) if len(tp) else float("nan")
    rep["true_pairs_lost"] = int((~found).sum())
    for s in (2, 3):
        m = tp["source"] == s
        rep[f"pair_completeness_s{s}"] = float(tp.loc[m, "found"].mean()) if m.any() else float("nan")
    if precap is not None:
        pre_pid = precap["s1_idx"].to_numpy(np.int64) * nv + precap["v_idx"].to_numpy(np.int64)
        order = np.argsort(pre_pid)
        pre_sorted, flags_sorted = pre_pid[order], precap["blk_flags"].to_numpy()[order]
        del pre_pid, order
        pos = np.searchsorted(pre_sorted, tp_pid)
        pos_ok = pos < len(pre_sorted)
        hit = np.zeros(len(tp_pid), dtype=bool)
        hit[pos_ok] = pre_sorted[pos[pos_ok]] == tp_pid[pos_ok]
        tp_flags = np.where(hit, flags_sorted[np.minimum(pos, max(0, len(pre_sorted) - 1))], 0).astype(np.int64) \
            if len(pre_sorted) else np.zeros(len(tp_pid), np.int64)
        rep["pair_completeness_before_cap"] = float(hit.mean()) if len(tp) else float("nan")
        rep["pairs_before_cap"] = int(len(precap))
        pc_flags = precap["blk_flags"].to_numpy()
        method_stats = {}
        for m in METHODS:
            bit = METHOD_BIT[m]
            has = (tp_flags & bit) > 0
            only = has & (tp_flags == bit)
            method_stats[m] = {
                "candidate_pairs": int(((pc_flags & bit) > 0).sum()),
                "recall": float(has.mean()) if len(tp) else float("nan"),
                "unique_true_pairs": int(only.sum()),
            }
        rep["per_method"] = method_stats

    # entity-level view and the ceiling blocking imposes on Macro F0.5
    n_true = li.n_true
    found_per_ent = np.bincount(tp.loc[tp["found"], "s1_idx"], minlength=n1)
    oracle = entity_fbeta(found_per_ent, found_per_ent, n_true, beta)
    labeled = li.labeled
    nonsingle = (n_true > 0) & labeled
    rep["nonsingleton_entities"] = int(nonsingle.sum())
    rep["nonsingleton_entities_all_matches_lost"] = int((nonsingle & (found_per_ent == 0)).sum())
    rep["oracle_macro_f0.5_given_blocking"] = float(oracle[labeled].mean()) if labeled.any() else float("nan")
    if split is not None:
        per_split = {}
        ent_split = split[tp["s1_idx"].to_numpy()] if len(tp) else np.array([])
        for name in pd.unique(split):
            ms = ent_split == name
            em = (split == name) & labeled
            per_split[str(name)] = {
                "pair_completeness": float(tp.loc[ms, "found"].mean()) if ms.any() else float("nan"),
                "oracle_macro_f0.5": float(oracle[em].mean()) if em.any() else float("nan"),
                "entities": int(em.sum()),
            }
        rep["per_split"] = per_split
    return rep


In [ ]:
%%writefile src/features.py
"""Pairwise features for the structural matcher (PDF: "Structural Matching via
Gradient Boosted Trees").

* String similarities optimised for short administrative text: Jaro-Winkler
  (prefix scale p = 0.1, prefix capped at 4 - as in the PDF), Levenshtein,
  token-sort/set ratios, char n-gram TF-IDF cosine.
* Soft-TFIDF (Cohen et al.): tokens align when their secondary similarity
  (Jaro-Winkler by default, Levenshtein optional) >= 0.90, weighted by IDF so
  frequent administrative words count little and rare tokens a lot.
* Exact-match boolean flags on parsed address components (house number,
  road, city, state, postcode, unit) + conflict and missing indicators.
* Legal-suffix agreement / conflict (the sequestered suffix column).
* Combined and contradiction features.
* Label-free context features computed from the candidate set only
  (rank of the pair among the entity's candidates, margin to the best
  alternative, reciprocal-best flag). They use no labels and no model output,
  so they are identical at training and inference time.

Missing information is encoded as NaN (LightGBM routes NaN natively).
"""
from __future__ import annotations

import gc
import math
import os
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from contextlib import contextmanager

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import Indel, JaroWinkler, Levenshtein
from rapidfuzz.process import cpdist

from .blocking import METHODS, TextRepresentations, rowwise_cosine
from .utils import ensure_dir, get_logger, load_json, mem_str, resolve_n_jobs, save_json

# Blocking flags usable as features. The semantic flag is excluded: the
# bi-encoder is trained on the fit split, so on fit pairs its retrieval is
# in-sample and the flag would leak label information into LightGBM.
BLOCKING_FEATURE_METHODS = [m for m in METHODS if m != "semantic"]


class FeatureContext:
    """Corpus statistics of ONE dataset (train or test): unpruned TF-IDF
    matrices and token IDF tables. Unsupervised - no labels involved."""

    def __init__(self, s1: pd.DataFrame, v: pd.DataFrame, cfg: dict):
        self.reps = TextRepresentations(s1, v, ngram_range=(2, 4), char_max_df=1.0, word_max_df=1.0)
        self.name_idf = self._idf(pd.concat([s1["name_tokens"], v["name_tokens"]]))
        self.addr_idf = self._idf(pd.concat([s1["addr_clean"], v["addr_clean"]]))
        fcfg = cfg.get("features", {})
        self._cols: dict = {}
        self.soft_thr = float(fcfg.get("soft_tfidf_threshold", 0.9))
        self.soft_secondary = fcfg.get("soft_tfidf_secondary", "jaro_winkler")
        self.jw_prefix = float(fcfg.get("jw_prefix_weight", 0.1))

    def column(self, df: pd.DataFrame, side: str, col: str) -> np.ndarray:
        """Object array of a record column, converted once and reused by every chunk."""
        key = (side, col)
        if key not in self._cols:
            self._cols[key] = df[col].to_numpy(object)
        return self._cols[key]

    @staticmethod
    def _idf(texts: pd.Series) -> dict:
        df = Counter()
        n = 0
        for t in texts:
            n += 1
            df.update(set(t.split()))
        return {tok: math.log((n + 1) / (c + 1)) + 1.0 for tok, c in df.items()}


# ----------------------------------------------------------------- helpers
def _sim(a, b, scorer, n_jobs, **kw) -> np.ndarray:
    out = cpdist(list(a), list(b), scorer=scorer, workers=n_jobs, **kw).astype(np.float32)
    return out


def _nan_if_missing(x: np.ndarray, a: np.ndarray, b: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    x[(a == "") | (b == "")] = np.nan
    return x


def _eq(a: np.ndarray, b: np.ndarray, nan_missing: bool = True) -> np.ndarray:
    out = (a == b).astype(np.float32)
    if nan_missing:
        out[(a == "") | (b == "")] = np.nan
    return out


# Globals for worker processes (set by the pool initializer).
_G: dict = {}


def _init_worker(name_idf, addr_idf, soft_thr, secondary, jw_prefix):
    _G.update(name_idf=name_idf, addr_idf=addr_idf, soft_thr=soft_thr,
              secondary=secondary, jw_prefix=jw_prefix,
              name_default=max(name_idf.values()) if name_idf else 1.0,
              addr_default=max(addr_idf.values()) if addr_idf else 1.0)


def soft_tfidf(ta: list[str], tb: list[str], idf: dict, thr: float = 0.9, secondary: str = "jaro_winkler",
               jw_prefix: float = 0.1, default: float | None = None) -> float:
    """Soft-TFIDF(S, T) = sum_{w in CLOSE(thr,S,T)} V(w,S) * V(w*,T) * sim(w, w*).

    `default` is the IDF of unseen tokens (pass it pre-computed in hot loops)."""
    if not ta or not tb:
        return float("nan")
    if default is None:
        default = max(idf.values()) if idf else 1.0

    def weights(tokens):
        c = Counter(tokens)
        w = {t: n * idf.get(t, default) for t, n in c.items()}
        norm = math.sqrt(sum(x * x for x in w.values())) or 1.0
        return {t: x / norm for t, x in w.items()}

    wa, wb = weights(ta), weights(tb)
    total = 0.0
    for t, w in wa.items():
        if t in wb:
            total += w * wb[t]
            continue
        best, best_u = 0.0, None
        for u in wb:
            if secondary == "levenshtein":
                s = Levenshtein.normalized_similarity(t, u)
            else:
                s = JaroWinkler.normalized_similarity(t, u, prefix_weight=jw_prefix)
            if s > best:
                best, best_u = s, u
        if best_u is not None and best >= thr:
            total += w * wb[best_u] * best
    return min(total, 1.0)


PY_FEATURES = ["name_soft_tfidf", "name_tok_jaccard", "name_tok_overlap_min", "name_idf_overlap",
               "name_max_shared_idf", "name_unshared_idf", "name_acronym", "addr_soft_tfidf",
               "addr_tok_jaccard", "num_jaccard", "num_conflict", "unit_eq", "unit_conflict",
               "suburb_overlap"]


def _python_block(args) -> np.ndarray:
    (n1, n2, a1, a2, i1, i2, num1, num2, u1, u2, sub1, sub2) = args
    name_idf, addr_idf = _G["name_idf"], _G["addr_idf"]
    thr, sec, jwp = _G["soft_thr"], _G["secondary"], _G["jw_prefix"]
    out = np.full((len(n1), len(PY_FEATURES)), np.nan, dtype=np.float32)
    for r in range(len(n1)):
        ta, tb = n1[r].split(), n2[r].split()
        sa, sb = set(ta), set(tb)
        out[r, 0] = soft_tfidf(ta, tb, name_idf, thr, sec, jwp, _G["name_default"])
        if sa and sb:
            inter, union = sa & sb, sa | sb
            out[r, 1] = len(inter) / len(union)
            out[r, 2] = len(inter) / min(len(sa), len(sb))
            wi = sum(name_idf.get(t, 1.0) for t in inter)
            wu = sum(name_idf.get(t, 1.0) for t in union)
            out[r, 3] = wi / wu if wu else 0.0
            out[r, 4] = max((name_idf.get(t, 1.0) for t in inter), default=0.0)
            out[r, 5] = sum(name_idf.get(t, 1.0) for t in (union - inter))
            ini_a, ini_b = i1[r], i2[r]
            nos_a, nos_b = "".join(ta), "".join(tb)
            out[r, 6] = float((len(ini_a) >= 2 and ini_a == nos_b) or (len(ini_b) >= 2 and ini_b == nos_a))
        xa, xb = a1[r].split(), a2[r].split()
        out[r, 7] = soft_tfidf(xa, xb, addr_idf, thr, sec, jwp, _G["addr_default"])
        if xa and xb:
            out[r, 8] = len(set(xa) & set(xb)) / len(set(xa) | set(xb))
        na, nb = set(num1[r].split()), set(num2[r].split())
        if na and nb:
            out[r, 9] = len(na & nb) / len(na | nb)
            out[r, 10] = float(not (na & nb))
        ua, ub = set(u1[r].split()), set(u2[r].split())
        if ua and ub:
            out[r, 11] = float(bool(ua & ub))
            out[r, 12] = float(not (ua & ub))
        sa2, sb2 = set(sub1[r].split()), set(sub2[r].split())
        if sa2 and sb2:
            out[r, 13] = len(sa2 & sb2) / min(len(sa2), len(sb2))
    return out


def _init_args(ctx: FeatureContext) -> tuple:
    return (ctx.name_idf, ctx.addr_idf, ctx.soft_thr, ctx.soft_secondary, ctx.jw_prefix)


@contextmanager
def feature_workers(ctx: FeatureContext, n_jobs: int):
    """One process pool for a whole feature run (created once, reused by every
    chunk). Yields None when running single-process."""
    if n_jobs <= 1:
        _init_worker(*_init_args(ctx))
        yield None
        return
    # Freeze the parent's objects before forking: otherwise the garbage
    # collector in every worker touches all inherited objects, triggering
    # copy-on-write and silently duplicating the parent's heap per worker.
    gc.collect()
    gc.freeze()
    try:
        with ProcessPoolExecutor(max_workers=n_jobs, initializer=_init_worker, initargs=_init_args(ctx)) as ex:
            yield ex
    finally:
        gc.unfreeze()


def _python_features(cols: dict, ctx: FeatureContext, executor=None, chunk: int = 25_000) -> np.ndarray:
    n = len(cols["n1"])
    keys = ["n1", "n2", "a1", "a2", "i1", "i2", "num1", "num2", "u1", "u2", "sub1", "sub2"]
    blocks = [tuple(list(cols[k][s:s + chunk]) for k in keys) for s in range(0, n, chunk)]
    if executor is None:
        if not _G or _G.get("name_idf") is not ctx.name_idf:
            _init_worker(*_init_args(ctx))
        results = [_python_block(b) for b in blocks]
    else:
        results = list(executor.map(_python_block, blocks))
    return np.vstack(results) if results else np.zeros((0, len(PY_FEATURES)), np.float32)


# -------------------------------------------------------------- main entry
def _base_features(pairs: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame, ctx: FeatureContext,
                   n_jobs: int, executor=None) -> pd.DataFrame:
    """All non-context features for ONE chunk of pairs (float32 columns)."""
    ia = pairs["s1_idx"].to_numpy(np.int64)
    ib = pairs["v_idx"].to_numpy(np.int64)
    g1 = lambda c: ctx.column(s1, "s1", c)[ia]
    g2 = lambda c: ctx.column(v, "v", c)[ib]
    F: dict[str, np.ndarray] = {}
    jw = lambda a, b: _sim(a, b, JaroWinkler.normalized_similarity, n_jobs, scorer_kwargs={"prefix_weight": ctx.jw_prefix})

    # ---------------- name
    raw1 = np.array([x.lower().strip() for x in g1("name")], dtype=object)
    raw2 = np.array([x.lower().strip() for x in g2("name")], dtype=object)
    core1, core2 = g1("name_core"), g2("name_core")
    tok1, tok2 = g1("name_tokens"), g2("name_tokens")
    nos1, nos2 = g1("name_nospace"), g2("name_nospace")
    F["name_exact_raw"] = _eq(raw1, raw2)
    F["name_exact_clean"] = _eq(g1("name_clean"), g2("name_clean"))
    F["name_exact_core"] = _eq(core1, core2)
    F["name_exact_nospace"] = _eq(nos1, nos2)
    F["name_jw"] = _nan_if_missing(jw(core1, core2), core1, core2)
    F["name_jw_nospace"] = _nan_if_missing(jw(nos1, nos2), nos1, nos2)
    F["name_lev"] = _nan_if_missing(_sim(core1, core2, Levenshtein.normalized_similarity, n_jobs), core1, core2)
    F["name_indel"] = _nan_if_missing(_sim(core1, core2, Indel.normalized_similarity, n_jobs), core1, core2)
    F["name_token_sort"] = _nan_if_missing(_sim(tok1, tok2, fuzz.token_sort_ratio, n_jobs) / 100, tok1, tok2)
    F["name_token_set"] = _nan_if_missing(_sim(tok1, tok2, fuzz.token_set_ratio, n_jobs) / 100, tok1, tok2)
    F["name_partial"] = _nan_if_missing(_sim(nos1, nos2, fuzz.partial_ratio, n_jobs) / 100, nos1, nos2)
    F["name_char_cos"] = rowwise_cosine(ctx.reps.s1("name_char"), ctx.reps.v("name_char"), ia, ib)
    F["name_word_cos"] = rowwise_cosine(ctx.reps.s1("name_word"), ctx.reps.v("name_word"), ia, ib)
    F["name_phonetic_eq"] = _eq(g1("name_phonetic"), g2("name_phonetic"))
    F["name_first_phonetic_eq"] = _eq(g1("name_first_phonetic"), g2("name_first_phonetic"))
    p1 = np.array([x[:4] for x in nos1], dtype=object)
    p2 = np.array([x[:4] for x in nos2], dtype=object)
    F["name_prefix4_eq"] = _eq(p1, p2)
    F["name_common_prefix"] = np.array([len(_common_prefix(a, b)) for a, b in zip(nos1, nos2)], np.float32)
    ft1 = np.array([x.split()[0] if x else "" for x in tok1], dtype=object)
    ft2 = np.array([x.split()[0] if x else "" for x in tok2], dtype=object)
    F["name_first_token_eq"] = _eq(ft1, ft2)
    l1 = np.array([len(x) for x in nos1], np.float32)
    l2 = np.array([len(x) for x in nos2], np.float32)
    F["name_len_1"], F["name_len_2"] = l1, l2
    F["name_len_ratio"] = np.where(np.maximum(l1, l2) > 0, np.minimum(l1, l2) / np.maximum(np.maximum(l1, l2), 1), np.nan).astype(np.float32)
    F["name_ntok_1"] = np.array([len(x.split()) for x in tok1], np.float32)
    F["name_ntok_2"] = np.array([len(x.split()) for x in tok2], np.float32)
    F["name_contains"] = np.array([float(bool(a) and bool(b) and (a in b or b in a)) for a, b in zip(nos1, nos2)], np.float32)
    d1, d2 = g1("name_digits"), g2("name_digits")
    F["name_digits_eq"] = _eq(d1, d2)
    # legal suffix (sequestered column) agreement
    sf1, sf2 = g1("name_suffix"), g2("name_suffix")
    fam1, fam2 = g1("name_suffix_family"), g2("name_suffix_family")
    F["suffix_present_1"] = (sf1 != "").astype(np.float32)
    F["suffix_present_2"] = (sf2 != "").astype(np.float32)
    F["suffix_eq"] = _eq(sf1, sf2)
    F["suffix_family_eq"] = _eq(fam1, fam2)
    F["suffix_conflict"] = ((sf1 != "") & (sf2 != "") & (fam1 != fam2)).astype(np.float32)

    # ---------------- address
    ad1, ad2 = g1("addr_clean"), g2("addr_clean")
    F["addr_missing_1"] = (ad1 == "").astype(np.float32)
    F["addr_missing_2"] = (ad2 == "").astype(np.float32)
    F["addr_exact"] = _eq(ad1, ad2)
    F["addr_jw"] = _nan_if_missing(jw(ad1, ad2), ad1, ad2)
    F["addr_token_set"] = _nan_if_missing(_sim(ad1, ad2, fuzz.token_set_ratio, n_jobs) / 100, ad1, ad2)
    F["addr_char_cos"] = rowwise_cosine(ctx.reps.s1("addr_char"), ctx.reps.v("addr_char"), ia, ib)
    F["full_char_cos"] = rowwise_cosine(ctx.reps.s1("full_char"), ctx.reps.v("full_char"), ia, ib)
    comps = {}
    for comp in ("house_number", "road", "city", "state", "postcode", "unit"):
        a, b = g1(f"addr_{comp}"), g2(f"addr_{comp}")
        comps[comp] = (a, b)
        both = (a != "") & (b != "")
        F[f"{comp}_both"] = both.astype(np.float32)
    hn1, hn2 = comps["house_number"]
    F["hn_eq"] = _eq(hn1, hn2)
    F["hn_conflict"] = ((hn1 != "") & (hn2 != "") & (hn1 != hn2)).astype(np.float32)
    r1, r2 = comps["road"]
    F["road_jw"] = _nan_if_missing(jw(r1, r2), r1, r2)
    F["road_token_set"] = _nan_if_missing(_sim(r1, r2, fuzz.token_set_ratio, n_jobs) / 100, r1, r2)
    c1, c2 = comps["city"]
    F["city_eq"] = _eq(c1, c2)
    F["city_jw"] = _nan_if_missing(jw(c1, c2), c1, c2)
    F["city_conflict"] = ((c1 != "") & (c2 != "") & (np.nan_to_num(F["city_jw"]) < 0.85)).astype(np.float32)
    s_1, s_2 = comps["state"]
    F["state_eq"] = _eq(s_1, s_2)
    F["state_conflict"] = ((s_1 != "") & (s_2 != "") & (s_1 != s_2)).astype(np.float32)
    pc1, pc2 = comps["postcode"]
    F["pc_eq"] = _eq(pc1, pc2)
    F["pc_prefix3_eq"] = _eq(np.array([x[:3] for x in pc1], dtype=object), np.array([x[:3] for x in pc2], dtype=object))
    F["pc_conflict"] = ((pc1 != "") & (pc2 != "") & (pc1 != pc2)).astype(np.float32)
    F["landmark_1"] = (g1("addr_landmark") != "").astype(np.float32)
    F["landmark_2"] = (g2("addr_landmark") != "").astype(np.float32)

    # ---------------- python-loop features (multiprocess)
    py = _python_features({
        "n1": tok1, "n2": tok2, "a1": ad1, "a2": ad2, "i1": g1("name_initials"), "i2": g2("name_initials"),
        "num1": g1("addr_numbers"), "num2": g2("addr_numbers"), "u1": g1("addr_unit"), "u2": g2("addr_unit"),
        "sub1": g1("addr_suburb"), "sub2": g2("addr_suburb")}, ctx, executor)
    for j, name in enumerate(PY_FEATURES):
        F[name] = py[:, j]

    # ---------------- cross-field and combined
    F["name_in_other_addr"] = np.maximum(
        _nan_if_missing(_sim(tok1, ad2, fuzz.token_set_ratio, n_jobs) / 100, tok1, ad2),
        _nan_if_missing(_sim(tok2, ad1, fuzz.token_set_ratio, n_jobs) / 100, tok2, ad1))
    name_strength = np.nan_to_num(F["name_soft_tfidf"])
    addr_strength = np.nan_to_num(F["addr_char_cos"])
    F["name_x_addr"] = (name_strength * addr_strength).astype(np.float32)
    F["strong_name_weak_addr"] = ((np.nan_to_num(F["name_jw"]) >= 0.92) & (addr_strength < 0.3)).astype(np.float32)
    F["weak_name_strong_addr"] = ((np.nan_to_num(F["name_jw"]) < 0.7) & (addr_strength >= 0.8)).astype(np.float32)
    agree = (np.nan_to_num(F["hn_eq"]) + (np.nan_to_num(F["road_jw"]) >= 0.9) + np.nan_to_num(F["city_eq"])
             + np.nan_to_num(F["state_eq"]) + np.nan_to_num(F["pc_eq"]) + np.nan_to_num(F["unit_eq"]))
    conflict = (F["hn_conflict"] + F["city_conflict"] + F["state_conflict"] + F["pc_conflict"]
                + np.nan_to_num(F["unit_conflict"]))
    F["addr_components_agree"] = agree.astype(np.float32)
    F["addr_components_conflict"] = conflict.astype(np.float32)
    F["addr_components_both"] = sum(F[f"{c}_both"] for c in ("house_number", "road", "city", "state", "postcode", "unit")).astype(np.float32)
    F["contradictions"] = (conflict + F["suffix_conflict"] + np.nan_to_num(F["num_conflict"])).astype(np.float32)
    F["src_is_3"] = (pairs["source"].to_numpy() == 3).astype(np.float32)

    # ---------------- blocking provenance (label-free)
    for m in BLOCKING_FEATURE_METHODS:
        col = f"blk_{m}"
        F[col] = pairs[col].to_numpy(np.float32) if col in pairs else np.zeros(len(pairs), np.float32)
    F["blk_n_methods_syntactic"] = sum(F[f"blk_{m}"] for m in BLOCKING_FEATURE_METHODS).astype(np.float32)

    return pd.DataFrame({k: np.asarray(val, dtype=np.float32) for k, val in F.items()}, index=pairs.index)


def _chunks(n: int, size: int):
    for a in range(0, n, max(1, size)):
        yield a, min(n, a + max(1, size))


def compute_features(pairs: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame, ctx: FeatureContext,
                     cfg: dict) -> pd.DataFrame:
    """In-memory feature frame (small inputs, tests). Large runs should use
    `build_feature_matrix`, which streams chunks to a disk-backed matrix."""
    log = get_logger()
    n_jobs = resolve_n_jobs(cfg.get("n_jobs", 1))
    size = int(cfg.get("features", {}).get("chunk_pairs", 250_000))
    with feature_workers(ctx, n_jobs if len(pairs) > 25_000 else 1) as ex:
        parts = [_base_features(pairs.iloc[a:b], s1, v, ctx, n_jobs, ex) for a, b in _chunks(len(pairs), size)]
    out = pd.concat(parts) if parts else _base_features(pairs, s1, v, ctx, n_jobs)
    if cfg.get("features", {}).get("context_features", True):
        out = add_context_features(pairs, out)
    log.info("Computed %d features for %d pairs", out.shape[1], len(out))
    return out


class FeatureMatrix:
    """Disk-backed float32 feature matrix (rows = candidate pairs).

    Stored as one or more row-major .npy parts (base features + context
    features) opened read-only with mmap: only the rows / columns a step needs
    are read into RAM, so the full matrix never has to fit in memory. (Pages
    read through the map are clean page cache that the OS can drop at will.)
    """

    def __init__(self, path: str, columns: list[str] | None = None):
        self.path = path
        meta = load_json(path + ".columns.json")
        if isinstance(meta, list):            # single-part layout
            meta = {"parts": [[os.path.basename(path), meta]]}
        folder = os.path.dirname(path)
        self.parts = [(np.load(os.path.join(folder, f), mmap_mode="r"), list(cols)) for f, cols in meta["parts"]]
        self.columns = [c for _, cols in self.parts for c in cols]
        if columns is not None and list(columns) != self.columns:
            raise ValueError("feature columns on disk differ from the expected columns")
        self.col_idx = {c: (pi, j) for pi, (_, cols) in enumerate(self.parts) for j, c in enumerate(cols)}

    def __len__(self) -> int:
        return int(self.parts[0][0].shape[0])

    @staticmethod
    def _rows(rows):
        if rows is None:
            return None
        rows = np.asarray(rows)
        return np.flatnonzero(rows) if rows.dtype == bool else rows.astype(np.int64)

    def _gather(self, r, cols: list[str], a: int | None = None, b: int | None = None,
                out: np.ndarray | None = None) -> np.ndarray:
        n = (b - a) if r is None else len(r)
        if out is None:
            out = np.empty((n, len(cols)), dtype=np.float32)
        for pi, (arr, _) in enumerate(self.parts):
            pos = [k for k, c in enumerate(cols) if self.col_idx[c][0] == pi]
            if not pos:
                continue
            lci = np.array([self.col_idx[cols[k]][1] for k in pos], dtype=np.int64)
            if r is None:
                out[:, pos] = arr[a:b][:, lci]
            else:
                out[:, pos] = arr[np.ix_(r, lci)]
        return out

    def frame(self, rows=None, cols: list[str] | None = None, index=None) -> pd.DataFrame:
        cols = list(cols) if cols is not None else self.columns
        r = self._rows(rows)
        data = self._gather(r, cols, 0, len(self)) if r is None else self._gather(r, cols)
        return pd.DataFrame(data, columns=cols, index=index)

    def fill(self, rows, cols: list[str], out: np.ndarray) -> np.ndarray:
        """Gather rows/cols straight into a pre-allocated float32 array."""
        return self._gather(self._rows(rows), list(cols), out=out)

    def blocks(self, cols: list[str], size: int = 500_000):
        for a, b in _chunks(len(self), size):
            yield a, b, self._gather(None, cols, a, b)


def build_feature_matrix(pairs: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame, ctx: FeatureContext,
                         cfg: dict, path: str) -> FeatureMatrix:
    """Compute features chunk by chunk and append them to `path` (.npy) with
    plain sequential writes (no writable memory map, so the process never
    accumulates dirty file pages). The three columns the context features need
    are kept in RAM; context features go to a small second file."""
    log = get_logger()
    fcfg = cfg.get("features", {})
    n_jobs = resolve_n_jobs(cfg.get("n_jobs", 1))
    size = int(fcfg.get("chunk_pairs", 250_000))
    n = len(pairs)
    use_ctx = bool(fcfg.get("context_features", True))
    ensure_dir(os.path.dirname(path))
    base_cols, fh, offset = None, None, 0
    keep = {c: np.empty(n, dtype=np.float32) for c in ("full_char_cos", "name_soft_tfidf", "name_char_cos")}
    try:
        with feature_workers(ctx, n_jobs) as ex:
            for i, (a, b) in enumerate(_chunks(n, size)):
                part = _base_features(pairs.iloc[a:b], s1, v, ctx, n_jobs, ex)
                if fh is None:
                    base_cols = list(part.columns)
                    # header + correctly sized (sparse) file, then plain appends
                    mm = np.lib.format.open_memmap(path, mode="w+", dtype=np.float32, shape=(n, len(base_cols)))
                    offset = mm.offset
                    del mm
                    fh = open(path, "r+b")
                block = np.ascontiguousarray(part[base_cols].to_numpy(np.float32))
                fh.seek(offset + a * len(base_cols) * 4)
                fh.write(block.tobytes())
                for c in keep:
                    keep[c][a:b] = part[c].to_numpy(np.float32)
                del part, block
                if i % 10 == 0 or b == n:
                    log.info("  features %d / %d pairs  [%s]", b, n, mem_str())
    finally:
        if fh is not None:
            fh.close()
    if base_cols is None:  # no pairs at all
        base_cols = list(_base_features(pairs, s1, v, ctx, 1).columns)
        np.lib.format.open_memmap(path, mode="w+", dtype=np.float32, shape=(0, len(base_cols)))
    parts = [[os.path.basename(path), base_cols]]
    if use_ctx:
        ctx_path = path[:-4] + "_context.npy" if path.endswith(".npy") else path + "_context.npy"
        arrs = context_arrays(pairs, keep["full_char_cos"], keep["name_soft_tfidf"], keep["name_char_cos"])
        np.save(ctx_path, np.column_stack([arrs[c] for c in CONTEXT_COLUMNS]).astype(np.float32)
                if n else np.zeros((0, len(CONTEXT_COLUMNS)), np.float32))
        del arrs
        parts.append([os.path.basename(ctx_path), CONTEXT_COLUMNS])
    del keep
    save_json({"parts": parts}, path + ".columns.json")
    fm = FeatureMatrix(path)
    log.info("Computed %d features for %d pairs -> %s  [%s]", len(fm.columns), n, path, mem_str())
    return fm


def remove_feature_matrix(path: str) -> None:
    """Delete a feature matrix and all of its parts (features.keep_matrix: false)."""
    meta_path = path + ".columns.json"
    files = [path]
    if os.path.exists(meta_path):
        meta = load_json(meta_path)
        if isinstance(meta, dict):
            files += [os.path.join(os.path.dirname(path), f) for f, _ in meta.get("parts", [])]
        files.append(meta_path)
    for f in dict.fromkeys(files):
        if os.path.exists(f):
            os.remove(f)


def predict_in_blocks(booster, feats, cols: list[str], size: int = 500_000) -> np.ndarray:
    """LightGBM prediction without materialising the whole feature matrix."""
    n = len(feats)
    out = np.empty(n, dtype=np.float64)
    if n == 0:
        return out
    if isinstance(feats, FeatureMatrix):
        for a, b, X in feats.blocks(cols, size):
            out[a:b] = booster.predict(X)
    else:
        for a, b in _chunks(n, size):
            out[a:b] = booster.predict(feats.iloc[a:b][cols].to_numpy(np.float32))
    return out


def _common_prefix(a: str, b: str) -> str:
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return a[:n]


def _rank_and_margin(codes: np.ndarray, values: np.ndarray):
    """Per group (integer `codes`): rank (1 = best), gap to the best, margin
    over the best OTHER member, group size. Fully vectorised."""
    v = np.nan_to_num(values.astype(np.float64), nan=-1.0)
    codes = np.asarray(codes, dtype=np.int64)
    n = len(v)
    if n == 0:
        z = np.zeros(0, np.float32)
        return z, z, z, z
    order = np.lexsort((-v, codes))
    sc, sv = codes[order], v[order]
    starts = np.r_[0, np.flatnonzero(np.diff(sc)) + 1]
    sizes = np.diff(np.r_[starts, n])
    grp = np.repeat(np.arange(len(starts)), sizes)
    top = sv[starts]
    second = np.where(sizes > 1, sv[np.minimum(starts + 1, n - 1)], -1.0)
    # rank with ties -> 'min' method: position of first element with same value
    new_val = np.r_[True, (sv[1:] != sv[:-1]) | (sc[1:] != sc[:-1])]
    first_pos = np.maximum.accumulate(np.where(new_val, np.arange(n), 0))
    rank_sorted = (first_pos - starts[grp] + 1).astype(np.float32)
    top_s, second_s = top[grp], second[grp]
    is_top = sv >= top_s
    margin_s = sv - np.where(is_top, second_s, top_s)
    gap_s = top_s - sv
    size_s = sizes[grp].astype(np.float32)
    inv = np.empty(n, dtype=np.int64)
    inv[order] = np.arange(n)
    return (rank_sorted[inv], gap_s[inv].astype(np.float32), margin_s[inv].astype(np.float32),
            size_s[inv])


CONTEXT_COLUMNS = ["ctx_rank", "ctx_gap_to_best", "ctx_margin", "ctx_n_cands", "ctx_name_rank",
                   "ctx_name_gap", "ctx_name_margin", "ctx_v_rank", "ctx_v_margin", "ctx_v_n_s1",
                   "ctx_mutual_best"]


def context_arrays(pairs: pd.DataFrame, full_char_cos: np.ndarray, name_soft_tfidf: np.ndarray,
                   name_char_cos: np.ndarray) -> dict[str, np.ndarray]:
    """Label-free features describing a pair relative to its competitors."""
    s1i = pairs["s1_idx"].to_numpy()
    src = pairs["source"].to_numpy()
    vi = pairs["v_idx"].to_numpy()
    base = np.nan_to_num(full_char_cos) * 0.5 + np.nan_to_num(name_soft_tfidf) * 0.5
    ent_src = s1i.astype(np.int64) * 4 + src.astype(np.int64)
    out = {}
    rank, gap, margin, size = _rank_and_margin(ent_src, base)
    out["ctx_rank"], out["ctx_gap_to_best"], out["ctx_margin"], out["ctx_n_cands"] = rank, gap, margin, size
    nrank, ngap, nmargin, _ = _rank_and_margin(ent_src, name_char_cos)
    out["ctx_name_rank"], out["ctx_name_gap"], out["ctx_name_margin"] = nrank, ngap, nmargin
    vrank, _, vmargin, vsize = _rank_and_margin(vi, base)
    out["ctx_v_rank"], out["ctx_v_margin"] = vrank, vmargin
    out["ctx_v_n_s1"] = np.minimum(vsize, 20).astype(np.float32)
    out["ctx_mutual_best"] = ((rank == 1) & (vrank == 1)).astype(np.float32)
    return {c: out[c].astype(np.float32) for c in CONTEXT_COLUMNS}


def add_context_features(pairs: pd.DataFrame, feats: pd.DataFrame) -> pd.DataFrame:
    feats = feats.copy()
    for c, arr in context_arrays(pairs, feats["full_char_cos"].to_numpy(), feats["name_soft_tfidf"].to_numpy(),
                                 feats["name_char_cos"].to_numpy()).items():
        feats[c] = arr
    return feats


def feature_columns(feats: pd.DataFrame) -> list[str]:
    return [c for c in feats.columns if c not in ("s1_idx", "v_idx", "source", "label")]


In [ ]:
%%writefile src/train_lightgbm.py
"""Structural matcher: LightGBM binary classifier on handcrafted features.

PDF: "LightGBM provides an exceptionally fast and highly accurate baseline
when supplied with rich, handcrafted features" and feature importance by GAIN
lets us audit which name/address components drive decisions - the gain table
is written to feature_importance.tsv.
"""
from __future__ import annotations

import os

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

from .utils import get_logger, save_json


def _safe_metrics(y: np.ndarray, p: np.ndarray) -> dict:
    out = {"n": int(len(y)), "positives": int(y.sum())}
    if len(np.unique(y)) == 2:
        out["auc"] = float(roc_auc_score(y, p))
        out["average_precision"] = float(average_precision_score(y, p))
        out["logloss"] = float(log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)))
    return out


def train_lightgbm(X_tr: pd.DataFrame, y_tr: np.ndarray, X_val: pd.DataFrame, y_val: np.ndarray,
                   cfg: dict, out_dir: str, seed: int = 42) -> tuple[lgb.Booster, dict]:
    log = get_logger()
    lcfg = cfg.get("lightgbm", {})
    params = dict(lcfg.get("params", {}))
    params.setdefault("objective", "binary")
    params.setdefault("verbose", -1)
    params["seed"] = seed
    params.setdefault("metric", ["binary_logloss", "average_precision"])
    params.setdefault("num_threads", max(1, int(cfg.get("n_jobs", 1) or 1)))
    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    valid_sets, valid_names, callbacks = [dtrain], ["train"], [lgb.log_evaluation(100)]
    has_val = X_val is not None and len(X_val) > 0 and len(np.unique(y_val)) == 2
    if has_val:
        dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
        valid_sets.append(dval)
        valid_names.append("val")
        callbacks.append(lgb.early_stopping(int(lcfg.get("early_stopping_rounds", 100)),
                                            first_metric_only=True, verbose=False))
    num_rounds = int(lcfg.get("num_boost_round", 3000)) if has_val else int(lcfg.get("num_boost_round_no_val", 300))
    log.info("LightGBM: %d train rows (%d pos), %d val rows (%d pos), %d features",
             len(X_tr), int(y_tr.sum()), 0 if X_val is None else len(X_val),
             0 if y_val is None else int(y_val.sum()), X_tr.shape[1])
    booster = lgb.train(params, dtrain, num_boost_round=num_rounds, valid_sets=valid_sets,
                        valid_names=valid_names, callbacks=callbacks)
    best_iter = booster.best_iteration or booster.current_iteration()
    report = {"best_iteration": int(best_iter), "params": params,
              "train": _safe_metrics(y_tr, booster.predict(X_tr, num_iteration=best_iter))}
    if has_val:
        report["val"] = _safe_metrics(y_val, booster.predict(X_val, num_iteration=best_iter))
    os.makedirs(out_dir, exist_ok=True)
    booster.save_model(os.path.join(out_dir, "lightgbm.txt"), num_iteration=best_iter)
    imp = pd.DataFrame({
        "feature": booster.feature_name(),
        "gain": booster.feature_importance("gain", iteration=best_iter),
        "split": booster.feature_importance("split", iteration=best_iter),
    }).sort_values("gain", ascending=False)
    imp["gain_share"] = imp["gain"] / max(imp["gain"].sum(), 1e-12)
    imp.to_csv(os.path.join(out_dir, "feature_importance.tsv"), sep="\t", index=False)
    report["top_features_by_gain"] = imp.head(25)[["feature", "gain_share"]].to_dict("records")
    save_json(report, os.path.join(out_dir, "lightgbm_report.json"))
    log.info("LightGBM best_iteration=%d  val=%s", best_iter, report.get("val"))
    return booster, report


def load_lightgbm(model_dir: str) -> lgb.Booster:
    return lgb.Booster(model_file=os.path.join(model_dir, "lightgbm.txt"))


def predict_lightgbm(booster: lgb.Booster, X: pd.DataFrame) -> np.ndarray:
    if len(X) == 0:
        return np.zeros(0)
    return booster.predict(X[booster.feature_name()])


In [ ]:
%%writefile src/calibration.py
"""Probability calibration (PDF: "Isotonic Regression and Decision Threshold
Calibration").

Calibrators are ALWAYS fitted on the `calib` split (entities never seen by
the matchers) and never on test data. Isotonic regression (the PDF's choice)
and Platt/logistic scaling are both available; `auto` picks the one with the
lower cross-validated Brier score and the comparison is reported either way.

Isotonic regression is a step function, so many pairs share one calibrated
value; a 1e-4 * raw-score tie-breaker keeps the order of the raw scores
(monotone, so it cannot change which pairs pass a threshold by more than
that epsilon) which the per-entity ranking rules rely on.
"""
from __future__ import annotations

import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold

_EPS = 1e-4


def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


class Calibrator:
    def __init__(self, method: str = "isotonic"):
        if method not in ("isotonic", "platt", "none"):
            raise ValueError(f"unknown calibration method {method}")
        self.method = method
        self.model = None

    def fit(self, scores, y) -> "Calibrator":
        scores = np.asarray(scores, dtype=float)
        y = np.asarray(y, dtype=int)
        if self.method == "none" or len(np.unique(y)) < 2:
            self.method = "none" if len(np.unique(y)) < 2 else self.method
            return self
        if self.method == "isotonic":
            self.model = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(scores, y)
        else:
            self.model = LogisticRegression(C=1e6, max_iter=1000).fit(_logit(scores).reshape(-1, 1), y)
        return self

    def transform(self, scores) -> np.ndarray:
        scores = np.asarray(scores, dtype=float)
        if self.method == "none" or self.model is None:
            return scores
        if self.method == "isotonic":
            cal = self.model.predict(scores)
        else:
            cal = self.model.predict_proba(_logit(scores).reshape(-1, 1))[:, 1]
        return np.clip(cal * (1 - _EPS) + _EPS * np.clip(scores, 0, 1), 0.0, 1.0)


def compare_methods(scores, y, folds: int = 3, seed: int = 0) -> dict:
    """Cross-validated Brier score / log loss of raw, isotonic and Platt."""
    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y, dtype=int)
    out = {"raw": {"brier": float(brier_score_loss(y, np.clip(scores, 0, 1)))}}
    if len(np.unique(y)) < 2 or min(np.bincount(y)) < folds:
        return out
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for method in ("isotonic", "platt"):
        oof = np.zeros(len(y))
        for tr, te in skf.split(scores, y):
            oof[te] = Calibrator(method).fit(scores[tr], y[tr]).transform(scores[te])
        out[method] = {"brier": float(brier_score_loss(y, oof)),
                       "logloss": float(log_loss(y, np.clip(oof, 1e-7, 1 - 1e-7))),
                       "ece": expected_calibration_error(oof, y)}
    return out


def fit_calibrator(scores, y, method: str = "isotonic", seed: int = 0) -> tuple[Calibrator, dict]:
    report = compare_methods(scores, y, seed=seed)
    chosen = method
    if method == "auto":
        cands = [m for m in ("isotonic", "platt") if m in report]
        chosen = min(cands, key=lambda m: report[m]["brier"]) if cands else "none"
    cal = Calibrator(chosen).fit(scores, y)
    report["chosen"] = cal.method
    if len(scores):
        report["reliability"] = reliability_table(cal.transform(scores), y)
    return cal, report


def expected_calibration_error(p, y, bins: int = 10) -> float:
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    ece = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(ece)


def reliability_table(p, y, bins: int = 10) -> list[dict]:
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    rows = []
    for b in range(bins):
        m = idx == b
        rows.append({"bin": f"{edges[b]:.1f}-{edges[b + 1]:.1f}", "n": int(m.sum()),
                     "mean_pred": float(p[m].mean()) if m.any() else None,
                     "frac_pos": float(y[m].mean()) if m.any() else None})
    return rows


In [ ]:
%%writefile src/scoring.py
"""Cascade and score blending.

Cascade (why): blocking yields tens of candidates per entity; a DeBERTa-v3
cross-encoder costs ~1000x a LightGBM prediction. The cheap structural model
therefore scores EVERY candidate, and only candidates that are plausible but
not already certain are re-scored by the cross-encoder:

    lgb_low <= p_lgb (calibrated)  and  rank within (entity, source) <= max_per_entity_source
    and (p_lgb <= lgb_high  or  rescore_confident)

Pairs outside the cascade keep their (calibrated) LightGBM probability.

Blending (tuned on the calib split by Macro F0.5, never on test):
  weighted  p = w * p_lgb + (1 - w) * p_ce
  rank      same on empirical-CDF ranks (robust to scale differences)
  stack     logistic regression on [logit p_lgb, logit p_ce, their product]
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


def cascade_mask(pairs: pd.DataFrame, p_lgb: np.ndarray, ccfg: dict) -> np.ndarray:
    low = float(ccfg.get("lgb_low", 0.02))
    high = float(ccfg.get("lgb_high", 0.995))
    k = int(ccfg.get("max_per_entity_source", 10))
    df = pd.DataFrame({"s1": pairs["s1_idx"].to_numpy(), "src": pairs["source"].to_numpy(), "p": p_lgb})
    rank = df.groupby(["s1", "src"], sort=False)["p"].rank(ascending=False, method="first").to_numpy()
    mask = (p_lgb >= low) & (rank <= k)
    if not ccfg.get("rescore_confident", False):
        mask &= p_lgb <= high
    return mask


def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


class Blender:
    def __init__(self, method: str = "weighted", weight: float = 1.0):
        self.method = method
        self.weight = float(weight)
        self.ref_lgb = None
        self.ref_ce = None
        self.stacker = None

    def fit(self, p_lgb, p_ce, has_ce, y=None) -> "Blender":
        has_ce = np.asarray(has_ce, dtype=bool)
        if self.method == "rank":
            self.ref_lgb = np.sort(np.asarray(p_lgb)[has_ce])
            self.ref_ce = np.sort(np.asarray(p_ce)[has_ce])
        elif self.method == "stack":
            X = self._stack_x(np.asarray(p_lgb)[has_ce], np.asarray(p_ce)[has_ce])
            yy = np.asarray(y)[has_ce]
            if len(np.unique(yy)) == 2:
                self.stacker = LogisticRegression(C=1.0, max_iter=1000).fit(X, yy)
        return self

    @staticmethod
    def _stack_x(a, b):
        la, lb = _logit(a), _logit(b)
        return np.column_stack([la, lb, la * lb])

    @staticmethod
    def _cdf(ref, x):
        if ref is None or len(ref) == 0:
            return np.asarray(x, dtype=float)
        return np.searchsorted(ref, x, side="right") / len(ref)

    def transform(self, p_lgb, p_ce, has_ce) -> np.ndarray:
        p_lgb = np.asarray(p_lgb, dtype=float)
        p_ce = np.nan_to_num(np.asarray(p_ce, dtype=float), nan=0.0)
        has_ce = np.asarray(has_ce, dtype=bool)
        out = p_lgb.copy()
        if not has_ce.any():
            return out
        a, b = p_lgb[has_ce], p_ce[has_ce]
        if self.method == "weighted":
            out[has_ce] = self.weight * a + (1 - self.weight) * b
        elif self.method == "rank":
            ra, rb = self._cdf(self.ref_lgb, a), self._cdf(self.ref_ce, b)
            blended_rank = self.weight * ra + (1 - self.weight) * rb
            # map the blended rank back onto the LightGBM probability scale so
            # that cascaded and non-cascaded pairs stay comparable
            ref = self.ref_lgb if self.ref_lgb is not None and len(self.ref_lgb) else np.sort(a)
            out[has_ce] = np.quantile(ref, np.clip(blended_rank, 0, 1))
        elif self.method == "stack":
            if self.stacker is not None:
                out[has_ce] = self.stacker.predict_proba(self._stack_x(a, b))[:, 1]
        else:
            raise ValueError(f"unknown blend method {self.method}")
        return out

    def describe(self) -> dict:
        d = {"method": self.method, "weight_lgb": self.weight}
        if self.stacker is not None:
            d["stack_coef"] = self.stacker.coef_.ravel().tolist()
            d["stack_intercept"] = float(self.stacker.intercept_[0])
        return d


In [ ]:
%%writefile src/decision.py
"""Decision layer: turn calibrated pair probabilities into one id SET per
Source-1 entity, tuned for Macro F0.5 on the calib split.

Rules (all deterministic, applied per Source-1 entity):
  1. graph cleanup mask (off / exclusive / betweenness)
  2. keep pair if p >= threshold of its source (Source 2 and 3 may differ -
     the vendors have different noise levels)
  3. keep pair only if p >= (best p of the entity) - margin
  4. singleton gate: the entity outputs anything only if its best p >= gate
  5. optional entity-level singleton model: output empty if
     P(entity has a true match among candidates) < matchable_threshold
  6. optional cap on matches per source

The search maximises the EXACT entity-level Macro F0.5 (every entity of the
split counts, including entities with no candidates and true matches lost in
blocking), NOT pairwise F1. No threshold value is assumed; everything is
swept on validation data.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from .metrics import entity_fbeta
from .utils import get_logger


@dataclass
class DecisionParams:
    threshold_s2: float = 0.5
    threshold_s3: float = 0.5
    margin: float = 1.0
    gate: float = 0.0
    max_per_source: int | None = None
    matchable_threshold: float = 0.0
    graph_mode: str = "off"

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "DecisionParams":
        known = {k: d[k] for k in cls.__dataclass_fields__ if k in d}
        if "graph_mode" in known and (known["graph_mode"] is False or known["graph_mode"] is None):
            known["graph_mode"] = "off"   # YAML parses bare `off` as False
        return cls(**known)


@dataclass
class DecisionTable:
    """Pre-computed arrays for fast repeated evaluation on one entity set."""
    ent: np.ndarray                 # entity position of every pair
    src: np.ndarray
    p: np.ndarray
    n_entities: int
    is_true: np.ndarray | None = None
    n_true: np.ndarray | None = None
    graph_masks: dict = field(default_factory=dict)
    matchable: np.ndarray | None = None
    beta: float = 0.5

    def __post_init__(self):
        self.order = np.lexsort((-self.p, self.src, self.ent))
        o_ent, o_src = self.ent[self.order], self.src[self.order]
        new_grp = np.r_[True, (o_ent[1:] != o_ent[:-1]) | (o_src[1:] != o_src[:-1])]
        self.grp_start_pos = np.maximum.accumulate(np.where(new_grp, np.arange(len(self.order)), 0))
        self._cache: dict = {}

    @classmethod
    def build(cls, pairs: pd.DataFrame, prob_col: str, entity_idx: np.ndarray, n_true: np.ndarray | None = None,
              graph_masks: dict | None = None, matchable: np.ndarray | None = None, beta: float = 0.5):
        pos = pd.Series(np.arange(len(entity_idx)), index=entity_idx)
        ent = pos.reindex(pairs["s1_idx"].to_numpy()).to_numpy()
        if np.isnan(ent.astype(float)).any():
            raise ValueError("pairs contain entities outside entity_idx")
        return cls(ent=ent.astype(np.int64), src=pairs["source"].to_numpy(np.int64),
                   p=pairs[prob_col].to_numpy(np.float64), n_entities=len(entity_idx),
                   is_true=(pairs["is_true"].to_numpy(bool) if "is_true" in pairs
                            else (pairs["label"].to_numpy() != 0) if "label" in pairs else None),
                   n_true=n_true, graph_masks=graph_masks or {}, matchable=matchable, beta=beta)

    def _peff_top(self, mode: str):
        if mode not in self._cache:
            keep = self.graph_masks.get(mode)
            p = self.p if keep is None else np.where(keep, self.p, -1.0)
            top = np.full(self.n_entities, -1.0)
            np.maximum.at(top, self.ent, p)
            self._cache[mode] = (p, top)
        return self._cache[mode]

    def select(self, prm: DecisionParams) -> np.ndarray:
        p, top = self._peff_top(prm.graph_mode)
        thr = np.where(self.src == 2, prm.threshold_s2, prm.threshold_s3)
        top_e = top[self.ent]
        ok = (p >= thr) & (p >= top_e - prm.margin) & (top_e >= prm.gate)
        if self.matchable is not None and prm.matchable_threshold > 0:
            ok &= self.matchable[self.ent] >= prm.matchable_threshold
        if prm.max_per_source:
            okc = ok[self.order].astype(np.int64)
            cs = np.cumsum(okc)
            before = cs[self.grp_start_pos] - okc[self.grp_start_pos]
            rank = cs - before
            capped = np.zeros(len(ok), dtype=bool)
            capped[self.order] = (okc == 1) & (rank <= prm.max_per_source)
            ok = capped
        return ok

    def score(self, prm: DecisionParams) -> float:
        sel = self.select(prm)
        tp = np.bincount(self.ent[sel & self.is_true], minlength=self.n_entities)
        npred = np.bincount(self.ent[sel], minlength=self.n_entities)
        return float(entity_fbeta(tp, npred, self.n_true, self.beta).mean())


def plateau_center(values: list, scores: list, tol: float = 1e-12):
    """Among grid values reaching the best score, return the middle of the
    longest contiguous run. Calibrated scores are often nearly binary, so
    many thresholds tie; the first one (argmax) sits on the edge of the
    plateau and generalises worst."""
    scores = np.asarray(scores, dtype=float)
    best = scores.max()
    good = scores >= best - tol
    runs, start = [], None
    for i, g in enumerate(good):
        if g and start is None:
            start = i
        if not g and start is not None:
            runs.append((start, i - 1))
            start = None
    if start is not None:
        runs.append((start, len(good) - 1))
    a, b = max(runs, key=lambda r: r[1] - r[0])
    return values[(a + b) // 2]


def effective_min_improvement(dcfg: dict, n_entities: int) -> float:
    """Gain an extra rule/blend must show on validation before it is adopted:
    max(min_improvement, min_entities_gain / n_entities). One entity is worth
    1/n of Macro F0.5, so on small validation sets a gain of a single entity
    is indistinguishable from noise."""
    return max(float(dcfg.get("min_improvement", 0.0005)),
               float(dcfg.get("min_entities_gain", 2)) / max(1, n_entities))


def search_decision(table: DecisionTable, dcfg: dict, graph_modes: list[str] | None = None,
                    quick: bool = False) -> tuple[DecisionParams, float, list]:
    """Coordinate-ascent search of the decision parameters (Macro F0.5)."""
    g = dcfg.get("threshold_grid", {"start": 0.05, "stop": 0.99, "step": 0.01})
    T = np.round(np.arange(g["start"], g["stop"] + 1e-9, g["step"]), 4)
    min_imp = effective_min_improvement(dcfg, table.n_entities)
    trace: list = []
    best = DecisionParams(graph_mode="off")
    # 1) global threshold sweep (shared by both sources)
    scores = []
    for t in T:
        prm = DecisionParams(threshold_s2=float(t), threshold_s3=float(t))
        scores.append(table.score(prm))
        trace.append(("init", "threshold", float(t), scores[-1]))
    t_best = plateau_center([float(t) for t in T], scores)
    best.threshold_s2 = best.threshold_s3 = t_best
    best_score = float(max(scores))
    if quick:
        return best, best_score, trace

    def sweep(name, values, simplest, needs_gain=True, plateau=False):
        """Evaluate `values` for one parameter (or a tuple of parameters).
        A non-simplest value is adopted only if it beats the current score by
        the required gain; thresholds use the centre of the best plateau."""
        nonlocal best, best_score
        names = name if isinstance(name, tuple) else (name,)
        results = []
        for val in values:
            vals = val if isinstance(name, tuple) else (val,)
            prm = DecisionParams(**{**best.to_dict(), **dict(zip(names, vals))})
            s = table.score(prm)
            results.append((s, val))
            trace.append((r, str(name), val, s))
        s_best = max(sc for sc, _ in results)
        if plateau:
            v_best = plateau_center([val for _, val in results], [sc for sc, _ in results])
        else:
            v_best = max(results, key=lambda x: x[0])[1]
        current = tuple(getattr(best, n) for n in names)
        current = current if isinstance(name, tuple) else current[0]
        gain_needed = min_imp if (needs_gain and v_best != simplest) else 0.0
        if v_best == current:
            return False
        if s_best > best_score + gain_needed or (not needs_gain and s_best >= best_score):
            vals = v_best if isinstance(name, tuple) else (v_best,)
            best = DecisionParams(**{**best.to_dict(), **dict(zip(names, vals))})
            best_score = s_best
            return True
        return False

    for r in range(int(dcfg.get("rounds", 3))):
        changed = False
        # shared threshold: free to move (centre of the best plateau)
        shared = [(float(t), float(t)) for t in T]
        if best.threshold_s2 == best.threshold_s3:
            changed |= sweep(("threshold_s2", "threshold_s3"), shared, None, needs_gain=False, plateau=True)
        if dcfg.get("per_source_thresholds", True):
            # a separate Source-2 / Source-3 threshold is extra complexity:
            # it must earn the required gain over the shared one
            changed |= sweep("threshold_s2", [float(t) for t in T], None, needs_gain=True, plateau=True)
            changed |= sweep("threshold_s3", [float(t) for t in T], None, needs_gain=True, plateau=True)
        changed |= sweep("margin", [float(m) for m in dcfg.get("margins", [1.0])], 1.0)
        tmax = max(best.threshold_s2, best.threshold_s3)
        gates = [0.0] + sorted({round(min(0.999, tmax + float(o)), 4) for o in dcfg.get("gate_offsets", [0.0])})
        changed |= sweep("gate", gates, 0.0)
        changed |= sweep("max_per_source", list(dcfg.get("max_per_source", [None])), None)
        if table.matchable is not None:
            changed |= sweep("matchable_threshold", [float(x) for x in dcfg.get("matchable_thresholds", [0.0])], 0.0)
        if graph_modes:
            changed |= sweep("graph_mode", list(graph_modes), "off")
        if not changed:
            break
    get_logger().info("Decision search: best Macro F%.1f = %.5f with %s", table.beta, best_score, best.to_dict())
    return best, best_score, trace


# ------------------------------------------------------ singleton model
# pair-level feature columns that entity_features() reads (besides the scores)
ENTITY_INPUT_COLUMNS = ["name_jw", "addr_char_cos", "contradictions", "ctx_mutual_best"]
ENTITY_FEATURES = ["top1", "top2", "margin12", "n_ge_03", "n_ge_05", "n_ge_07", "n_ge_09", "n_cands",
                   "top1_s2", "top1_s3", "top1_lgb", "top1_ce", "top1_model_gap", "top1_name_jw",
                   "top1_addr_cos", "top1_contradictions", "top1_mutual_best", "mean_p", "has_ce"]


def entity_features(scored: pd.DataFrame, entity_idx: np.ndarray, prob_col: str = "p_final") -> pd.DataFrame:
    """Score-distribution features per Source-1 entity (label-free)."""
    df = scored[["s1_idx", "source", prob_col]].copy()
    df["p"] = df[prob_col]
    for c, src in (("p_lgb", "p_lgb"), ("p_ce", "p_ce"), ("name_jw", "name_jw"),
                   ("addr_char_cos", "addr_char_cos"), ("contradictions", "contradictions"),
                   ("ctx_mutual_best", "ctx_mutual_best")):
        df[c] = scored[src].to_numpy() if src in scored else np.nan
    df = df.sort_values(["s1_idx", "p"], ascending=[True, False])
    g = df.groupby("s1_idx", sort=False)
    first = g.head(1).set_index("s1_idx")
    second = g.nth(1).set_index("s1_idx")["p"] if len(df) else pd.Series(dtype=float)
    out = pd.DataFrame(index=pd.Index(entity_idx, name="s1_idx"))
    out["top1"] = first["p"]
    out["top2"] = second
    out["margin12"] = out["top1"] - out["top2"].fillna(0)
    for thr, name in ((0.3, "n_ge_03"), (0.5, "n_ge_05"), (0.7, "n_ge_07"), (0.9, "n_ge_09")):
        out[name] = df[df["p"] >= thr].groupby("s1_idx").size()
    out["n_cands"] = g.size()
    out["top1_s2"] = df[df["source"] == 2].groupby("s1_idx")["p"].max()
    out["top1_s3"] = df[df["source"] == 3].groupby("s1_idx")["p"].max()
    out["top1_lgb"] = first["p_lgb"]
    out["top1_ce"] = first["p_ce"]
    out["top1_model_gap"] = (first["p_lgb"] - first["p_ce"]).abs()
    out["top1_name_jw"] = first["name_jw"]
    out["top1_addr_cos"] = first["addr_char_cos"]
    out["top1_contradictions"] = first["contradictions"]
    out["top1_mutual_best"] = first["ctx_mutual_best"]
    out["mean_p"] = g["p"].mean()
    out["has_ce"] = first["p_ce"].notna().astype(float)
    for c in ("n_ge_03", "n_ge_05", "n_ge_07", "n_ge_09", "n_cands"):
        out[c] = out[c].fillna(0)
    return out[ENTITY_FEATURES].astype(float)


_SINGLETON_PARAMS = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 15, "min_data_in_leaf": 10,
                     "feature_fraction": 0.9, "bagging_fraction": 0.9, "bagging_freq": 1, "lambda_l2": 1.0,
                     "verbose": -1,
                     # tiny data: a single thread avoids OpenMP spin-wait slowdowns
                     # when other processes share the CPU (seen: 289 s vs 0.2 s)
                     "num_threads": 1}


def train_singleton_model(ent_feats: pd.DataFrame, y: np.ndarray, folds: int = 5, seed: int = 42,
                          rounds: int = 200):
    """Out-of-fold P(matchable) on calib entities + final model on all of them."""
    y = np.asarray(y, dtype=int)
    params = {**_SINGLETON_PARAMS, "seed": seed}
    oof = np.full(len(y), np.nan)
    if len(np.unique(y)) < 2 or min(np.bincount(y)) < folds:
        return None, None, {"skipped": "not enough entities of both classes"}
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for tr, te in skf.split(ent_feats, y):
        b = lgb.train(params, lgb.Dataset(ent_feats.iloc[tr], label=y[tr]), num_boost_round=rounds)
        oof[te] = b.predict(ent_feats.iloc[te])
    final = lgb.train(params, lgb.Dataset(ent_feats, label=y), num_boost_round=rounds)
    from sklearn.metrics import roc_auc_score
    report = {"entities": int(len(y)), "matchable_rate": float(y.mean()), "oof_auc": float(roc_auc_score(y, oof))}
    return final, oof, report


In [ ]:
%%writefile src/graph_cleanup.py
"""Graph cleanup of transitive false positives (PDF: GraLMatch / TransClean).

Analysis for THIS task (Source-1-centric):
  * We never have to cluster the whole dataset: the output is, for each
    Source-1 entity, its set of Source-2/3 ids. Full transitive closure is
    neither required nor desirable.
  * Source 1 is de-duplicated, so two different Source-1 nodes are, by
    definition, different businesses. In the bipartite graph
    (S1 nodes <-> vendor nodes, edge weight = match probability) the only
    transitive error that can hurt the output is a vendor record linked to
    two S1 entities: transitivity would claim S1_a == S1_b, contradicting
    de-duplication, so at least one of those edges is a false merge.
  * Therefore a connected component may contain at most ONE Source-1 node.

Modes (chosen on the calib split by Macro F0.5; "off" wins unless cleanup
actually helps):
  off          no cleanup
  exclusive    each vendor record keeps only its highest-probability S1 edge
               (the minimal fix of the above contradiction)
  betweenness  PDF procedure on the thresholded graph: while a component has
               more than one S1 node (or exceeds `max_component_size`), first
               cut the weakest *separating bridge* ("minimum edge cut
               analysis": one edge whose removal splits S1 nodes apart), else
               cut the edge with the highest edge-betweenness centrality
               weighted by (1 - p) (weak bottleneck edges); repeat.

The ground-truth statistic `vendor_records_matched_to_multiple_s1` (labels
report) tells whether the at-most-one-S1-per-vendor assumption holds.
"""
from __future__ import annotations

from collections import deque

import networkx as nx
import numpy as np
import pandas as pd


def exclusive_mask(s1_idx: np.ndarray, v_idx: np.ndarray, p: np.ndarray) -> np.ndarray:
    df = pd.DataFrame({"v": v_idx, "p": p})
    best = df.groupby("v")["p"].transform("max").to_numpy()
    return p >= best


def _separating_bridges(g: nx.Graph):
    """Bridges whose removal leaves S1 nodes on both sides."""
    out = []
    for u, w in list(nx.bridges(g)):
        g.remove_edge(u, w)
        side = nx.node_connected_component(g, u)
        g.add_edge(u, w, **g.graph["_attrs"][(min(u, w), max(u, w))])
        s1_side = sum(1 for n in side if n[0] == "s")
        s1_total = g.graph["_n_s1"]
        if 0 < s1_side < s1_total:
            out.append((u, w))
    return out


def betweenness_mask(s1_idx: np.ndarray, v_idx: np.ndarray, p: np.ndarray, min_edge_prob: float = 0.2,
                     max_component_size: int = 50) -> tuple[np.ndarray, dict]:
    keep = np.ones(len(p), dtype=bool)
    in_graph = p >= min_edge_prob
    edge_id = {}
    G = nx.Graph()
    for i in np.where(in_graph)[0]:
        a, b = ("s", int(s1_idx[i])), ("v", int(v_idx[i]))
        G.add_edge(a, b, p=float(p[i]), dist=1.0 - float(p[i]) + 1e-6)
        edge_id[(a, b)] = i
    stats = {"edges": G.number_of_edges(), "components": 0, "conflict_components": 0,
             "oversized_components": 0, "edges_removed": 0, "bridges_removed": 0}
    queue = deque()
    for comp in nx.connected_components(G):
        stats["components"] += 1
        n_s1 = sum(1 for n in comp if n[0] == "s")
        if n_s1 > 1 or len(comp) > max_component_size:
            stats["conflict_components"] += int(n_s1 > 1)
            stats["oversized_components"] += int(len(comp) > max_component_size)
            queue.append(comp)
    guard = 0
    while queue and guard < 100_000:
        guard += 1
        comp = queue.popleft()
        sub = G.subgraph(comp).copy()
        n_s1 = sum(1 for n in sub if n[0] == "s")
        if n_s1 <= 1 and sub.number_of_nodes() <= max_component_size:
            continue
        if sub.number_of_edges() == 0:
            continue
        sub.graph["_attrs"] = {(min(u, w), max(u, w)): d for u, w, d in sub.edges(data=True)}
        sub.graph["_n_s1"] = n_s1
        cand = _separating_bridges(sub) if n_s1 > 1 else []
        if cand:
            u, w = min(cand, key=lambda e: sub.edges[e]["p"])
            stats["bridges_removed"] += 1
        else:
            eb = nx.edge_betweenness_centrality(sub, weight="dist")
            u, w = max(eb, key=lambda e: eb[e] * (1.0 - sub.edges[e]["p"]) + 1e-9 * eb[e])
        G.remove_edge(u, w)
        stats["edges_removed"] += 1
        a, b = (u, w) if u[0] == "s" else (w, u)
        keep[edge_id[(a, b)]] = False
        for new_comp in nx.connected_components(G.subgraph(comp)):
            queue.append(set(new_comp))
    return keep, stats


def normalize_mode(mode) -> str:
    """YAML turns a bare `off` into False - accept that spelling too."""
    if mode is False or mode is None:
        return "off"
    return str(mode).lower()


def graph_keep_mask(pairs: pd.DataFrame, prob_col: str, mode: str, min_edge_prob: float = 0.2,
                    max_component_size: int = 50) -> tuple[np.ndarray, dict]:
    mode = normalize_mode(mode)
    s1 = pairs["s1_idx"].to_numpy()
    v = pairs["v_idx"].to_numpy()
    p = pairs[prob_col].to_numpy(dtype=float)
    if mode == "off":
        return np.ones(len(p), dtype=bool), {"mode": "off"}
    if mode == "exclusive":
        keep = exclusive_mask(s1, v, p)
        return keep, {"mode": "exclusive", "edges_removed": int((~keep).sum())}
    if mode == "betweenness":
        keep, stats = betweenness_mask(s1, v, p, min_edge_prob, max_component_size)
        stats["mode"] = "betweenness"
        return keep, stats
    raise ValueError(f"unknown graph mode {mode}")


In [ ]:
%%writefile src/train_biencoder.py
"""Semantic blocking: supervised-contrastive bi-encoder + ANN retrieval
(PDF: "Supervised Contrastive Blocking for Semantic Alignment", SC-Block).

* Encoder: a pre-trained transformer (default MiniLM, PDF: "a RoBERTa variant
  or a distilled MiniLM") with mean pooling and L2 normalisation, shared by
  Source-1 and vendor records (siamese towers; records never interact, so all
  vendor embeddings can be pre-computed and indexed).
* Loss: batch-wise Supervised Contrastive loss (Khosla et al. 2020)

      L = sum_i  -1/|P(i)| sum_{p in P(i)} log( exp(z_i.z_p/t) / sum_{a in A(i)} exp(z_i.z_a/t) )

  Labels come from the ground truth: a Source-1 entity and all of its
  labelled Source-2/3 records share one label; singletons and unlabelled
  vendor records get unique labels and act as in-batch negatives only.
* Hard batching: entities sharing a blocking key (name prefix) are placed in
  the same batch so in-batch negatives are look-alikes; extra vendor records
  with the same keys are added as hard negatives.
* Retrieval: FAISS inner-product index (exact `flat` or `hnsw`) per source,
  top-k neighbours per Source-1 entity; numpy fallback if faiss is missing.

AdaFlood (named in the PDF) is NOT implemented: the PDF gives no formula and
its per-sample flood levels require auxiliary models whose training protocol
for SupCon is not specified. `flood_level` implements only the original,
constant Flooding regulariser (Ishida et al. 2020, |L - b| + b); it is
disabled by default and is not a substitute for AdaFlood.

Pre-trained weights are loaded from a LOCAL directory (`local_files_only`);
nothing is downloaded during training or matching.
"""
from __future__ import annotations

import json
import math
import os
import random

import numpy as np
import pandas as pd

from .utils import ensure_dir, get_logger

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from transformers import AutoModel, AutoTokenizer
except ImportError:  # pragma: no cover
    torch = None

try:
    import faiss  # type: ignore
except ImportError:  # pragma: no cover
    faiss = None


def require_torch():
    if torch is None:
        raise ImportError("Full mode needs torch and transformers: pip install -r requirements-full.txt")


def resolve_device(name: str = "auto"):
    require_torch()
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def record_texts(df: pd.DataFrame, text_source: str = "normalized") -> list[str]:
    if text_source == "raw":
        return [f"{n} | {a}".strip(" |") for n, a in zip(df["name"], df["address"])]
    return [f"{n} | {a}".strip(" |") for n, a in zip(df["name_clean"], df["addr_clean"])]


def linear_warmup_schedule(optimizer, warmup_steps: int, total_steps: int):
    def fn(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


if torch is not None:
    class BiEncoder(nn.Module):
        def __init__(self, path: str, pooling: str = "mean", local_files_only: bool = True):
            super().__init__()
            self.backbone = AutoModel.from_pretrained(path, local_files_only=local_files_only)
            self.pooling = pooling

        def forward(self, input_ids, attention_mask, **kw):
            out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kw).last_hidden_state
            if self.pooling == "cls":
                emb = out[:, 0]
            else:
                m = attention_mask.unsqueeze(-1).to(out.dtype)
                emb = (out * m).sum(1) / m.sum(1).clamp(min=1e-6)
            return F.normalize(emb, dim=-1)

    def supcon_loss(z: "torch.Tensor", labels: "torch.Tensor", temperature: float = 0.07) -> "torch.Tensor":
        """Supervised contrastive loss; anchors without positives are ignored."""
        n = z.shape[0]
        sim = z @ z.T / temperature
        self_mask = torch.eye(n, dtype=torch.bool, device=z.device)
        sim = sim.masked_fill(self_mask, -1e9)
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        pos = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~self_mask
        n_pos = pos.sum(1)
        valid = n_pos > 0
        if not valid.any():
            return (z * 0).sum()
        mean_log_prob_pos = (log_prob * pos).sum(1)[valid] / n_pos[valid]
        return -mean_log_prob_pos.mean()


def _batches(groups: list[list[int]], keys: list[str], batch_size: int, rng: random.Random,
             hard: bool) -> list[list[int]]:
    """Pack whole groups (S1 record + its positives) into batches."""
    order = list(range(len(groups)))
    rng.shuffle(order)
    if hard:
        order.sort(key=lambda g: keys[g])  # stable: random within equal keys
    batches, cur, cur_n = [], [], 0
    for g in order:
        if cur and cur_n + len(groups[g]) > batch_size:
            batches.append(cur)
            cur, cur_n = [], 0
        cur.append(g)
        cur_n += len(groups[g])
    if cur:
        batches.append(cur)
    rng.shuffle(batches)
    return batches


def train_biencoder(s1: pd.DataFrame, v: pd.DataFrame, true_pairs: pd.DataFrame, split: np.ndarray,
                    cfg: dict, out_dir: str, seed: int = 42) -> dict:
    """Train on `fit` entities, early-stop on recall@k of `fit_val` entities."""
    require_torch()
    log = get_logger()
    bcfg = cfg.get("biencoder", {})
    device = resolve_device(bcfg.get("device", "auto"))
    rng = random.Random(seed)
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(bcfg["pretrained"], local_files_only=bcfg.get("local_files_only", True))
    model = BiEncoder(bcfg["pretrained"], bcfg.get("pooling", "mean"),
                      bcfg.get("local_files_only", True)).to(device)
    text_src = bcfg.get("text_source", "normalized")
    t1, tv = record_texts(s1, text_src), record_texts(v, text_src)
    max_pos = int(bcfg.get("max_positives_per_entity", 4))

    pos_by_s1 = true_pairs.groupby("s1_idx")["v_idx"].apply(list).to_dict()
    fit_ents = np.where(split == "fit")[0]
    val_ents = np.where(split == "fit_val")[0]
    # items: ("s", idx) for S1 records, ("v", idx) for vendor records
    items: list[tuple[str, int]] = []
    groups, keys = [], []
    for e in fit_ents:
        g = [len(items)]
        items.append(("s", int(e)))
        for vi in pos_by_s1.get(e, [])[:max_pos]:
            g.append(len(items))
            items.append(("v", int(vi)))
        groups.append(g)
        keys.append(s1["name_nospace"].iat[e][:3] + "|" + s1["addr_postcode"].iat[e])
    labels_all = np.empty(len(items), dtype=np.int64)
    for gi, g in enumerate(groups):
        labels_all[g] = gi
    positive_v = set(true_pairs["v_idx"].tolist())
    neg_pool = [i for i in range(len(v)) if i not in positive_v]
    neg_by_key: dict[str, list[int]] = {}
    for i in neg_pool:
        neg_by_key.setdefault(v["name_nospace"].iat[i][:3], []).append(i)
    extra_n = int(bcfg.get("extra_negatives_per_batch", 32))

    bs = int(bcfg.get("batch_size", 128))
    epochs = int(bcfg.get("epochs", 5))
    steps_per_epoch = max(1, len(_batches(groups, keys, bs, random.Random(0), False)))
    opt = torch.optim.AdamW(model.parameters(), lr=float(bcfg.get("lr", 3e-5)),
                            weight_decay=float(bcfg.get("weight_decay", 0.01)))
    total = steps_per_epoch * epochs
    sched = linear_warmup_schedule(opt, int(float(bcfg.get("warmup_ratio", 0.1)) * total), total)
    temp = float(bcfg.get("temperature", 0.07))
    flood = bcfg.get("flood_level")
    max_len = int(bcfg.get("max_length", 64))
    k_eval = int(cfg.get("blocking", {}).get("methods", {}).get("semantic", {}).get("top_k", 30))
    best_recall, history = -1.0, []
    ensure_dir(out_dir)

    def encode_batch(texts):
        enc = tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        return model(enc["input_ids"], enc["attention_mask"])

    for epoch in range(epochs):
        model.train()
        losses = []
        for batch_groups in _batches(groups, keys, bs, rng, bool(bcfg.get("hard_batching", True))):
            idx = [i for g in batch_groups for i in groups[g]]
            texts = [t1[items[i][1]] if items[i][0] == "s" else tv[items[i][1]] for i in idx]
            labels = list(labels_all[idx])
            if extra_n and neg_pool:
                bkeys = {s1["name_nospace"].iat[items[groups[g][0]][1]][:3] for g in batch_groups}
                cand = [j for kk in bkeys for j in neg_by_key.get(kk, [])]
                picks = rng.sample(cand, min(len(cand), extra_n // 2))
                picks += rng.sample(neg_pool, min(len(neg_pool), extra_n - len(picks)))
                texts += [tv[j] for j in picks]
                labels += list(range(-1, -1 - len(picks), -1))  # unique negative labels
            z = encode_batch(texts)
            loss = supcon_loss(z, torch.tensor(labels, device=device), temp)
            if flood is not None:
                loss = (loss - float(flood)).abs() + float(flood)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(float(loss.detach().cpu()))
        recall = _recall_at_k(model, tok, t1, tv, val_ents, pos_by_s1, v, k_eval, max_len, device,
                              int(bcfg.get("encode_batch_size", 256)), seed)
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)) if losses else None,
                        "val_recall_at_k": recall})
        log.info("bi-encoder epoch %d: loss=%.4f  fit_val recall@%d=%.4f", epoch + 1,
                 history[-1]["loss"] or float("nan"), k_eval, recall)
        if recall > best_recall:
            best_recall = recall
            model.backbone.save_pretrained(out_dir)
            tok.save_pretrained(out_dir)
            with open(os.path.join(out_dir, "biencoder_config.json"), "w") as fh:
                json.dump({"pooling": model.pooling, "max_length": max_len, "text_source": text_src}, fh)
    report = {"history": history, "best_fit_val_recall_at_k": best_recall, "k": k_eval,
              "train_entities": int(len(fit_ents)), "adaflood": "not implemented (see module docstring)",
              "flood_level": flood}
    with open(os.path.join(out_dir, "biencoder_report.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    return report


def _recall_at_k(model, tok, t1, tv, val_ents, pos_by_s1, v, k, max_len, device, bs, seed) -> float:
    """Recall@k of fit_val positives against a vendor pool (all positives of
    the val entities + a random sample of other vendor records)."""
    ents = [e for e in val_ents if pos_by_s1.get(e)]
    if not ents:
        return 0.0
    rng = np.random.default_rng(seed)
    pool = set(vi for e in ents for vi in pos_by_s1[e])
    others = rng.choice(len(v), size=min(len(v), 20000), replace=False)
    pool = np.array(sorted(pool | set(others.tolist())))
    ze = _encode(model, tok, [t1[e] for e in ents], max_len, device, bs)
    zv = _encode(model, tok, [tv[i] for i in pool], max_len, device, bs)
    src = v["source"].to_numpy()[pool]
    hits = total = 0
    for s in (2, 3):
        m = src == s
        if not m.any():
            continue
        sims = ze @ zv[m].T
        kk = min(k, int(m.sum()))
        top = np.argpartition(-sims, kk - 1, axis=1)[:, :kk]
        pool_s = pool[m]
        for row, e in enumerate(ents):
            truth = {vi for vi in pos_by_s1[e] if v["source"].iat[vi] == s}
            total += len(truth)
            hits += len(truth & set(pool_s[top[row]].tolist()))
    return hits / total if total else 0.0


def _encode(model, tok, texts, max_len, device, bs) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i + bs], padding=True, truncation=True, max_length=max_len,
                      return_tensors="pt").to(device)
            out.append(model(enc["input_ids"], enc["attention_mask"]).float().cpu().numpy())
    if not out:
        return np.zeros((0, 1), dtype=np.float32)
    return np.vstack(out).astype(np.float32)


class SemanticRetriever:
    """Encode records with the trained bi-encoder and search FAISS indices."""

    def __init__(self, model_dir: str, cfg: dict):
        require_torch()
        bcfg = cfg.get("biencoder", {})
        with open(os.path.join(model_dir, "biencoder_config.json")) as fh:
            meta = json.load(fh)
        self.device = resolve_device(bcfg.get("device", "auto"))
        self.tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        self.model = BiEncoder(model_dir, meta.get("pooling", "mean"), local_files_only=True).to(self.device)
        self.max_len = int(meta.get("max_length", 64))
        self.text_source = meta.get("text_source", "normalized")
        self.bs = int(bcfg.get("encode_batch_size", 256))
        self.index_type = bcfg.get("faiss_index", "flat")
        self._key = None
        self.indices: dict[int, object] = {}

    def prepare(self, s1: pd.DataFrame, v: pd.DataFrame):
        key = (len(s1), len(v), s1["id"].iat[0] if len(s1) else "", v["id"].iat[0] if len(v) else "")
        if key == self._key:
            return
        log = get_logger()
        self.z1 = _encode(self.model, self.tok, record_texts(s1, self.text_source), self.max_len, self.device, self.bs)
        self.zv = _encode(self.model, self.tok, record_texts(v, self.text_source), self.max_len, self.device, self.bs)
        self.src = v["source"].to_numpy()
        self.indices = {}
        for s in (2, 3):
            rows = np.where(self.src == s)[0]
            self.indices[s] = (self._build_index(self.zv[rows]), rows)
        self._key = key
        log.info("Semantic retriever: encoded %d S1 / %d vendor records (index=%s, faiss=%s)",
                 len(s1), len(v), self.index_type, faiss is not None)

    def _build_index(self, emb: np.ndarray):
        if faiss is None or len(emb) == 0:
            return emb
        d = emb.shape[1]
        if self.index_type == "hnsw":
            index = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
            index.hnsw.efSearch = 128
        else:
            index = faiss.IndexFlatIP(d)
        index.add(np.ascontiguousarray(emb, dtype=np.float32))
        return index

    def search(self, s1: pd.DataFrame, v: pd.DataFrame, source: int, k: int) -> pd.DataFrame:
        self.prepare(s1, v)
        index, rows = self.indices[source]
        if len(rows) == 0:
            return pd.DataFrame({"s1_idx": [], "v_idx": [], "sim": []})
        kk = min(k, len(rows))
        if faiss is not None and not isinstance(index, np.ndarray):
            sims, idx = index.search(np.ascontiguousarray(self.z1, dtype=np.float32), kk)
        else:
            sims_full = self.z1 @ index.T
            idx = np.argpartition(-sims_full, kk - 1, axis=1)[:, :kk]
            sims = np.take_along_axis(sims_full, idx, axis=1)
        valid = idx >= 0
        s1_idx = np.repeat(np.arange(len(self.z1)), kk).reshape(-1, kk)[valid]
        return pd.DataFrame({"s1_idx": s1_idx.astype(np.int64), "v_idx": rows[idx[valid]].astype(np.int64),
                             "sim": sims[valid].astype(np.float32)})

    def save_indices(self, out_dir: str):
        ensure_dir(out_dir)
        for s, (index, rows) in self.indices.items():
            np.save(os.path.join(out_dir, f"semantic_rows_s{s}.npy"), rows)
            if faiss is not None and not isinstance(index, np.ndarray):
                faiss.write_index(index, os.path.join(out_dir, f"semantic_s{s}.faiss"))
            else:
                np.save(os.path.join(out_dir, f"semantic_emb_s{s}.npy"), index)


In [ ]:
%%writefile src/train_crossencoder.py
"""Semantic matcher: Ditto-style Transformer cross-encoder (default backbone
DeBERTa-v3), PDF "Deep Semantic Matching with Transformer Cross-Encoders".

Serialisation (Ditto):
    [CLS] COL name VAL <name> COL address VAL <address> [SEP]
          COL name VAL <name> COL address VAL <address> [SEP]
The tokenizer inserts [CLS]/[SEP]; the [CLS] hidden state feeds a linear
head producing one match logit.

Training techniques from the PDF:
 1. Domain-knowledge injection: offline regexes + the parsed address wrap
    discriminative alphanumerics in special tags - postcodes as
    "[PC] 95113 [/PC]", other digit-bearing tokens (house numbers, suites) as
    "[NUM] 500 [/NUM]". The tags are added to the tokenizer vocabulary.
 2. MixDA augmentation (Ditto): an augmented copy of each training pair is
    produced with one operator - span deletion (del), span shuffle (swap),
    attribute drop (drop_col), token deletion (token_del), attribute
    shuffle (attr_shuffle) or entity swap (entry_swap) - and the [CLS]
    encodings of original and augmented input are interpolated,
    h = lam*h_orig + (1-lam)*h_aug, lam ~ Beta(a, a), lam = max(lam, 1-lam),
    before the classification head; the label is the original label.
 3. TF-IDF summarisation: values longer than `tfidf_summarize_max_words`
    keep only their highest-IDF words (original order preserved).
 4. Focal loss FL = -alpha_t (1 - p_t)^gamma log(p_t) against the extreme
    class imbalance of candidate pairs.
Hard negatives: the highest-scoring LightGBM negatives of each training
entity (plus a few random ones) form the negative training pairs.
"""
from __future__ import annotations

import inspect
import json
import math
import os
import random
import re

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, log_loss

from .train_biencoder import linear_warmup_schedule, require_torch, resolve_device
from .utils import ensure_dir, get_logger

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from transformers import AutoModel, AutoTokenizer
except ImportError:  # pragma: no cover
    torch = None

SPECIAL_TOKENS = ["[PC]", "[/PC]", "[NUM]", "[/NUM]"]
_UNIT_RE = re.compile(r"\[(PC|NUM)\] \S+ \[/\1\]|\S+")
_DIGIT_RE = re.compile(r"\d")


class DittoSerializer:
    def __init__(self, cfg: dict, idf: dict | None = None):
        ccfg = cfg.get("crossencoder", {})
        self.text_source = ccfg.get("text_source", "raw")
        self.domain_tags = bool(ccfg.get("domain_tags", True))
        self.max_words = int(ccfg.get("tfidf_summarize_max_words", 40))
        self.idf = idf or {}

    def _summarize(self, words: list[str]) -> list[str]:
        if len(words) <= self.max_words or not self.idf:
            return words
        default = max(self.idf.values()) if self.idf else 1.0
        scores = [self.idf.get(w.lower(), default) for w in words]
        keep = set(np.argsort(scores)[::-1][: self.max_words].tolist())
        return [w for i, w in enumerate(words) if i in keep]

    def _tag(self, words: list[str], postcode: str) -> list[str]:
        """Domain-knowledge injection: tag postcodes and digit-bearing tokens."""
        if not self.domain_tags:
            return words
        out = []
        pc = postcode.replace(" ", "").lower()
        for w in words:
            core = re.sub(r"[^\w\-/]", "", w).lower()
            is_pc = bool(pc and core) and (core == pc or core.startswith(pc + "-")
                                           or (len(core) >= 3 and core in pc and not core.isalpha()))
            if is_pc:
                out.append(f"[PC] {w} [/PC]")
            elif _DIGIT_RE.search(w):
                out.append(f"[NUM] {w} [/NUM]")
            else:
                out.append(w)
        return out

    def serialize_frame(self, df: pd.DataFrame) -> list[str]:
        if self.text_source == "raw":
            names, addrs = df["name"], df["address"]
        else:
            names, addrs = df["name_clean"], df["addr_clean"]
        out = []
        for n, a, pc in zip(names, addrs, df["addr_postcode"]):
            aw = self._tag(self._summarize(str(a).split()), pc)
            nw = self._tag(self._summarize(str(n).split()), "")
            out.append(f"COL name VAL {' '.join(nw)} COL address VAL {' '.join(aw)}")
        return out


# ------------------------------------------------------------------ MixDA
def _split_attrs(text: str) -> list[tuple[str, str]]:
    parts = re.split(r"COL (\w+) VAL ?", text)
    return [(parts[i], parts[i + 1].strip()) for i in range(1, len(parts) - 1, 2)]


def _join_attrs(attrs: list[tuple[str, str]]) -> str:
    return " ".join(f"COL {k} VAL {val}".strip() for k, val in attrs)


def augment_pair(a: str, b: str, op: str, rng: random.Random) -> tuple[str, str]:
    """One Ditto augmentation operator on a serialised pair."""
    if op == "entry_swap":
        return b, a
    side = rng.randint(0, 1)
    attrs = _split_attrs(a if side == 0 else b)
    if not attrs:
        return a, b
    if op == "attr_shuffle":
        rng.shuffle(attrs)
    elif op == "drop_col":
        i = rng.randrange(len(attrs))
        attrs[i] = (attrs[i][0], "")
    else:
        i = rng.randrange(len(attrs))
        units = [m.group(0) for m in _UNIT_RE.finditer(attrs[i][1])]
        if units:
            if op == "token_del" and len(units) > 1:
                del units[rng.randrange(len(units))]
            elif op in ("del", "swap") and len(units) > 1:
                span = rng.randint(1, min(4, len(units) - 1 if op == "del" else len(units)))
                start = rng.randrange(0, len(units) - span + 1)
                if op == "del":
                    del units[start:start + span]
                else:
                    seg = units[start:start + span]
                    rng.shuffle(seg)
                    units[start:start + span] = seg
            attrs[i] = (attrs[i][0], " ".join(units))
    new = _join_attrs(attrs)
    return (new, b) if side == 0 else (a, new)


if torch is not None:
    class CrossEncoder(nn.Module):
        def __init__(self, path: str, local_files_only: bool = True, dropout: float = 0.1):
            super().__init__()
            self.backbone = AutoModel.from_pretrained(path, local_files_only=local_files_only)
            hidden = self.backbone.config.hidden_size
            self.dropout = nn.Dropout(dropout)
            self.head = nn.Linear(hidden, 1)
            self._accepts = set(inspect.signature(self.backbone.forward).parameters)

        def encode(self, enc: dict) -> "torch.Tensor":
            kw = {k: val for k, val in enc.items() if k in self._accepts}
            return self.backbone(**kw).last_hidden_state[:, 0]

        def forward(self, enc: dict, aug_enc: dict | None = None, lam: float | None = None):
            h = self.encode(enc)
            if aug_enc is not None and lam is not None:
                h = lam * h + (1.0 - lam) * self.encode(aug_enc)
            return self.head(self.dropout(h)).squeeze(-1)

    def focal_loss(logits, targets, gamma: float = 2.0, alpha: float | None = None):
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = ce * (1 - p_t) ** gamma
        if alpha is not None:
            loss = loss * (alpha * targets + (1 - alpha) * (1 - targets))
        return loss.mean()


def _amp_dtype(device):
    """bf16 where supported (DeBERTa-v3 is prone to fp16 overflow), else fp16."""
    if device.type == "cuda" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def _tokenize(tok, a: list[str], b: list[str], max_len: int, device):
    enc = tok(a, b, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return {k: val.to(device) for k, val in enc.items()}


def _load_tokenizer(path: str, local_only: bool):
    tok = AutoTokenizer.from_pretrained(path, local_files_only=local_only)
    tok.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    return tok


def train_crossencoder(train_a: list[str], train_b: list[str], train_y: np.ndarray, val_a: list[str],
                       val_b: list[str], val_y: np.ndarray, cfg: dict, out_dir: str, seed: int = 42) -> dict:
    require_torch()
    log = get_logger()
    ccfg = cfg.get("crossencoder", {})
    device = resolve_device(ccfg.get("device", "auto"))
    rng = random.Random(seed)
    torch.manual_seed(seed)
    local_only = ccfg.get("local_files_only", True)
    tok = _load_tokenizer(ccfg["pretrained"], local_only)
    model = CrossEncoder(ccfg["pretrained"], local_only)
    model.backbone.resize_token_embeddings(len(tok))
    model.to(device)
    max_len = int(ccfg.get("max_length", 128))
    bs = int(ccfg.get("batch_size", 32))
    epochs = int(ccfg.get("epochs", 3))
    gamma = float(ccfg.get("focal_gamma", 2.0))
    alpha = ccfg.get("focal_alpha")
    alpha = None if alpha is None else float(alpha)
    mix = ccfg.get("mixda", {})
    use_mix = bool(mix.get("enabled", True))
    ops = list(mix.get("ops", ["del", "swap", "drop_col"]))
    mix_alpha = float(mix.get("alpha", 0.8))
    use_amp = bool(ccfg.get("fp16", True)) and device.type == "cuda"
    amp_dtype = _amp_dtype(device)
    scaler = torch.amp.GradScaler("cuda") if use_amp and amp_dtype == torch.float16 else None
    n = len(train_y)
    steps = math.ceil(n / bs) * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=float(ccfg.get("lr", 2e-5)),
                            weight_decay=float(ccfg.get("weight_decay", 0.01)))
    sched = linear_warmup_schedule(opt, int(float(ccfg.get("warmup_ratio", 0.1)) * steps), steps)
    ensure_dir(out_dir)
    best_ap, history = -1.0, []
    np_rng = np.random.default_rng(seed)
    log.info("Cross-encoder: %d training pairs (%d pos), %d validation pairs, device=%s, MixDA=%s",
             n, int(train_y.sum()), len(val_y), device, use_mix)
    for epoch in range(epochs):
        model.train()
        perm = np_rng.permutation(n)
        losses = []
        for s in range(0, n, bs):
            idx = perm[s:s + bs]
            a = [train_a[i] for i in idx]
            b = [train_b[i] for i in idx]
            y = torch.tensor(train_y[idx], dtype=torch.float32, device=device)
            enc = _tokenize(tok, a, b, max_len, device)
            aug_enc, lam = None, None
            if use_mix and ops:
                aug = [augment_pair(x, z, rng.choice(ops), rng) for x, z in zip(a, b)]
                aug_enc = _tokenize(tok, [p[0] for p in aug], [p[1] for p in aug], max_len, device)
                lam = float(np_rng.beta(mix_alpha, mix_alpha))
                lam = max(lam, 1 - lam)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(enc, aug_enc, lam)
            loss = focal_loss(logits.float(), y, gamma, alpha)
            opt.zero_grad()
            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            losses.append(float(loss.detach().cpu()))
        val_p = _predict(model, tok, val_a, val_b, max_len, device, int(ccfg.get("eval_batch_size", 128)), use_amp)
        rec = {"epoch": epoch + 1, "train_loss": float(np.mean(losses)) if losses else None}
        if len(np.unique(val_y)) == 2:
            rec["val_average_precision"] = float(average_precision_score(val_y, val_p))
            rec["val_logloss"] = float(log_loss(val_y, np.clip(val_p, 1e-7, 1 - 1e-7)))
        history.append(rec)
        log.info("cross-encoder epoch %d: %s", epoch + 1, rec)
        ap = rec.get("val_average_precision", -rec["train_loss"] if rec["train_loss"] is not None else 0)
        if ap > best_ap:
            best_ap = ap
            save_crossencoder(model, tok, out_dir, max_len, ccfg)
    report = {"history": history, "best_val_average_precision": best_ap, "train_pairs": int(n),
              "train_positives": int(train_y.sum()), "mixda_ops": ops if use_mix else [], "focal_gamma": gamma,
              "focal_alpha": alpha}
    with open(os.path.join(out_dir, "crossencoder_report.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    return report


def save_crossencoder(model, tok, out_dir: str, max_len: int, ccfg: dict):
    ensure_dir(out_dir)
    model.backbone.save_pretrained(os.path.join(out_dir, "backbone"))
    tok.save_pretrained(os.path.join(out_dir, "backbone"))
    torch.save(model.head.state_dict(), os.path.join(out_dir, "head.pt"))
    with open(os.path.join(out_dir, "crossencoder_config.json"), "w") as fh:
        json.dump({"max_length": max_len, "text_source": ccfg.get("text_source", "raw"),
                   "domain_tags": bool(ccfg.get("domain_tags", True))}, fh)


def _predict(model, tok, a, b, max_len, device, bs, use_amp=False) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for s in range(0, len(a), bs):
            enc = _tokenize(tok, a[s:s + bs], b[s:s + bs], max_len, device)
            with torch.autocast(device_type=device.type, dtype=_amp_dtype(device), enabled=use_amp):
                logits = model(enc)
            out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


class CrossEncoderPredictor:
    def __init__(self, model_dir: str, cfg: dict):
        require_torch()
        ccfg = cfg.get("crossencoder", {})
        with open(os.path.join(model_dir, "crossencoder_config.json")) as fh:
            self.meta = json.load(fh)
        self.device = resolve_device(ccfg.get("device", "auto"))
        bdir = os.path.join(model_dir, "backbone")
        self.tok = AutoTokenizer.from_pretrained(bdir, local_files_only=True)
        self.model = CrossEncoder(bdir, local_files_only=True)
        self.model.head.load_state_dict(torch.load(os.path.join(model_dir, "head.pt"), map_location="cpu"))
        self.model.to(self.device)
        self.bs = int(ccfg.get("eval_batch_size", 128))
        self.use_amp = bool(ccfg.get("fp16", True)) and self.device.type == "cuda"

    def predict(self, a: list[str], b: list[str]) -> np.ndarray:
        return _predict(self.model, self.tok, a, b, int(self.meta["max_length"]), self.device, self.bs, self.use_amp)


In [ ]:
%%writefile src/inference.py
"""Shared scoring + decision code. The SAME functions score the holdout split
during training and the test set during prediction, so the reported holdout
numbers describe exactly the procedure used for the submission.

    candidates -> features -> LightGBM -> isotonic -> cascade -> cross-encoder
    -> isotonic -> blend -> final calibration -> graph cleanup -> decision rules
    (+ singleton gate) -> one id set per Source-1 entity
"""
from __future__ import annotations

import os

import lightgbm as lgb
import numpy as np
import pandas as pd

from .decision import DecisionParams, DecisionTable, entity_features
from .graph_cleanup import graph_keep_mask
from .scoring import cascade_mask
from .features import predict_in_blocks
from .train_lightgbm import load_lightgbm
from .utils import get_logger, load_json, load_pickle


class TrainedPipeline:
    """Every fitted artefact needed to score a new dataset."""

    def __init__(self, model_dir: str, cfg: dict):
        path = os.path.join(model_dir, "pipeline.json")
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} not found - run `python train.py --config ...` first")
        self.dir = model_dir
        self.cfg = cfg
        self.meta = load_json(path)
        self.feature_columns: list[str] = self.meta["feature_columns"]
        self.booster = load_lightgbm(model_dir)
        self.cal_lgb = load_pickle(os.path.join(model_dir, "calibrator_lgb.pkl"))
        self.cal_ce = load_pickle(os.path.join(model_dir, "calibrator_ce.pkl")) if self.meta.get("has_ce") else None
        self.blender = load_pickle(os.path.join(model_dir, "blender.pkl"))
        self.cal_final = load_pickle(os.path.join(model_dir, "calibrator_final.pkl"))
        self.decision = DecisionParams.from_dict(self.meta["decision"])
        sm = os.path.join(model_dir, "singleton_model.txt")
        self.singleton_model = lgb.Booster(model_file=sm) if self.meta.get("has_singleton_model") and os.path.exists(sm) else None
        self.graph_cfg = self.meta.get("graph", {})
        self.cascade_cfg = self.meta.get("cascade", {})
        self._ce = None
        self._serializer_idf = None

    @property
    def has_ce(self) -> bool:
        return bool(self.meta.get("has_ce"))

    def ce_predictor(self):
        if self._ce is None:
            from .train_crossencoder import CrossEncoderPredictor
            self._ce = CrossEncoderPredictor(os.path.join(self.dir, "crossencoder"), self.cfg)
        return self._ce

    def serializer(self):
        from .train_crossencoder import DittoSerializer
        if self._serializer_idf is None:
            self._serializer_idf = load_pickle(os.path.join(self.dir, "serializer_idf.pkl"))
        return DittoSerializer(self.cfg, self._serializer_idf)


def score_candidates(pairs: pd.DataFrame, feats, s1: pd.DataFrame, v: pd.DataFrame,
                     pipe: TrainedPipeline, ce_rows: np.ndarray | None = None) -> pd.DataFrame:
    """Return pairs + p_lgb_raw, p_lgb, sent_to_ce, p_ce_raw, p_ce, p_blend, p_final.

    `feats` is a DataFrame or a disk-backed FeatureMatrix (scored in blocks)."""
    log = get_logger()
    out = pairs[["s1_idx", "v_idx", "source"]].copy()
    raw = predict_in_blocks(pipe.booster, feats, pipe.feature_columns)
    out["p_lgb_raw"] = raw
    out["p_lgb"] = pipe.cal_lgb.transform(raw)
    out["sent_to_ce"] = False
    out["p_ce_raw"] = np.nan
    out["p_ce"] = np.nan
    if pipe.has_ce:
        mask = cascade_mask(out, out["p_lgb"].to_numpy(), pipe.cascade_cfg)
        if ce_rows is not None:
            mask &= ce_rows
        out["sent_to_ce"] = mask
        if mask.any():
            ser = pipe.serializer()
            t1, tv = ser.serialize_frame(s1), ser.serialize_frame(v)
            ia = out.loc[mask, "s1_idx"].to_numpy()
            ib = out.loc[mask, "v_idx"].to_numpy()
            log.info("Cascade: %d of %d pairs (%.1f%%) sent to the cross-encoder", int(mask.sum()), len(out),
                     100 * mask.mean())
            p = pipe.ce_predictor().predict([t1[i] for i in ia], [tv[j] for j in ib])
            out.loc[mask, "p_ce_raw"] = p
            out.loc[mask, "p_ce"] = pipe.cal_ce.transform(p)
    has_ce = out["p_ce"].notna().to_numpy()
    out["p_blend"] = pipe.blender.transform(out["p_lgb"].to_numpy(), out["p_ce"].to_numpy(), has_ce)
    out["p_final"] = pipe.cal_final.transform(out["p_blend"].to_numpy())
    return out


def decide(scored: pd.DataFrame, entity_idx: np.ndarray, pipe: TrainedPipeline,
           params: DecisionParams | None = None, extra_cols: pd.DataFrame | None = None):
    """Apply graph cleanup + decision rules; returns (selected, info)."""
    params = params or pipe.decision
    gcfg = pipe.graph_cfg
    keep, gstats = graph_keep_mask(scored, "p_final", params.graph_mode, float(gcfg.get("min_edge_prob", 0.2)),
                                   int(gcfg.get("max_component_size", 50)))
    matchable = None
    if pipe.singleton_model is not None and params.matchable_threshold > 0:
        src = scored if extra_cols is None else pd.concat([scored, extra_cols], axis=1)
        ef = entity_features(src, entity_idx)
        matchable = pipe.singleton_model.predict(ef)
    table = DecisionTable.build(scored, "p_final", entity_idx, graph_masks={params.graph_mode: keep},
                                matchable=matchable)
    selected = table.select(params)
    info = {"graph": gstats, "matchable": matchable, "graph_keep": keep}
    return selected, info


In [ ]:
%%writefile src/submission.py
"""Output generation, format validation and packaging.

matching_results.tsv  one row per Source-1 entity (every entity, in the
                      original Source-1 order); its predicted Source-2/3 ids as
                      a comma-separated list, EMPTY for predicted singletons.
                      Columns mirror the label file (transcript: the label
                      format "mirrors exactly what you'll submit") unless a
                      sample submission is configured.
candidate_pairs.tsv   the blocking output BEFORE any model scoring
                      ("candidate/unscored pairs").
"""
from __future__ import annotations

import fnmatch
import os
import zipfile

import numpy as np
import pandas as pd

from .data_io import SEP, parse_id_list, read_tsv, write_tsv
from .schema import SubmissionSchema
from .utils import get_logger


def _fmt(ids: list[str], sch: SubmissionSchema) -> str:
    if not ids:
        return "[]" if sch.list_format == "bracketed" else ""
    body = sch.list_separator.join(ids) if sch.list_format != "bracketed" else ", ".join(ids)
    return f"[{body}]" if sch.list_format == "bracketed" else body


def build_matching_results(s1: pd.DataFrame, v: pd.DataFrame, scored: pd.DataFrame, selected: np.ndarray,
                           sch: SubmissionSchema) -> pd.DataFrame:
    sel = scored.loc[selected, ["s1_idx", "v_idx", "p_final"]].sort_values(["s1_idx", "p_final"],
                                                                         ascending=[True, False])
    vid = v["id"].to_numpy()
    vsrc = v["source"].to_numpy()
    by_ent: dict[int, list[int]] = {}
    for a, b in zip(sel["s1_idx"].to_numpy(), sel["v_idx"].to_numpy()):
        by_ent.setdefault(int(a), []).append(int(b))
    order = s1.sort_values("row_order")
    rows = []
    for s1_idx, s1_id in zip(order["s1_idx"].to_numpy(), order["id"].to_numpy()):
        matched = by_ent.get(int(s1_idx), [])
        row = {sch.id_column: s1_id}
        if len(sch.match_columns) == 1:
            row[sch.match_columns[0]] = _fmt(list(dict.fromkeys(vid[j] for j in matched)), sch)
        else:
            for col in sch.match_columns:
                src = sch.column_sources[col]
                row[col] = _fmt(list(dict.fromkeys(vid[j] for j in matched if vsrc[j] == src)), sch)
        rows.append(row)
    return pd.DataFrame(rows, columns=[sch.id_column, *sch.match_columns])


def write_candidate_pairs(cand: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame, path: str,
                          sch: SubmissionSchema) -> None:
    cols = sch.candidate_columns
    out = pd.DataFrame({
        cols.get("source1_id", "source1_id"): s1["id"].to_numpy()[cand["s1_idx"].to_numpy()],
        cols.get("candidate_id", "candidate_id"): v["id"].to_numpy()[cand["v_idx"].to_numpy()],
        cols.get("candidate_source", "candidate_source"): v["source"].to_numpy()[cand["v_idx"].to_numpy()],
    })
    write_tsv(out, path)


def validate_submission(path: str, s1_ids: list[str], s2_ids: set, s3_ids: set, sch: SubmissionSchema) -> dict:
    """Format checks mirroring what an organiser validator would reject.

    (The official validation script mentioned in the transcript should still
    be run before uploading.)"""
    errors, warnings = [], []
    if not os.path.exists(path):
        return {"ok": False, "errors": [f"{path} does not exist"], "warnings": []}
    with open(path, "rb") as fh:
        head = fh.readline()
    if b"\t" not in head:
        errors.append("header has no TAB character - file is not tab-separated")
    df = read_tsv(path)
    expected = [sch.id_column, *sch.match_columns]
    if list(df.columns) != expected:
        errors.append(f"columns {list(df.columns)} != expected {expected}")
        return {"ok": False, "errors": errors, "warnings": warnings}
    ids = df[sch.id_column].astype(str).str.strip()
    dup = ids[ids.duplicated()]
    if len(dup):
        errors.append(f"{len(dup)} duplicated Source-1 ids, e.g. {dup.head(3).tolist()}")
    missing = set(s1_ids) - set(ids)
    extra = set(ids) - set(s1_ids)
    if missing:
        errors.append(f"{len(missing)} Source-1 entities missing (every entity needs a row), e.g. {sorted(missing)[:3]}")
    if extra:
        errors.append(f"{len(extra)} ids not in Source 1, e.g. {sorted(extra)[:3]}")
    all_vendor = s2_ids | s3_ids
    n_pred, unknown, empty_rows = 0, [], 0
    for col in sch.match_columns:
        allowed = all_vendor
        if sch.column_sources:
            allowed = s2_ids if sch.column_sources[col] == 2 else s3_ids
        for val in df[col]:
            lst = parse_id_list(val, sch.list_separator)
            n_pred += len(lst)
            unknown.extend(x for x in lst if x not in allowed)
    for _, row in df.iterrows():
        if all(not parse_id_list(row[c], sch.list_separator) for c in sch.match_columns):
            empty_rows += 1
    if unknown:
        errors.append(f"{len(unknown)} predicted ids not found in Source 2/3, e.g. {unknown[:3]}")
    if n_pred == 0:
        warnings.append("no matches predicted at all")
    return {"ok": not errors, "errors": errors, "warnings": warnings, "rows": int(len(df)),
            "predicted_ids": int(n_pred), "empty_rows(predicted_singletons)": int(empty_rows)}


DEFAULT_CODE_PATTERNS = ["README.md", "METHODOLOGY.md", "requirements*.txt", "config.yaml", "configs/*.yaml", "pytest.ini", "notebooks/*.ipynb",
                         "*.py", "src/*.py", "scripts/*.py", "tests/*.py"]


def package_submission(root: str, files: dict[str, str], archive: str, include_models_dir: str | None = None,
                       patterns: list[str] | None = None) -> dict:
    """Create the single archive: final matches + candidate pairs + complete
    reproducible pipeline + methodology document (transcript deliverables)."""
    log = get_logger()
    patterns = patterns or DEFAULT_CODE_PATTERNS
    added = []
    os.makedirs(os.path.dirname(os.path.abspath(archive)), exist_ok=True)
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for arcname, path in files.items():
            if not os.path.exists(path):
                raise FileNotFoundError(f"required submission file missing: {path}")
            zf.write(path, arcname)
            added.append(arcname)
        for dirpath, _, filenames in os.walk(root):
            rel_dir = os.path.relpath(dirpath, root)
            if rel_dir.startswith((".git", "data", "outputs", "models", "__pycache__", ".pytest_cache")) \
                    or "/__pycache__" in rel_dir:
                continue
            for fn in filenames:
                rel = os.path.normpath(os.path.join(rel_dir, fn))
                if any(fnmatch.fnmatch(rel, p) for p in patterns):
                    zf.write(os.path.join(dirpath, fn), os.path.join("pipeline", rel))
                    added.append(os.path.join("pipeline", rel))
        if include_models_dir and os.path.isdir(include_models_dir):
            for dirpath, _, filenames in os.walk(include_models_dir):
                for fn in filenames:
                    full = os.path.join(dirpath, fn)
                    arc = os.path.join("pipeline", os.path.relpath(full, root))
                    zf.write(full, arc)
                    added.append(arc)
    log.info("Wrote %s with %d files", archive, len(added))
    return {"archive": archive, "files": added}


In [ ]:
%%writefile src/error_analysis.py
"""Error analysis on labelled splits (calib / holdout).

Writes, for every erroneous pair, the Source-1 and candidate raw and
normalised name/address, the key similarity features, the LightGBM,
cross-encoder and final scores:

  false_positives.tsv            selected but not a true match (false merges)
  false_negatives.tsv            true matches not selected (incl. lost in blocking)
  singleton_errors.tsv           singletons that received >=1 match
  nonsingletons_predicted_empty.tsv
  top_confident_errors.tsv       highest-probability false positives
  low_confidence_true_matches.tsv  true matches with the lowest scores
  difficult_cases.tsv            pairs near the threshold or where models disagree
  error_summary.json
"""
from __future__ import annotations

import os

import numpy as np
import pandas as pd

from .data_io import write_tsv
from .utils import ensure_dir, save_json

KEY_FEATURES = ["name_jw", "name_soft_tfidf", "name_token_set", "name_char_cos", "addr_char_cos",
                "addr_soft_tfidf", "hn_eq", "road_jw", "city_eq", "state_eq", "pc_eq", "unit_eq",
                "suffix_eq", "suffix_conflict", "contradictions", "ctx_rank", "ctx_mutual_best", "n_methods"]
SCORE_COLS = ["p_lgb_raw", "p_lgb", "sent_to_ce", "p_ce_raw", "p_ce", "p_blend", "p_final", "selected"]


def _describe_pairs(df: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame) -> pd.DataFrame:
    a = s1.set_index("s1_idx").loc[df["s1_idx"].to_numpy()]
    b = v.set_index("v_idx").loc[df["v_idx"].to_numpy()]
    out = pd.DataFrame({
        "s1_id": a["id"].to_numpy(), "s1_name": a["name"].to_numpy(), "s1_address": a["address"].to_numpy(),
        "s1_name_core": a["name_core"].to_numpy(), "s1_suffix": a["name_suffix"].to_numpy(),
        "s1_addr_clean": a["addr_clean"].to_numpy(),
        "cand_id": b["id"].to_numpy(), "cand_source": b["source"].to_numpy(), "cand_name": b["name"].to_numpy(),
        "cand_address": b["address"].to_numpy(), "cand_name_core": b["name_core"].to_numpy(),
        "cand_suffix": b["name_suffix"].to_numpy(), "cand_addr_clean": b["addr_clean"].to_numpy(),
    })
    for c in [*SCORE_COLS, *KEY_FEATURES, "is_true", "in_candidates"]:
        if c in df:
            out[c] = df[c].to_numpy()
    return out


def run_error_analysis(scored: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame, true_pairs: pd.DataFrame,
                       n_true: np.ndarray, entity_idx: np.ndarray, threshold: float, out_dir: str,
                       top_n: int = 200) -> dict:
    ensure_dir(out_dir)
    ents = set(int(e) for e in entity_idx)
    df = scored[scored["s1_idx"].isin(ents)].copy()
    df["in_candidates"] = True
    sel = df["selected"].to_numpy(bool)
    tru = df["is_true"].to_numpy(bool)
    fp = df[sel & ~tru].sort_values("p_final", ascending=False)
    fn_cand = df[~sel & tru]
    tp_all = true_pairs[true_pairs["s1_idx"].isin(ents)]
    lost = tp_all.merge(df[["s1_idx", "v_idx"]], on=["s1_idx", "v_idx"], how="left", indicator=True)
    lost = lost.loc[lost["_merge"] == "left_only", ["s1_idx", "v_idx"]]
    lost = lost.assign(in_candidates=False, is_true=True, selected=False)
    fn = pd.concat([fn_cand, lost], ignore_index=True)
    fn["error_type"] = np.where(fn["in_candidates"].astype(bool), "missed_by_matcher", "lost_in_blocking")

    n_pred = df[sel].groupby("s1_idx").size()
    ent = pd.DataFrame({"s1_idx": list(ents)})
    ent["n_true"] = n_true[ent["s1_idx"].to_numpy()]
    ent["n_pred"] = ent["s1_idx"].map(n_pred).fillna(0).astype(int)
    singleton_err = ent[(ent["n_true"] == 0) & (ent["n_pred"] > 0)]
    nonsingle_empty = ent[(ent["n_true"] > 0) & (ent["n_pred"] == 0)]

    fp_desc = _describe_pairs(fp, s1, v) if len(fp) else pd.DataFrame()
    fn_desc = _describe_pairs(fn, s1, v) if len(fn) else pd.DataFrame()
    if len(fn_desc):
        fn_desc["error_type"] = fn["error_type"].to_numpy()
    write_tsv(fp_desc, os.path.join(out_dir, "false_positives.tsv"))
    write_tsv(fn_desc, os.path.join(out_dir, "false_negatives.tsv"))
    se = fp[fp["s1_idx"].isin(set(singleton_err["s1_idx"]))]
    write_tsv(_describe_pairs(se, s1, v) if len(se) else pd.DataFrame(), os.path.join(out_dir, "singleton_errors.tsv"))
    ne = df[df["s1_idx"].isin(set(nonsingle_empty["s1_idx"])) & tru]
    write_tsv(_describe_pairs(ne, s1, v) if len(ne) else pd.DataFrame(),
              os.path.join(out_dir, "nonsingletons_predicted_empty.tsv"))
    write_tsv(fp_desc.head(top_n), os.path.join(out_dir, "top_confident_errors.tsv"))
    low_true = df[tru].sort_values("p_final").head(top_n)
    write_tsv(_describe_pairs(low_true, s1, v) if len(low_true) else pd.DataFrame(),
              os.path.join(out_dir, "low_confidence_true_matches.tsv"))
    near = (df["p_final"] - threshold).abs() < 0.1
    disagree = (df["p_lgb"] - df["p_ce"]).abs() > 0.5 if "p_ce" in df else pd.Series(False, index=df.index)
    hard = df[near | disagree.fillna(False)].sort_values("p_final", ascending=False).head(top_n * 2)
    write_tsv(_describe_pairs(hard, s1, v) if len(hard) else pd.DataFrame(), os.path.join(out_dir, "difficult_cases.tsv"))
    summary = {
        "entities": int(len(ent)),
        "false_positive_pairs": int(len(fp)),
        "false_negative_pairs": int(len(fn)),
        "false_negatives_missed_by_matcher": int((fn["error_type"] == "missed_by_matcher").sum()),
        "false_negatives_lost_in_blocking": int((fn["error_type"] == "lost_in_blocking").sum()),
        "singletons_with_false_merges": int(len(singleton_err)),
        "nonsingletons_predicted_empty": int(len(nonsingle_empty)),
        "false_positives_by_source": fp["source"].value_counts().to_dict() if len(fp) else {},
        "difficult_cases": int(len(hard)),
    }
    save_json(summary, os.path.join(out_dir, "error_summary.json"))
    return summary


In [ ]:
%%writefile src/training.py
"""Training pipeline as RESUMABLE STAGES (python train.py --config config.yaml).

  prepare       ingest TSVs, validate schema, parse labels, normalise names,
                parse addresses (offline), resolve labels, entity-level split
  biencoder     [full] SupCon bi-encoder on fit entities (early stop on fit_val)
  blocking      hybrid blocking over ALL training Source-1 entities + report
  features      pairwise features streamed to a disk-backed matrix, labels,
                hard-negative tags, features of true pairs lost in blocking
  lightgbm      LightGBM on fit pairs (early stop on fit_val) + isotonic
                calibration on calib
  crossencoder  [full] Ditto cross-encoder on fit pairs, cascade scoring of
                calib / holdout, isotonic calibration on calib
  decide        blend selection, final calibration, singleton model, graph
                modes and Macro-F0.5 decision search on calib; unbiased holdout
                evaluation + error analysis; pipeline.json

Every stage reads its inputs from outputs/<run>/train/checkpoints/ and writes
its outputs there, then returns - so its memory is released. `run_training`
simply runs the stages in order; `--resume` skips completed stages, and a
notebook can restart its kernel between any two stages.
"""
from __future__ import annotations

import gc
import os
import platform
import time

import numpy as np
import pandas as pd
import yaml

from .blocking import Blocker, blocking_report
from .calibration import Calibrator, fit_calibrator
from .config import get, is_full, model_dir, output_dir, semantic_enabled
from .data_io import load_labels, load_sources
from .decision import (ENTITY_INPUT_COLUMNS, DecisionParams, DecisionTable, effective_min_improvement,
                       entity_features, search_decision, train_singleton_model)
from .error_analysis import KEY_FEATURES, run_error_analysis
from .features import (FeatureContext, FeatureMatrix, build_feature_matrix, compute_features, predict_in_blocks,
                       remove_feature_matrix)
from .graph_cleanup import graph_keep_mask, normalize_mode
from .inference import TrainedPipeline, decide
from .labels import attach_labels, build_label_index, missed_positive_pairs, tag_negative_types
from .metrics import pairwise_prf, summarize
from .preprocess import preprocess_dataset
from .scoring import Blender, cascade_mask
from .splits import make_splits, split_summary
from .train_lightgbm import train_lightgbm
from .utils import (ensure_dir, get_logger, load_json, load_pickle, mem_str, release_memory, save_json, save_pickle,
                    set_seed, setup_logging, timer)

FIT_SPLITS = ("fit", "fit_val")
STAGES = ["prepare", "biencoder", "blocking", "features", "lightgbm", "crossencoder", "decide"]
NEG_TYPE_COLUMNS = ["addr_char_cos", "hn_eq", "pc_eq", "name_jw", "name_char_cos"]


# ------------------------------------------------------------ bookkeeping
def train_dir(cfg: dict) -> str:
    return os.path.join(output_dir(cfg), "train")


def ckpt(cfg: dict, name: str) -> str:
    return os.path.join(train_dir(cfg), "checkpoints", name)


def _start(cfg: dict, stage: str):
    ensure_dir(model_dir(cfg))
    ensure_dir(os.path.join(train_dir(cfg), "checkpoints"))
    log = setup_logging(os.path.join(train_dir(cfg), "train.log"))
    set_seed(int(cfg.get("seed", 42)))
    log.info("=== TRAIN STAGE %-12s mode=%s run=%s  [%s]", stage, cfg.get("mode"), cfg.get("run_name"), mem_str())
    return log


def _report_path(cfg: dict) -> str:
    return os.path.join(train_dir(cfg), "training_report.json")


def load_report(cfg: dict) -> dict:
    path = _report_path(cfg)
    return load_json(path) if os.path.exists(path) else {}


def _update_report(cfg: dict, **sections) -> dict:
    rep = load_report(cfg)
    rep.update(sections)
    save_json(rep, _report_path(cfg))
    return rep


def _record_time(cfg: dict, stage: str, seconds: float, extra: dict | None = None):
    rep = load_report(cfg)
    t = rep.get("timings_sec", {})
    t[stage] = round(seconds, 1)
    t.update(extra or {})
    t["total"] = round(sum(val for k, val in t.items() if k in STAGES), 1)
    rep["timings_sec"] = t
    save_json(rep, _report_path(cfg))


def stage_done(cfg: dict, stage: str) -> bool:
    return os.path.exists(ckpt(cfg, f"{stage}.done"))


def _mark_done(cfg: dict, stage: str):
    with open(ckpt(cfg, f"{stage}.done"), "w") as fh:
        fh.write(time.strftime("%Y-%m-%d %H:%M:%S"))


def _invalidate_from(cfg: dict, stage: str):
    """Re-running a stage makes every later stage stale."""
    for st in STAGES[STAGES.index(stage):]:
        path = ckpt(cfg, f"{st}.done")
        if os.path.exists(path):
            os.remove(path)


def _require(cfg: dict, *stages: str):
    for st in stages:
        if not stage_done(cfg, st):
            raise RuntimeError(f"stage '{st}' has not been run for run_name={cfg.get('run_name')!r} "
                               f"(expected {ckpt(cfg, st + '.done')}). Run it first.")


def _load_prepare(cfg: dict) -> dict:
    return load_pickle(ckpt(cfg, "prepare.pkl"))


def _entity_metrics(scored: pd.DataFrame, selected: np.ndarray, entity_idx: np.ndarray, n_true_all: np.ndarray,
                    beta: float) -> dict:
    ents = pd.Index(entity_idx)
    pos = pd.Series(np.arange(len(ents)), index=ents)
    e = pos.reindex(scored["s1_idx"].to_numpy()).to_numpy()
    tp = np.bincount(e[selected & scored["is_true"].to_numpy(bool)], minlength=len(ents))
    npred = np.bincount(e[selected], minlength=len(ents))
    out = summarize(tp, npred, n_true_all[entity_idx], beta)
    out["pairwise_on_candidates"] = pairwise_prf(selected, scored["is_true"].to_numpy(bool), beta)
    return out


# ------------------------------------------------------------------ stages
def stage_prepare(cfg: dict) -> dict:
    log = _start(cfg, "prepare")
    t0 = time.time()
    save_json({"mode": cfg.get("mode"), "run_name": cfg.get("run_name"), "config": cfg.get("_config_path")},
              _report_path(cfg))
    with open(os.path.join(model_dir(cfg), "config_snapshot.yaml"), "w") as fh:
        yaml.safe_dump({k: val for k, val in cfg.items() if not k.startswith("_")}, fh, sort_keys=False)
    with timer("load training data"):
        sources, load_stats = load_sources(cfg, "train")
        labels_df, label_file_stats = load_labels(get(cfg, "data.train.labels"), cfg)
    with timer("normalise + parse addresses"):
        s1, v, prep_stats = preprocess_dataset(sources, cfg)
    del sources
    li = build_label_index(labels_df, s1, v, cfg)
    split = make_splits(li, v["source"], get(cfg, "splits.fractions", {"fit": .7, "calib": .15, "holdout": .15}),
                        float(get(cfg, "splits.inner_val_fraction", 0.15)), int(cfg.get("seed", 42)))
    save_pickle({"s1": s1, "v": v, "li": li, "split": split}, ckpt(cfg, "prepare.pkl"))
    splits = split_summary(split, li)
    log.info("Splits: %s", splits)
    _update_report(cfg, data={"load": load_stats, "label_file": label_file_stats, "preprocess": prep_stats},
                   labels=li.stats, splits=splits)
    _record_time(cfg, "prepare", time.time() - t0)
    return {"source1": len(s1), "vendors": len(v), "splits": splits}


def stage_biencoder(cfg: dict) -> dict:
    log = _start(cfg, "biencoder")
    if not semantic_enabled(cfg):
        log.info("semantic blocking disabled (mode=%s) - bi-encoder stage skipped", cfg.get("mode"))
        return {"skipped": True}
    _require(cfg, "prepare")
    from .train_biencoder import train_biencoder
    t0 = time.time()
    prep = _load_prepare(cfg)
    li, split = prep["li"], prep["split"]
    fit_mask = np.isin(split[li.true_pairs["s1_idx"].to_numpy()], FIT_SPLITS)
    with timer("train SupCon bi-encoder"):
        rep = train_biencoder(prep["s1"], prep["v"], li.true_pairs[fit_mask], split, cfg,
                              os.path.join(model_dir(cfg), "biencoder"), int(cfg.get("seed", 42)))
    _update_report(cfg, biencoder=rep)
    _record_time(cfg, "biencoder", time.time() - t0)
    return rep


def stage_blocking(cfg: dict) -> dict:
    log = _start(cfg, "blocking")
    _require(cfg, "prepare")
    t0 = time.time()
    prep = _load_prepare(cfg)
    s1, v, li, split = prep["s1"], prep["v"], prep["li"], prep["split"]
    retriever = None
    if semantic_enabled(cfg):
        from .train_biencoder import SemanticRetriever
        _require(cfg, "biencoder")
        retriever = SemanticRetriever(os.path.join(model_dir(cfg), "biencoder"), cfg)
    with timer("blocking"):
        cand, precap, bstats = Blocker(cfg).generate(s1, v, retriever)
    if retriever is not None:
        retriever.save_indices(os.path.join(model_dir(cfg), "biencoder_index_train"))
        del retriever
    brep = blocking_report(cand, precap, s1, v, li, split, float(get(cfg, "decision.beta", 0.5)))
    del precap
    brep["generation"] = bstats
    save_json(brep, os.path.join(train_dir(cfg), "blocking_report.json"))
    save_pickle(cand, ckpt(cfg, "candidates.pkl"))
    summary = {k: brep[k] for k in ("candidate_pairs", "reduction_ratio", "pair_completeness",
                                    "pair_completeness_s2", "pair_completeness_s3", "true_pairs_lost",
                                    "oracle_macro_f0.5_given_blocking", "pair_completeness_before_cap") if k in brep}
    log.info("Blocking: %d candidates, reduction ratio %.6f, pair completeness %.4f (before cap %.4f), "
             "%d true pairs lost, oracle Macro F0.5 ceiling %.4f", brep["candidate_pairs"], brep["reduction_ratio"],
             brep.get("pair_completeness", float("nan")), brep.get("pair_completeness_before_cap", float("nan")),
             brep.get("true_pairs_lost", 0), brep.get("oracle_macro_f0.5_given_blocking", float("nan")))
    _update_report(cfg, blocking=summary)
    _record_time(cfg, "blocking", time.time() - t0)
    return summary


def stage_features(cfg: dict) -> dict:
    log = _start(cfg, "features")
    _require(cfg, "prepare", "blocking")
    t0 = time.time()
    prep = _load_prepare(cfg)
    s1, v, li, split = prep["s1"], prep["v"], prep["li"], prep["split"]
    cand = load_pickle(ckpt(cfg, "candidates.pkl"))
    with timer("pairwise features (streamed to disk)"):
        ctx = FeatureContext(s1, v, cfg)
        if is_full(cfg):
            save_pickle({**ctx.addr_idf, **ctx.name_idf}, os.path.join(model_dir(cfg), "serializer_idf.pkl"))
        fm = build_feature_matrix(cand, s1, v, ctx, cfg, ckpt(cfg, "features.npy"))
    feat_cols = list(fm.columns)
    cand["label"] = attach_labels(cand, li, v).to_numpy()
    cand["is_true"] = cand["label"] != 0          # id is in the entity's label list
    cand["split"] = pd.Categorical(split[cand["s1_idx"].to_numpy()])
    neg = (cand["label"] == 0).to_numpy()
    # report statistic only -> a sample, so the whole matrix is not read back
    neg_rows = np.flatnonzero(neg)
    if len(neg_rows) > 500_000:
        neg_rows = np.sort(np.random.default_rng(0).choice(neg_rows, 500_000, replace=False))
    neg_types = tag_negative_types(fm.frame(neg_rows, NEG_TYPE_COLUMNS))
    summary = {
        "candidate_pairs": int(len(cand)), "positives": int((cand["label"] == 1).sum()),
        "negatives": int(neg.sum()), "uncertain_excluded": int((cand["label"] == -1).sum()),
        "positive_rate": float((cand["label"] == 1).mean()) if len(cand) else 0.0,
        "hard_negative_types(sample)": neg_types.value_counts().to_dict(),
    }
    log.info("Pairs: %s", summary)
    extra = None
    if get(cfg, "training.add_missed_positives", True):
        mp = missed_positive_pairs(cand, li)
        mp = mp[np.isin(split[mp["s1_idx"].to_numpy()], FIT_SPLITS)].copy()
        if len(mp):
            mp["source"] = v["source"].to_numpy()[mp["v_idx"].to_numpy()]
            mp_feats = compute_features(mp, s1, v, ctx, {**cfg, "features": {**cfg.get("features", {}),
                                                                          "context_features": False}})
            for c in feat_cols:
                if c not in mp_feats:
                    mp_feats[c] = np.float32(np.nan)   # context of an unretrieved pair is undefined
            extra = {"X": mp_feats[feat_cols].astype(np.float32), "split": split[mp["s1_idx"].to_numpy()]}
            log.info("Added %d true pairs missed by blocking as extra fit positives", len(mp))
    del ctx, fm
    save_pickle(cand, ckpt(cfg, "pairs.pkl"))
    save_pickle({"feature_columns": feat_cols, "missed_positives": extra}, ckpt(cfg, "features_meta.pkl"))
    _update_report(cfg, training_pairs=summary)
    _record_time(cfg, "features", time.time() - t0)
    return summary


def stage_lightgbm(cfg: dict) -> dict:
    log = _start(cfg, "lightgbm")
    _require(cfg, "features")
    t0 = time.time()
    seed = int(cfg.get("seed", 42))
    mdir = model_dir(cfg)
    pairs = load_pickle(ckpt(cfg, "pairs.pkl"))
    meta = load_pickle(ckpt(cfg, "features_meta.pkl"))
    feat_cols = meta["feature_columns"]
    fm = FeatureMatrix(ckpt(cfg, "features.npy"), feat_cols)
    label = pairs["label"].to_numpy()
    sp = pairs["split"].astype(str).to_numpy()
    rng = np.random.default_rng(seed)
    ds = float(get(cfg, "training.neg_downsample", 1.0))
    usable = label >= 0
    if ds < 1.0:
        usable &= (label == 1) | (rng.random(len(pairs)) < ds)
    tr = np.flatnonzero(usable & (sp == "fit"))
    va = np.flatnonzero(usable & (sp == "fit_val"))
    extra = meta.get("missed_positives")
    m_fit = (extra["split"] == "fit") if extra is not None else np.zeros(0, dtype=bool)
    n_extra = int(m_fit.sum())
    # one pre-allocated training matrix (no concat copy of the large block)
    X = np.empty((len(tr) + n_extra, len(feat_cols)), dtype=np.float32)
    fm.fill(tr, feat_cols, X[:len(tr)])
    if n_extra:
        X[len(tr):] = extra["X"][m_fit][feat_cols].to_numpy(np.float32)
    X_tr = pd.DataFrame(X, columns=feat_cols, copy=False)
    y_tr = np.r_[label[tr].astype(int), np.ones(n_extra, dtype=int)]
    X_va, y_va = fm.frame(va, feat_cols), label[va].astype(int)
    log.info("LightGBM training matrix %s (neg_downsample=%.2f)  [%s]", X_tr.shape, ds, mem_str())
    with timer("train LightGBM"):
        booster, lgb_report = train_lightgbm(X_tr, y_tr, X_va, y_va, cfg, mdir, seed)
    del X_tr, X_va, X
    release_memory()
    with timer("LightGBM scoring of all candidate pairs (blocks)"):
        p_raw = predict_in_blocks(booster, fm, feat_cols, int(get(cfg, "features.predict_block", 500_000)))
    calib_m = (sp == "calib") & (label >= 0)
    cal_lgb, cal_rep = fit_calibrator(p_raw[calib_m], label[calib_m], get(cfg, "calibration.method", "isotonic"), seed)
    save_pickle(cal_lgb, os.path.join(mdir, "calibrator_lgb.pkl"))
    scores = pd.DataFrame({"p_lgb_raw": p_raw, "p_lgb": cal_lgb.transform(p_raw)}, index=pairs.index)
    save_pickle(scores, ckpt(cfg, "scores_lgb.pkl"))
    _update_report(cfg, lightgbm=lgb_report, calibration_lgb=cal_rep)
    _record_time(cfg, "lightgbm", time.time() - t0)
    return {"best_iteration": lgb_report.get("best_iteration"), "val": lgb_report.get("val")}


def stage_crossencoder(cfg: dict) -> dict:
    log = _start(cfg, "crossencoder")
    if not is_full(cfg):
        log.info("baseline mode - cross-encoder stage skipped")
        return {"skipped": True}
    _require(cfg, "prepare", "features", "lightgbm")
    from .train_crossencoder import CrossEncoderPredictor, DittoSerializer, train_crossencoder
    t0 = time.time()
    seed = int(cfg.get("seed", 42))
    mdir = model_dir(cfg)
    prep = _load_prepare(cfg)
    s1, v, li, split = prep["s1"], prep["v"], prep["li"], prep["split"]
    pairs = load_pickle(ckpt(cfg, "pairs.pkl"))
    cand = pd.concat([pairs[["s1_idx", "v_idx", "source", "label", "split"]],
                      load_pickle(ckpt(cfg, "scores_lgb.pkl"))], axis=1)
    del pairs
    cand["split"] = cand["split"].astype(str)
    ser = DittoSerializer(cfg, load_pickle(os.path.join(mdir, "serializer_idf.pkl")))
    t1, tv = ser.serialize_frame(s1), ser.serialize_frame(v)
    ccfg = cfg.get("crossencoder", {})
    tr_df = _crossencoder_training_pairs(cand, ccfg, seed,
                                         li if get(cfg, "training.add_missed_positives", True) else None, split)
    mask_all = cascade_mask(cand, cand["p_lgb"].to_numpy(), cfg.get("cascade", {}))
    va_df = cand[mask_all & (cand["split"] == "fit_val").to_numpy() & (cand["label"].to_numpy() >= 0)]
    with timer("train cross-encoder"):
        ce_rep = train_crossencoder(
            [t1[i] for i in tr_df["s1_idx"]], [tv[j] for j in tr_df["v_idx"]], tr_df["label"].to_numpy(),
            [t1[i] for i in va_df["s1_idx"]], [tv[j] for j in va_df["v_idx"]], va_df["label"].to_numpy(),
            cfg, os.path.join(mdir, "crossencoder"), seed)
    predictor = CrossEncoderPredictor(os.path.join(mdir, "crossencoder"), cfg)
    eval_m = mask_all & cand["split"].isin(["calib", "holdout"]).to_numpy()
    out = pd.DataFrame({"sent_to_ce": eval_m, "p_ce_raw": np.nan, "p_ce": np.nan}, index=cand.index)
    with timer("cross-encoder scoring of calib/holdout cascade pairs"):
        p = predictor.predict([t1[i] for i in cand.loc[eval_m, "s1_idx"]], [tv[j] for j in cand.loc[eval_m, "v_idx"]])
    out.loc[eval_m, "p_ce_raw"] = p
    calib_m = (cand["split"] == "calib").to_numpy() & (cand["label"].to_numpy() >= 0)
    cm = eval_m & calib_m
    cal_ce, cal_ce_rep = fit_calibrator(out.loc[cm, "p_ce_raw"], cand.loc[cm, "label"],
                                        get(cfg, "calibration.method", "isotonic"), seed)
    out.loc[eval_m, "p_ce"] = cal_ce.transform(out.loc[eval_m, "p_ce_raw"].to_numpy())
    save_pickle(cal_ce, os.path.join(mdir, "calibrator_ce.pkl"))
    save_pickle(out, ckpt(cfg, "scores_ce.pkl"))
    n_eval = max(1, int(cand["split"].isin(["calib", "holdout"]).sum()))
    cascade = {"pairs_sent_calib_holdout": int(eval_m.sum()), "share_of_calib_holdout_pairs": float(eval_m.sum() / n_eval)}
    _update_report(cfg, crossencoder=ce_rep, calibration_ce=cal_ce_rep, cascade=cascade)
    _record_time(cfg, "crossencoder", time.time() - t0)
    return cascade


def stage_decide(cfg: dict) -> dict:
    log = _start(cfg, "decide")
    _require(cfg, "prepare", "features", "lightgbm")
    t0 = time.time()
    seed = int(cfg.get("seed", 42))
    beta = float(get(cfg, "decision.beta", 0.5))
    mdir, odir = model_dir(cfg), train_dir(cfg)
    prep = _load_prepare(cfg)
    s1, v, li, split = prep["s1"], prep["v"], prep["li"], prep["split"]
    pairs = load_pickle(ckpt(cfg, "pairs.pkl"))
    feat_cols = load_pickle(ckpt(cfg, "features_meta.pkl"))["feature_columns"]
    fm = FeatureMatrix(ckpt(cfg, "features.npy"), feat_cols)
    has_ce = is_full(cfg)
    if has_ce:
        _require(cfg, "crossencoder")
        ce = load_pickle(ckpt(cfg, "scores_ce.pkl"))
    else:
        ce = pd.DataFrame({"sent_to_ce": False, "p_ce_raw": np.nan, "p_ce": np.nan}, index=pairs.index)
    cand = pd.concat([pairs[["s1_idx", "v_idx", "source", "label", "is_true", "split", "n_methods"]],
                      load_pickle(ckpt(cfg, "scores_lgb.pkl")), ce], axis=1)
    del pairs, ce
    cand["split"] = cand["split"].astype(str)
    n_true = li.n_true
    dcfg = cfg.get("decision", {})
    eval_cols = [c for c in dict.fromkeys([*KEY_FEATURES, *ENTITY_INPUT_COLUMNS]) if c in fm.col_idx]

    # ------------------------------------------- blend + decision on calib
    calib_ents = np.where((split == "calib") & li.labeled)[0]
    calib_rows = np.flatnonzero((cand["split"] == "calib").to_numpy())
    cdf = cand.iloc[calib_rows].copy()
    cfeats = fm.frame(calib_rows, [c for c in ENTITY_INPUT_COLUMNS if c in fm.col_idx], index=cdf.index)
    blender, cal_final, blend_trace = _select_blend(cdf, calib_ents, n_true, cfg, has_ce, seed, beta)
    cdf["p_blend"] = blender.transform(cdf["p_lgb"], cdf["p_ce"], cdf["p_ce"].notna())
    cdf["p_final"] = cal_final.transform(cdf["p_blend"].to_numpy())

    gcfg = cfg.get("graph", {})
    graph_modes = [normalize_mode(m) for m in gcfg.get("modes_to_try", ["off"])]
    masks, gstats = {}, {}
    for mode in graph_modes:
        masks[mode], gstats[mode] = graph_keep_mask(cdf, "p_final", mode, float(gcfg.get("min_edge_prob", 0.2)),
                                                    int(gcfg.get("max_component_size", 50)))

    matchable, singleton_model, sm_rep = None, None, None
    if get(cfg, "singleton_model.enabled", True):
        ef = entity_features(pd.concat([cdf, cfeats], axis=1), calib_ents)
        y_match = np.zeros(len(calib_ents), dtype=int)
        y_match[np.isin(calib_ents, cdf.loc[cdf["is_true"], "s1_idx"].unique())] = 1
        singleton_model, oof, sm_rep = train_singleton_model(ef, y_match, int(get(cfg, "singleton_model.folds", 5)), seed)
        if singleton_model is not None:
            matchable = oof
            singleton_model.save_model(os.path.join(mdir, "singleton_model.txt"))

    table = DecisionTable.build(cdf, "p_final", calib_ents, n_true[calib_ents], masks, matchable, beta)
    if dcfg.get("search", True):
        with timer("Macro-F0.5 decision search (calib)"):
            params, calib_score, trace = search_decision(table, dcfg, graph_modes)
        pd.DataFrame(trace, columns=["round", "param", "value", "macro_f0.5"]).to_csv(
            os.path.join(odir, "decision_search_trace.tsv"), sep="\t", index=False)
    else:
        params = DecisionParams.from_dict(dcfg.get("default", {}))
        calib_score = table.score(params)
    del table, cdf, cfeats

    # save pipeline so that holdout is scored through the exact inference path
    save_pickle(blender, os.path.join(mdir, "blender.pkl"))
    save_pickle(cal_final, os.path.join(mdir, "calibrator_final.pkl"))
    cal_lgb = load_pickle(os.path.join(mdir, "calibrator_lgb.pkl"))
    cal_ce = load_pickle(os.path.join(mdir, "calibrator_ce.pkl")) if has_ce else None
    meta = {
        "mode": cfg.get("mode"), "feature_columns": feat_cols, "decision": params.to_dict(),
        "has_ce": has_ce, "has_singleton_model": singleton_model is not None,
        "has_semantic_blocking": semantic_enabled(cfg), "graph": gcfg, "cascade": cfg.get("cascade", {}),
        "blend": blender.describe(), "calibration": {"lgb": cal_lgb.method, "ce": cal_ce.method if cal_ce else None,
                                                     "final": cal_final.method},
        "label_key_mode": li.key_mode, "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "python": platform.python_version(),
    }
    save_json(meta, os.path.join(mdir, "pipeline.json"))
    pipe = TrainedPipeline(mdir, cfg)

    # ------------------------------------------------ holdout evaluation
    evaluation, scored_all = {}, []
    for name in ("calib", "holdout"):
        ents = np.where((split == name) & li.labeled)[0]
        rows = np.flatnonzero((cand["split"] == name).to_numpy())
        if len(ents) == 0:
            continue
        sc = cand.iloc[rows][["s1_idx", "v_idx", "source", "label", "is_true", "p_lgb_raw", "p_lgb", "sent_to_ce",
                              "p_ce_raw", "p_ce"]].copy()
        sc["p_blend"] = pipe.blender.transform(sc["p_lgb"], sc["p_ce"], sc["p_ce"].notna())
        sc["p_final"] = pipe.cal_final.transform(sc["p_blend"].to_numpy())
        extra = fm.frame(rows, eval_cols, index=sc.index)
        sel, info = decide(sc, ents, pipe, extra_cols=extra)
        sc["selected"] = sel
        m = _entity_metrics(sc, sel, ents, n_true, beta)
        m["graph"] = info["graph"]
        evaluation[name] = m
        sc = pd.concat([sc, extra, cand.iloc[rows][["n_methods", "split"]]], axis=1)
        scored_all.append(sc)
        if name == "holdout":
            with timer("error analysis (holdout)"):
                evaluation["holdout_errors"] = run_error_analysis(
                    sc, s1, v, li.true_pairs, n_true, ents, min(params.threshold_s2, params.threshold_s3),
                    os.path.join(odir, "error_analysis_holdout"), int(get(cfg, "evaluation.error_analysis_top_n", 200)))
    evaluation["ablation"] = _ablation(cand, split, li, n_true, dcfg, beta, has_ce)
    if "holdout" in evaluation:
        h = evaluation["holdout"]
        log.info("HOLDOUT  Macro F0.5 = %.5f | pair P = %.4f R = %.4f | singletons: %d true, %d false merges | "
                 "predicted singletons %d", h["macro_f0.5"], h["pair_precision"], h["pair_recall"],
                 h["true_singletons"], h["singleton_false_merges"], h["predicted_singletons"])
    save_pickle({"scored": pd.concat(scored_all) if scored_all else pd.DataFrame(), "split": split,
                 "n_true": n_true, "true_pairs": li.true_pairs, "true_keys": li.true_keys,
                 "labeled": li.labeled, "key_mode": li.key_mode, "s1": s1, "v": v},
                os.path.join(odir, "eval_state.pkl"))
    _update_report(cfg, blend={"chosen": blender.describe(), "calibration": cal_final.method, "trace": blend_trace},
                   graph_calib=gstats, singleton_model=sm_rep,
                   decision={"params": params.to_dict(), "calib_macro_f0.5(optimistic, tuned here)": calib_score},
                   evaluation=evaluation)
    _record_time(cfg, "decide", time.time() - t0)
    log.info("Artefacts: %s | reports: %s  [%s]", mdir, odir, mem_str())
    return evaluation.get("holdout", {})


STAGE_FUNCS = {"prepare": stage_prepare, "biencoder": stage_biencoder, "blocking": stage_blocking,
               "features": stage_features, "lightgbm": stage_lightgbm, "crossencoder": stage_crossencoder,
               "decide": stage_decide}


def run_stage(cfg: dict, stage: str) -> dict:
    """Run ONE stage (what each notebook cell calls)."""
    if stage not in STAGE_FUNCS:
        raise ValueError(f"unknown stage {stage!r}; stages are {STAGES}")
    noop = (stage == "biencoder" and not semantic_enabled(cfg)) or (stage == "crossencoder" and not is_full(cfg))
    if not noop:   # a stage that does nothing in this mode must not invalidate later ones
        _invalidate_from(cfg, stage)
    out = STAGE_FUNCS[stage](cfg)
    _mark_done(cfg, stage)
    release_memory()
    get_logger().info("stage %s finished  [%s]", stage, mem_str())
    return out


def run_training(cfg: dict, stages: list[str] | None = None, resume: bool = False) -> dict:
    """Run the stages in order. resume=True skips stages already completed."""
    ensure_dir(os.path.join(train_dir(cfg), "checkpoints"))
    setup_logging(os.path.join(train_dir(cfg), "train.log"))
    for stage in stages or STAGES:
        if resume and stage_done(cfg, stage):
            get_logger().info("stage %s already done - skipped (resume)", stage)
            continue
        run_stage(cfg, stage)
    if get(cfg, "features.keep_matrix", True) is False and stage_done(cfg, "decide"):
        remove_feature_matrix(ckpt(cfg, "features.npy"))
    return load_report(cfg)


def _crossencoder_training_pairs(cand: pd.DataFrame, ccfg: dict, seed: int, li, split) -> pd.DataFrame:
    """Positives + LightGBM-hard negatives + random negatives of fit entities."""
    fit = cand[(cand["split"] == "fit") & (cand["label"] >= 0)]
    pos = fit[fit["label"] == 1][["s1_idx", "v_idx", "label"]]
    neg = fit[fit["label"] == 0].sort_values(["s1_idx", "p_lgb_raw"], ascending=[True, False])
    hard = neg.groupby("s1_idx").head(int(ccfg.get("hard_negatives_per_entity", 8)))
    rest = neg.drop(hard.index)
    rnd = rest.sample(frac=1.0, random_state=seed).groupby("s1_idx").head(int(ccfg.get("random_negatives_per_entity", 2)))
    parts = [pos, hard[["s1_idx", "v_idx", "label"]], rnd[["s1_idx", "v_idx", "label"]]]
    if li is not None:
        mp = missed_positive_pairs(cand, li)
        mp = mp[split[mp["s1_idx"].to_numpy()] == "fit"]
        parts.append(mp.assign(label=1))
    out = pd.concat(parts, ignore_index=True).drop_duplicates(["s1_idx", "v_idx"])
    cap = int(ccfg.get("max_train_pairs", 400000))
    if len(out) > cap:
        pos_part = out[out["label"] == 1]
        neg_part = out[out["label"] == 0].sample(n=max(0, cap - len(pos_part)), random_state=seed)
        out = pd.concat([pos_part, neg_part])
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def _select_blend(cdf, calib_ents, n_true, cfg, has_ce, seed, beta):
    """Choose blend method/weight + final calibration by calib Macro F0.5."""
    log = get_logger()
    bcfg = cfg.get("blend", {})
    cal_method = get(cfg, "calibration.method", "isotonic")
    dcfg = cfg.get("decision", {})
    y = cdf["label"].clip(lower=0).to_numpy()
    has = cdf["p_ce"].notna().to_numpy()
    if not has_ce or not has.any():
        return Blender("weighted", 1.0), Calibrator("none"), []
    # A blend fitted on a handful of cascaded pairs is noise-fitting: require
    # enough cross-encoder-scored calib pairs of BOTH classes, else keep LightGBM.
    n_pos, n_neg = int((y[has] == 1).sum()), int((y[has] == 0).sum())
    min_pairs, min_cls = int(bcfg.get("min_ce_pairs", 200)), int(bcfg.get("min_ce_per_class", 20))
    if has.sum() < min_pairs or min(n_pos, n_neg) < min_cls:
        log.warning("Only %d cascaded calib pairs (%d pos / %d neg) < blend.min_ce_pairs=%d or "
                    "min_ce_per_class=%d -> keeping LightGBM alone", int(has.sum()), n_pos, n_neg, min_pairs, min_cls)
        return Blender("weighted", 1.0), Calibrator("none"), [{"skipped": "too few cross-encoder calib pairs",
                                                               "pairs": int(has.sum()), "pos": n_pos, "neg": n_neg}]
    methods = ["weighted", "rank", "stack"] if bcfg.get("method", "auto") == "auto" else [bcfg["method"]]
    min_imp = effective_min_improvement(dcfg, len(calib_ents))
    trace, best = [], None
    # LightGBM alone is the reference; a blend must beat it by min_improvement
    candidates = [("weighted", 1.0)] + [(m, float(w)) for m in methods
                                        for w in (bcfg.get("weight_grid", [0.5]) if m != "stack" else [1.0])
                                        if not (m == "weighted" and float(w) == 1.0)]
    for method, w in candidates:
        bl = Blender(method, w).fit(cdf["p_lgb"], cdf["p_ce"], has, y)
        pb = bl.transform(cdf["p_lgb"], cdf["p_ce"], has)
        lgb_only = method == "weighted" and w == 1.0     # p_lgb is already calibrated
        if get(cfg, "calibration.calibrate_blend", True) and not lgb_only:
            cal = Calibrator(cal_method if cal_method != "auto" else "isotonic").fit(pb, y)
        else:
            cal = Calibrator("none")
        tmp = cdf.assign(p_final=cal.transform(pb))
        table = DecisionTable.build(tmp, "p_final", calib_ents, n_true[calib_ents], beta=beta)
        _, score, _ = search_decision(table, dcfg, quick=True)
        trace.append({"method": method, "weight_lgb": w, "calib_macro_f0.5_quick": score})
        if best is None or score > best[0] + min_imp:
            best = (score, bl, cal)
    log.info("Blend selection: %s", best[1].describe())
    return best[1], best[2], trace


def _ablation(cand, split, li, n_true, dcfg, beta, has_ce) -> dict:
    """Holdout Macro F0.5 of simpler scorers, each thresholded on calib."""
    out = {}
    cols = ["p_lgb"] + (["p_ce_or_lgb"] if has_ce else [])
    ce = np.where((split == "calib") & li.labeled)[0]
    he = np.where((split == "holdout") & li.labeled)[0]
    if len(ce) == 0 or len(he) == 0:
        return out
    keep = ["s1_idx", "v_idx", "source", "label", "is_true", "p_lgb", "p_ce"]
    c = cand.loc[(cand["split"] == "calib").to_numpy(), keep].copy()
    h = cand.loc[(cand["split"] == "holdout").to_numpy(), keep].copy()
    for df in (c, h):
        df["p_ce_or_lgb"] = df["p_ce"].fillna(df["p_lgb"])
    for col in cols:
        tc = DecisionTable.build(c.assign(p=c[col]), "p", ce, n_true[ce], beta=beta)
        prm, s_c, _ = search_decision(tc, dcfg, quick=True)
        th = DecisionTable.build(h.assign(p=h[col]), "p", he, n_true[he], beta=beta)
        out[col] = {"threshold": prm.threshold_s2, "calib_macro_f0.5": s_c, "holdout_macro_f0.5": th.score(prm)}
    return out


In [ ]:
%%writefile src/prediction.py
"""Test-time pipeline as RESUMABLE STAGES (python predict.py --config config.yaml).

  prepare   load test TSVs -> normalise names / parse addresses (offline)
  blocking  [full] encode + FAISS index -> hybrid blocking
            -> WRITE candidate_pairs.tsv (before any model scoring)
  features  pairwise features streamed to a disk-backed matrix
  score     LightGBM -> calibration -> cascade -> cross-encoder -> blend
            -> final calibration -> graph cleanup -> decision rules
            -> WRITE matching_results.tsv (+ scored_candidates.tsv) -> validate

Checkpoints live in outputs/<run>/<split>/checkpoints/, so a notebook can
restart its kernel between stages. No labels exist for the test set and none
are used; every threshold comes from models/<run_name>/pipeline.json, fitted
on the training calib split.
"""
from __future__ import annotations

import os
import time

import numpy as np
import pandas as pd

from .blocking import METHOD_BIT, METHODS, Blocker, blocking_report
from .config import get, model_dir, output_dir
from .data_io import detect_list_format, load_sources, read_sample_submission_header, read_tsv, write_tsv
from .decision import ENTITY_INPUT_COLUMNS
from .features import FeatureContext, FeatureMatrix, build_feature_matrix, remove_feature_matrix
from .inference import TrainedPipeline, decide, score_candidates
from .schema import label_schema, submission_schema
from .submission import build_matching_results, validate_submission, write_candidate_pairs
from .utils import (ensure_dir, load_json, load_pickle, mem_str, release_memory, save_json, save_pickle, set_seed,
                    setup_logging, timer)

PREDICT_STAGES = ["prepare", "blocking", "features", "score"]


def _list_format_from_labels(cfg: dict) -> str:
    path = get(cfg, "data.train.labels")
    if path and os.path.exists(path):
        df = read_tsv(path, cfg)
        cols = [c for c in label_schema(cfg).match_columns if c in df]
        if cols:
            return detect_list_format(df[cols[0]])
    return "plain"


def _odir(cfg: dict, split: str) -> str:
    return os.path.join(output_dir(cfg), split)


def _ck(cfg: dict, split: str, name: str) -> str:
    return os.path.join(_odir(cfg, split), "checkpoints", name)


def _start(cfg: dict, split: str, stage: str):
    ensure_dir(os.path.join(_odir(cfg, split), "checkpoints"))
    log = setup_logging(os.path.join(_odir(cfg, split), "predict.log"))
    set_seed(int(cfg.get("seed", 42)))
    log.info("=== PREDICT STAGE %-9s split=%s run=%s  [%s]", stage, split, cfg.get("run_name"), mem_str())
    return log


def _schema(cfg: dict):
    return submission_schema(cfg, label_schema(cfg), _list_format_from_labels(cfg), read_sample_submission_header(cfg))


def _timing(cfg: dict, split: str, stage: str, seconds: float):
    path = _ck(cfg, split, "timings.json")
    t = load_json(path) if os.path.exists(path) else {}
    t[stage] = round(seconds, 1)
    save_json(t, path)


def predict_prepare(cfg: dict, split: str = "test") -> dict:
    _start(cfg, split, "prepare")
    t0 = time.time()
    with timer("load data"):
        sources, load_stats = load_sources(cfg, split)
    from .preprocess import preprocess_dataset
    with timer("normalise + parse addresses"):
        s1, v, prep_stats = preprocess_dataset(sources, cfg)
    del sources
    save_pickle({"s1": s1, "v": v, "load": load_stats, "preprocess": prep_stats}, _ck(cfg, split, "prepare.pkl"))
    _timing(cfg, split, "prepare", time.time() - t0)
    return {"source1": len(s1), "vendors": len(v)}


def predict_blocking(cfg: dict, split: str = "test") -> dict:
    log = _start(cfg, split, "blocking")
    t0 = time.time()
    prep = load_pickle(_ck(cfg, split, "prepare.pkl"))
    s1, v = prep["s1"], prep["v"]
    pipe_meta = load_json(os.path.join(model_dir(cfg), "pipeline.json"))
    retriever = None
    if pipe_meta.get("has_semantic_blocking"):
        from .train_biencoder import SemanticRetriever
        retriever = SemanticRetriever(os.path.join(model_dir(cfg), "biencoder"), cfg)
    with timer("blocking"):
        cand, precap, bstats = Blocker(cfg).generate(s1, v, retriever)
    if retriever is not None:
        retriever.save_indices(os.path.join(_odir(cfg, split), "semantic_index"))
        del retriever
    cand_path = os.path.join(_odir(cfg, split), get(cfg, "submission.candidate_pairs", "candidate_pairs.tsv"))
    write_candidate_pairs(cand, s1, v, cand_path, _schema(cfg))   # BEFORE the matcher runs
    brep = blocking_report(cand, precap, s1, v)
    del precap
    brep["generation"] = bstats
    save_json(brep, os.path.join(_odir(cfg, split), "blocking_report.json"))
    save_pickle(cand, _ck(cfg, split, "candidates.pkl"))
    log.info("Wrote %s (%d candidate pairs, reduction ratio %.6f)", cand_path, len(cand), brep["reduction_ratio"])
    _timing(cfg, split, "blocking", time.time() - t0)
    return {"candidate_pairs": len(cand), "reduction_ratio": brep["reduction_ratio"], "file": cand_path}


def predict_features(cfg: dict, split: str = "test") -> dict:
    _start(cfg, split, "features")
    t0 = time.time()
    prep = load_pickle(_ck(cfg, split, "prepare.pkl"))
    cand = load_pickle(_ck(cfg, split, "candidates.pkl"))
    with timer("pairwise features (streamed to disk)"):
        ctx = FeatureContext(prep["s1"], prep["v"], cfg)
        fm = build_feature_matrix(cand, prep["s1"], prep["v"], ctx, cfg, _ck(cfg, split, "features.npy"))
    _timing(cfg, split, "features", time.time() - t0)
    return {"pairs": len(fm), "features": len(fm.columns)}


def _write_scored(scored: pd.DataFrame, cand: pd.DataFrame, s1: pd.DataFrame, v: pd.DataFrame, info: dict,
                  entity_idx: np.ndarray, path: str, block: int = 1_000_000):
    """Debug file written in blocks (no full-size copy of the scored frame)."""
    names = [m for m in METHODS if f"blk_{m}" in cand]
    flags = cand["blk_flags"].to_numpy() if "blk_flags" in cand else None
    matchable = None
    if info.get("matchable") is not None:
        pos = pd.Series(np.arange(len(entity_idx)), index=entity_idx)
        matchable = info["matchable"][pos.reindex(scored["s1_idx"]).to_numpy()]
    s1_ids, v_ids = s1["id"].to_numpy(), v["id"].to_numpy()
    ensure_dir(os.path.dirname(path))
    for i, a in enumerate(range(0, max(len(scored), 1), block)):
        b = min(len(scored), a + block)
        part = scored.iloc[a:b].copy()
        part.insert(0, "source1_id", s1_ids[part["s1_idx"].to_numpy()])
        part.insert(1, "candidate_id", v_ids[part["v_idx"].to_numpy()])
        if flags is not None:
            f = flags[a:b]
            part["blocking_methods"] = [",".join(m for m in names if x & METHOD_BIT[m]) for x in f]
        part["graph_keep"] = info["graph_keep"][a:b]
        if matchable is not None:
            part["entity_p_matchable"] = matchable[a:b]
        part.to_csv(path, sep="\t", index=False, mode="w" if i == 0 else "a", header=i == 0, lineterminator="\n")


def predict_score(cfg: dict, split: str = "test") -> dict:
    log = _start(cfg, split, "score")
    t0 = time.time()
    odir = _odir(cfg, split)
    pipe = TrainedPipeline(model_dir(cfg), cfg)
    log.info("decision parameters: %s", pipe.decision.to_dict())
    prep = load_pickle(_ck(cfg, split, "prepare.pkl"))
    s1, v = prep["s1"], prep["v"]
    cand = load_pickle(_ck(cfg, split, "candidates.pkl"))
    fm = FeatureMatrix(_ck(cfg, split, "features.npy"))
    with timer("scoring (LightGBM / cascade / cross-encoder / blend)"):
        scored = score_candidates(cand, fm, s1, v, pipe)
    entity_idx = s1["s1_idx"].to_numpy()
    with timer("graph cleanup + decision"):
        extra = fm.frame(None, [c for c in ENTITY_INPUT_COLUMNS if c in fm.col_idx], index=scored.index)
        selected, info = decide(scored, entity_idx, pipe, extra_cols=extra)
        del extra
    scored["selected"] = selected
    sch = _schema(cfg)
    results = build_matching_results(s1, v, scored, selected, sch)
    res_path = os.path.join(odir, get(cfg, "submission.matching_results", "matching_results.tsv"))
    write_tsv(results, res_path)
    if get(cfg, "submission.write_scored_candidates", True):
        with timer("write scored_candidates.tsv"):
            _write_scored(scored, cand, s1, v, info, entity_idx,
                          os.path.join(odir, get(cfg, "submission.scored_candidates", "scored_candidates.tsv")))
    val = validate_submission(res_path, s1["id"].tolist(), set(v.loc[v["source"] == 2, "id"]),
                              set(v.loc[v["source"] == 3, "id"]), sch)
    n_match = int(selected.sum())
    per_ent = pd.Series(selected).groupby(scored["s1_idx"].to_numpy()).sum().reindex(entity_idx, fill_value=0)
    brep = load_json(os.path.join(odir, "blocking_report.json"))
    cand_path = os.path.join(odir, get(cfg, "submission.candidate_pairs", "candidate_pairs.tsv"))
    _timing(cfg, split, "score", time.time() - t0)
    report = {
        "split": split, "records": {"source1": len(s1), "source2": int((v["source"] == 2).sum()),
                                    "source3": int((v["source"] == 3).sum())},
        "load": prep["load"], "preprocess": prep["preprocess"],
        "blocking": {k: brep[k] for k in ("candidate_pairs", "reduction_ratio", "candidates_per_entity",
                                          "entities_without_candidates") if k in brep},
        "cascade_pairs_sent_to_ce": int(scored["sent_to_ce"].sum()),
        "graph": info["graph"], "decision": pipe.decision.to_dict(),
        "predicted_matches": n_match,
        "predicted_matches_s2": int((selected & (scored["source"] == 2).to_numpy()).sum()),
        "predicted_matches_s3": int((selected & (scored["source"] == 3).to_numpy()).sum()),
        "predicted_singletons": int((per_ent == 0).sum()),
        "predicted_singleton_rate": float((per_ent == 0).mean()) if len(per_ent) else 0.0,
        "validation": val, "outputs": {"matching_results": res_path, "candidate_pairs": cand_path},
        "timings_sec": load_json(_ck(cfg, split, "timings.json")),
    }
    save_json(report, os.path.join(odir, "prediction_report.json"))
    log.info("Predicted %d matches; %d of %d Source-1 entities predicted as singletons (%.1f%%)", n_match,
             report["predicted_singletons"], len(entity_idx), 100 * report["predicted_singleton_rate"])
    log.info("Validation: %s", "OK" if val["ok"] else val["errors"])
    log.info("Wrote %s  [%s]", res_path, mem_str())
    return report


PREDICT_FUNCS = {"prepare": predict_prepare, "blocking": predict_blocking, "features": predict_features,
                 "score": predict_score}


def run_prediction(cfg: dict, split: str = "test", stages: list[str] | None = None) -> dict:
    out = {}
    for stage in stages or PREDICT_STAGES:
        if stage not in PREDICT_FUNCS:
            raise ValueError(f"unknown prediction stage {stage!r}; stages are {PREDICT_STAGES}")
        out = PREDICT_FUNCS[stage](cfg, split)
        release_memory()
    if get(cfg, "features.keep_matrix", True) is False:
        remove_feature_matrix(_ck(cfg, split, "features.npy"))
    return out


### Configuration files and helper scripts

In [ ]:
%%writefile config.yaml
# =============================================================================
# Amazon ML Challenge 2026 - Business Entity Resolution
# Main configuration (single source of truth). Other configs can inherit from
# this file with `base: ../config.yaml` and override only what they change.
#
# All relative paths are resolved against the directory you run the commands
# from (run everything from the repository root).
#
# BEFORE THE FIRST REAL RUN: run `python inspect_data.py --config config.yaml`
# and set the `data:` paths and the `schema:` column names to the ACTUAL
# challenge files. The transcript does not specify file or column names, so
# the values below are placeholders.
# =============================================================================

run_name: default
# baseline: normalisation -> multi-pass blocking -> features -> LightGBM
#           -> isotonic calibration -> Macro-F0.5 decision search -> output
# full:     baseline + SupCon bi-encoder semantic blocking (FAISS)
#           + Ditto-style DeBERTa-v3 cross-encoder cascade + blending
mode: baseline
seed: 42
n_jobs: 4                     # CPU worker processes for feature computation

paths:
  models_dir: models          # artefacts -> models/<run_name>/
  outputs_dir: outputs        # outputs   -> outputs/<run_name>/

data:
  train:
    source1: data/train/source1.tsv
    source2: data/train/source2.tsv
    source3: data/train/source3.tsv
    labels: data/train/labels.tsv
  test:
    source1: data/test/source1.tsv
    source2: data/test/source2.tsv
    source3: data/test/source3.tsv
  # If the organisers ship a sample submission, point to it; its header then
  # defines the output columns of matching_results.tsv.
  sample_submission: null

io:
  encoding: utf-8-sig         # utf-8 that also strips a BOM if present
  quoting: minimal            # minimal (pandas default; "quoted" fields) | none (quotes literal)

schema:
  # Column names per source. `address` is a LIST: several columns
  # (street, city, state, zip, ...) are joined with `address_join`.
  source1: {id: id, name: name, address: [address]}
  source2: {id: id, name: name, address: [address]}
  source3: {id: id, name: name, address: [address]}
  address_join: ", "
  duplicate_ids: warn_keep_first   # error | warn_keep_first
  labels:
    source1_id: source1_id
    # One combined column holding S2 and S3 ids ("comma-separated list of all
    # its matching IDs"), or one column per source.
    match_columns: [matches]
    # Optional explicit mapping {column: 2|3}. null -> ids are resolved to a
    # source by looking them up in the Source 2 / Source 3 id sets.
    column_sources: null
    list_separator: ","
    missing_source1_policy: singleton   # S1 ids absent from labels: singleton | drop | error
    ambiguous_id_policy: skip           # id present in BOTH S2 and S3: skip | both | error
    unknown_id_policy: warn             # id in no source: warn | error

submission:
  matching_results: matching_results.tsv
  candidate_pairs: candidate_pairs.tsv
  scored_candidates: scored_candidates.tsv   # debugging file, not part of the official output
  write_scored_candidates: true
  id_column: null            # null -> mirror the label file's Source-1 id column
  match_columns: null        # null -> mirror the label file's match column(s)
  list_separator: ","
  list_format: auto          # auto (mirror labels) | plain ("a,b") | bracketed ("[a, b]")
  candidate_columns:
    source1_id: source1_id
    candidate_id: candidate_id
    candidate_source: candidate_source
  archive_name: submission.zip
  methodology_doc: METHODOLOGY.md

normalization:
  transliterate: false       # romanise non-Latin scripts with `unidecode` (offline table)
  strip_leading_the: true
  extra_legal_suffixes: {}   # {canonical: [variant, ...]} e.g. {pty ltd: [proprietary limited]}
  extra_name_abbreviations: {}      # {variant: canonical}
  extra_address_abbreviations: {}   # {variant: canonical}

address:
  parser: auto               # auto (libpostal if installed, else rules) | libpostal | rules
  extra_states: {}           # {"full state name": "code"}
  extra_countries: []
  postcode_patterns: null    # null -> built-in multi-region patterns (see src/address_parser.py)

splits:
  # Unit of splitting = one Source-1 entity together with ALL of its labelled
  # Source-2/3 records. fit -> model training; calib -> calibration, blending
  # and decision search; holdout -> untouched unbiased estimate.
  fractions: {fit: 0.70, calib: 0.15, holdout: 0.15}
  inner_val_fraction: 0.15   # share of `fit` entities used only for early stopping

blocking:
  prefix_length: 4
  max_block_size: 2000       # key blocks with more vendor records are purged
  max_candidates_per_source: 60   # final cap per (S1 entity, source)
  cap_method_bonus: 0.02     # cap rank = name+address TF-IDF cosine + bonus x (#methods that found the pair)
  chunk_size: 2000
  char_ngram_range: [2, 4]
  char_max_df: 0.05          # drop char n-grams in >5% of records (keeps sparse products sparse)
  word_max_df: 0.10
  methods:
    name_prefix:        {enabled: true, top_k: 20}
    postcode:           {enabled: true, top_k: 20}
    postcode_phonetic:  {enabled: true, top_k: 20}
    city_state:         {enabled: true, top_k: 15}
    phonetic_name:      {enabled: true, top_k: 20}
    address_key:        {enabled: true, top_k: 20}
    tfidf_name_char:    {enabled: true, top_k: 25}
    tfidf_name_word:    {enabled: true, top_k: 15}
    tfidf_full_char:    {enabled: true, top_k: 25}
    tfidf_address:      {enabled: true, top_k: 15}
    minhash_lsh:        {enabled: true, top_k: 20, num_perm: 64, bands: 16, shingle_size: 3, max_bucket_size: 300,
                         s1_chunk: 20000}   # S1 records per LSH chunk (bounds memory)
    semantic:           {enabled: auto, top_k: 30}   # auto -> enabled in full mode only

features:
  soft_tfidf_threshold: 0.90       # PDF: secondary similarity tolerance 0.90
  soft_tfidf_secondary: jaro_winkler   # jaro_winkler | levenshtein
  jw_prefix_weight: 0.1            # PDF: Winkler scaling factor p = 0.1 (prefix capped at 4)
  context_features: true
  chunk_pairs: 250000              # pairs per feature chunk (lower = less RAM)
  predict_block: 500000            # rows per LightGBM prediction block
  keep_matrix: true                # keep checkpoints/features.npy (disk) after the run

training:
  add_missed_positives: true       # add true pairs lost by blocking to the LightGBM training rows
  neg_downsample: 1.0              # 1.0 = keep every candidate negative

lightgbm:
  params:
    objective: binary
    learning_rate: 0.05
    num_leaves: 63
    min_data_in_leaf: 40
    feature_fraction: 0.8
    bagging_fraction: 0.8
    bagging_freq: 1
    lambda_l2: 1.0
    verbose: -1
  num_boost_round: 3000
  early_stopping_rounds: 100

biencoder:
  # Pre-trained weights must be on local disk (see scripts/download_pretrained.py);
  # nothing is fetched from the internet during training or matching.
  pretrained: models/pretrained/all-MiniLM-L6-v2
  local_files_only: true
  text_source: normalized          # normalized | raw
  max_length: 64
  pooling: mean                    # mean | cls
  temperature: 0.07
  batch_size: 128
  epochs: 5
  lr: 3.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  max_positives_per_entity: 4
  hard_batching: true              # co-locate entities sharing a blocking key in a batch
  extra_negatives_per_batch: 32    # unlabeled vendor records added as in-batch negatives
  flood_level: null                # constant Flooding (Ishida et al. 2020). NOT AdaFlood.
  faiss_index: flat                # flat (exact) | hnsw
  encode_batch_size: 256
  device: auto

crossencoder:
  pretrained: models/pretrained/deberta-v3-base
  local_files_only: true
  text_source: raw                 # raw | normalized
  max_length: 128
  domain_tags: true                # wrap postcodes / numbers in special tags
  tfidf_summarize_max_words: 40    # prune lowest-IDF words above this length
  batch_size: 32
  eval_batch_size: 128
  epochs: 3
  lr: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  focal_gamma: 2.0
  focal_alpha: null                # null = no class re-weighting
  mixda:
    enabled: true
    alpha: 0.8
    ops: [del, swap, drop_col, token_del, attr_shuffle, entry_swap]
  hard_negatives_per_entity: 8
  random_negatives_per_entity: 2
  max_train_pairs: 400000
  device: auto
  fp16: true                       # mixed precision on GPU (bf16 when supported, else fp16)

cascade:
  # Only plausible-but-uncertain pairs reach the cross-encoder.
  lgb_low: 0.02                    # calibrated LightGBM prob below this -> never sent to CE
  lgb_high: 0.995                  # above this -> accepted on LightGBM score (unless rescore_confident)
  rescore_confident: false
  max_per_entity_source: 10        # at most this many pairs per (S1 entity, source) go to the CE

blend:
  method: auto                     # auto | weighted | rank | stack
  min_ce_pairs: 200                # blend only if calib has this many cross-encoder-scored pairs
  min_ce_per_class: 20             # ... with at least this many positives AND negatives
  weight_grid: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

calibration:
  method: isotonic                 # isotonic (PDF) | platt | auto (lower CV Brier score)
  calibrate_blend: true

graph:
  modes_to_try: ["off", "exclusive", "betweenness"]   # quote "off": bare off is a YAML boolean
  min_edge_prob: 0.2
  max_component_size: 50

singleton_model:
  # Entity-level LightGBM predicting P(at least one true match among the
  # candidates) from score-distribution features (top-1, top-2, margin, model
  # agreement, ...). Trained out-of-fold on the calib split only.
  enabled: true
  folds: 5

decision:
  beta: 0.5
  search: true
  threshold_grid: {start: 0.05, stop: 0.99, step: 0.01}
  margins: [1.0, 0.5, 0.3, 0.2, 0.1, 0.05]
  gate_offsets: [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
  max_per_source: [null, 1, 2, 3, 5]
  # Entity output is emptied when P(matchable) < value. 0.0 disables the gate.
  matchable_thresholds: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
  per_source_thresholds: true
  min_improvement: 0.0005          # an extra rule / blend must beat the simpler one by
  min_entities_gain: 2             #   max(min_improvement, min_entities_gain / #calib entities)
  rounds: 3
  # Used only when search: false
  default: {threshold_s2: 0.5, threshold_s3: 0.5, margin: 1.0, gate: 0.5, max_per_source: null, matchable_threshold: 0.0, graph_mode: "off"}

evaluation:
  error_analysis_top_n: 200


In [ ]:
%%writefile configs/low_memory.yaml
# Memory-saving overlay for machines with ~12-16 GB RAM (Colab / Kaggle).
#   python train.py --config configs/low_memory.yaml --set data.train.source1=...   (or edit paths in config.yaml)
# Every setting trades a little speed or recall for a smaller peak; check the
# blocking report (pair completeness before/after cap) to see what the lower
# candidate cap costs on YOUR data.
base: ../config.yaml
n_jobs: 2                         # fewer worker processes
blocking:
  max_candidates_per_source: 30   # fewer candidate pairs -> smaller feature matrix
  chunk_size: 500                 # S1 rows per sparse TF-IDF product
  max_block_size: 1000
  methods:
    minhash_lsh: {enabled: true, top_k: 20, num_perm: 64, bands: 16, shingle_size: 3, max_bucket_size: 200,
                  s1_chunk: 5000}
features:
  chunk_pairs: 100000
  predict_block: 200000
training:
  neg_downsample: 0.3             # LightGBM sees all positives + 30% of negatives (calibration uses all)
submission:
  write_scored_candidates: false  # skip the large debugging file
crossencoder:
  batch_size: 16
  eval_batch_size: 64
biencoder:
  batch_size: 64
  encode_batch_size: 128


In [ ]:
%%writefile scripts/make_synthetic_data.py
#!/usr/bin/env python
"""Generate a small SYNTHETIC dataset with the challenge's structure so the
whole pipeline can be exercised end to end before the real data is used.

It is NOT challenge data and says nothing about real performance. Structure:
  * Source 1: clean, de-duplicated reference records (US / India / UK / DE)
  * Source 2: light vendor noise; Source 3: heavier noise
  * 0, 1 or 2 matches per source per entity; ~30% singletons
  * hard negatives: chain branches (same brand, other address), different
    businesses at the same address, near-name distractors
  * labels: one row per Source-1 entity -> comma-separated matching ids

    python scripts/make_synthetic_data.py --out data/synthetic --n-train 800 --n-test 400
"""
import argparse
import os
import random

import pandas as pd

SYL = ["zor", "vex", "ta", "lin", "mar", "kor", "pix", "del", "ran", "sol", "tek", "nov", "bri", "qua", "hel",
       "dor", "mex", "ri", "sa", "phon", "lum", "cap", "gen", "tor", "vis", "ar", "om", "zen", "kal", "fer"]
SURNAMES = ["Sharma", "Patel", "Miller", "Johnson", "Schmidt", "Garcia", "Iyer", "Reddy", "Brown", "Wagner",
            "Kumar", "Singh", "Taylor", "Fischer", "Nair", "Gupta", "Wilson", "Becker", "Mehta", "Clark"]
INDUSTRY = ["Robotics", "Manufacturing", "Technologies", "Traders", "Enterprises", "Consulting", "Foods",
            "Logistics", "Pharmaceuticals", "Services", "Solutions", "Industries", "Holdings", "Bakery",
            "Motors", "Textiles", "Electricals", "Hardware", "Associates", "International", "Systems",
            "Dental Clinic", "Pharmacy", "Cafe", "Studio", "Brothers", "Engineering", "Laboratories"]
ABBREV = {"Manufacturing": "Mfg", "Technologies": "Tech", "International": "Intl", "Services": "Svcs",
          "Brothers": "Bros", "Associates": "Assoc", "Enterprises": "Ent", "Holdings": "Hldgs",
          "Industries": "Inds", "Engineering": "Engg", "Laboratories": "Labs", "Systems": "Sys",
          "Pharmaceuticals": "Pharma", "Consulting": "Consultants"}
REGIONS = {
    "US": {
        "cities": [("San Jose", "CA", "California", "951"), ("Austin", "TX", "Texas", "787"),
                   ("Seattle", "WA", "Washington", "981"), ("Denver", "CO", "Colorado", "802"),
                   ("Boston", "MA", "Massachusetts", "021"), ("Portland", "OR", "Oregon", "972")],
        "streets": ["Market", "Elm", "Oak", "Maple", "Main", "Park", "Lake", "Hill", "Cedar", "Pine", "5th", "Mission"],
        "types": [("Street", "St"), ("Avenue", "Ave"), ("Road", "Rd"), ("Boulevard", "Blvd"), ("Drive", "Dr"), ("Lane", "Ln")],
        "suffixes": [["Inc", "Inc.", "Incorporated"], ["LLC", "L.L.C."], ["Corp", "Corporation"], ["Co", "Company"]],
    },
    "IN": {
        "cities": [("Bengaluru", "", "Karnataka", "560"), ("Mumbai", "", "Maharashtra", "400"),
                   ("Chennai", "", "Tamil Nadu", "600"), ("Pune", "", "Maharashtra", "411"),
                   ("Kolkata", "", "West Bengal", "700"), ("Gurgaon", "", "Haryana", "122")],
        "streets": ["MG", "Station", "Temple", "Ring", "Nehru", "Gandhi", "Lake View", "Church", "Hosur", "Link"],
        "types": [("Road", "Rd"), ("Marg", "Marg"), ("Main Road", "Main Rd")],
        "areas": ["Indiranagar", "Koramangala", "Andheri East", "Bandra West", "Salt Lake", "Anna Nagar",
                  "Sector 18", "Kothrud", "Whitefield", "Powai"],
        "landmarks": ["Bus Stand", "Railway Station", "City Mall", "Hanuman Temple", "Post Office", "City Hall"],
        "suffixes": [["Pvt Ltd", "Private Limited", "Pvt. Ltd."], ["LLP"], ["Ltd", "Limited"]],
    },
    "UK": {
        "cities": [("London", "", "", "SW1"), ("Manchester", "", "", "M1"), ("Leeds", "", "", "LS1"),
                   ("Bristol", "", "", "BS1")],
        "streets": ["Downing", "King", "Queen", "High", "Church", "Station", "Victoria"],
        "types": [("Street", "St"), ("Road", "Rd"), ("Lane", "Ln")],
        "suffixes": [["Ltd", "Limited"], ["PLC", "plc"], ["LLP"]],
    },
    "DE": {
        "cities": [("Berlin", "", "", "101"), ("Hamburg", "", "", "201"), ("München", "", "", "803")],
        "streets": ["Haupt", "Bahnhof", "Garten", "Schiller", "Goethe", "Linden"],
        "types": [("straße", "str.")],
        "suffixes": [["GmbH"], ["AG"], ["GmbH & Co. KG"]],
    },
}


class Gen:
    def __init__(self, seed):
        self.r = random.Random(seed)

    def brand(self):
        r = self.r
        return "".join(r.choice(SYL) for _ in range(r.choice([2, 2, 3]))).capitalize()

    def business(self, region):
        r = self.r
        reg = REGIONS[region]
        style = r.random()
        if style < 0.55:
            core = f"{self.brand()} {r.choice(INDUSTRY)}"
        elif style < 0.8:
            core = f"{r.choice(SURNAMES)} {r.choice(INDUSTRY)}"
        else:
            core = f"{r.choice(SURNAMES)} & Sons {r.choice(INDUSTRY)}"
        suffix_group = r.choice(reg["suffixes"])
        return {"core": core, "suffix_group": suffix_group, "suffix": suffix_group[0] if r.random() < 0.85 else ""}

    def address(self, region):
        r = self.r
        reg = REGIONS[region]
        city = r.choice(reg["cities"])
        a = {"region": region, "num": str(r.randint(1, 2500)), "street": r.choice(reg["streets"]),
             "type": r.choice(reg["types"]), "city": city[0], "state_code": city[1], "state": city[2],
             "unit": f"Suite {r.randint(100, 900)}" if (region == "US" and r.random() < 0.25) else ""}
        if region == "US":
            a["post"] = city[3] + f"{r.randint(0, 99):02d}"
        elif region == "IN":
            a["post"] = city[3] + f"{r.randint(0, 999):03d}"
            a["area"] = r.choice(reg["areas"])
            a["landmark"] = r.choice(reg["landmarks"]) if r.random() < 0.5 else ""
            a["unit"] = f"Shop No {r.randint(1, 40)}" if r.random() < 0.3 else ""
        elif region == "UK":
            a["post"] = f"{city[3]}{r.choice('ABCDEFGH')} {r.randint(1, 9)}{r.choice('ABDEFGHJ')}{r.choice('LNPQRSTU')}"
        else:
            a["post"] = city[3] + f"{r.randint(0, 99):02d}"
        return a


def render_name(b, noise, g):
    r = g.r
    core = b["core"]
    suffix = b["suffix"]
    if noise:
        ops = r.sample(["suffix", "abbrev", "case", "typo", "drop", "and", "the", "punct", "typo2", "brand_only"],
                       k=min(noise, 10))
        for op in ops:
            if op == "suffix":
                suffix = r.choice(b["suffix_group"] + [""])
            elif op == "abbrev":
                for k, val in ABBREV.items():
                    core = core.replace(k, val)
            elif op == "case":
                core, suffix = (core.upper(), suffix.upper()) if r.random() < 0.5 else (core.lower(), suffix.lower())
            elif op == "typo":
                core = typo(core, r)
            elif op == "typo2":
                core = typo(typo(core, r), r)
            elif op == "brand_only":
                core = core.split()[0]
            elif op == "drop":
                toks = core.split()
                if len(toks) > 2:
                    toks.pop(r.randrange(1, len(toks)))
                    core = " ".join(toks)
            elif op == "and":
                core = core.replace("&", "and") if "&" in core else core
            elif op == "the":
                core = "The " + core
            elif op == "punct":
                suffix = (", " + suffix) if suffix else suffix
    return f"{core} {suffix}".replace(" ,", ",").strip()


def typo(text, r):
    if len(text) < 5:
        return text
    i = r.randrange(1, len(text) - 2)
    op = r.random()
    if op < 0.33:
        return text[:i] + text[i + 1] + text[i] + text[i + 2:]
    if op < 0.66:
        return text[:i] + text[i + 1:]
    return text[:i] + r.choice("aeiourstln") + text[i:]


def render_address(a, noise, g):
    r = g.r
    reg = a["region"]
    ops = set(r.sample(["abbrev", "state", "drop_post", "drop_city", "unit", "typo", "landmark", "case",
                        "drop_num", "space"], k=noise)) if noise else set()
    t_full, t_short = a["type"]
    stype = t_short if "abbrev" in ops else t_full
    street = typo(a["street"], r) if "typo" in ops and len(a["street"]) > 4 else a["street"]
    num = "" if "drop_num" in ops else a["num"]
    post = "" if "drop_post" in ops else a["post"]
    city = "" if "drop_city" in ops else a["city"]
    unit = a.get("unit", "")
    if "unit" in ops:
        unit = "" if unit else ("Ste " + str(r.randint(100, 900)) if reg == "US" else unit)
    if reg == "US":
        state = a["state"] if "state" in ops else a["state_code"]
        parts = [f"{num} {street} {stype}".strip() + (f" {unit}" if unit else ""), city, f"{state} {post}".strip()]
    elif reg == "IN":
        state = a["state"]
        if "space" in ops and post:
            post = post[:3] + " " + post[3:]
        lm = a.get("landmark", "")
        if "landmark" in ops:
            lm = lm or r.choice(REGIONS["IN"]["landmarks"])
        parts = [unit, f"No {num}" if num and r.random() < 0.5 else num, f"{street} {stype}",
                 f"Near {lm}" if lm else "", a["area"], city, f"{state} {post}".strip()]
    elif reg == "UK":
        parts = [f"{num} {street} {stype}".strip(), f"{city} {post}".strip()]
    else:
        parts = [f"{street}{stype} {num}".strip(), f"{post} {city}".strip()]
    text = ", ".join(p for p in parts if p)
    if "case" in ops:
        text = text.upper()
    return text


def generate(n, seed, id_offset, test=False):
    g = Gen(seed)
    r = g.r
    s1, s2, s3, labels = [], [], [], []
    c1 = c2 = c3 = 0
    entities = []
    for i in range(n):
        region = r.choices(["US", "IN", "UK", "DE"], weights=[0.4, 0.35, 0.15, 0.1])[0]
        b = g.business(region)
        a = g.address(region)
        entities.append((b, a))
        # chain branch: same brand, another address, separate S1 entity
        if r.random() < 0.25 and len(entities) < n:
            entities.append(({**b}, g.address(region)))
    entities = entities[:n]
    for b, a in entities:
        c1 += 1
        s1_id = f"A{id_offset + c1:07d}"
        s1.append({"record_id": s1_id, "business_name": render_name(b, 0, g), "full_address": render_address(a, 0, g)})
        k2 = r.choices([0, 1, 2], weights=[0.35, 0.55, 0.10])[0]
        k3 = r.choices([0, 1, 2], weights=[0.45, 0.47, 0.08])[0]
        if r.random() < 0.18:
            k2 = k3 = 0   # extra singletons
        matches = []
        for _ in range(k2):
            c2 += 1
            vid = f"B{id_offset + c2:07d}"
            s2.append({"record_id": vid, "business_name": render_name(b, r.randint(0, 3), g),
                       "full_address": render_address(a, r.randint(0, 3), g)})
            matches.append(vid)
        for _ in range(k3):
            c3 += 1
            vid = f"C{id_offset + c3:07d}"
            s3.append({"record_id": vid, "business_name": render_name(b, r.randint(1, 4), g),
                       "full_address": render_address(a, r.randint(1, 5), g)})
            matches.append(vid)
        labels.append({"source1_id": s1_id, "matched_ids": ",".join(matches)})
        # hard negatives in vendor data
        if r.random() < 0.25:   # other business at the same address
            other = g.business(a["region"])
            c2 += 1
            s2.append({"record_id": f"B{id_offset + c2:07d}", "business_name": render_name(other, 1, g),
                       "full_address": render_address(a, 1, g)})
        if r.random() < 0.30:   # same brand, different location (not in S1)
            c3 += 1
            s3.append({"record_id": f"C{id_offset + c3:07d}", "business_name": render_name(b, 1, g),
                       "full_address": render_address(g.address(a["region"]), 1, g)})
    # unrelated vendor records
    for _ in range(int(0.3 * n)):
        region = r.choice(list(REGIONS))
        c2 += 1
        s2.append({"record_id": f"B{id_offset + c2:07d}", "business_name": render_name(g.business(region), 1, g),
                   "full_address": render_address(g.address(region), 1, g)})
        c3 += 1
        s3.append({"record_id": f"C{id_offset + c3:07d}", "business_name": render_name(g.business(region), 2, g),
                   "full_address": render_address(g.address(region), 2, g)})
    shuf = lambda rows: r.sample(rows, len(rows))
    return (pd.DataFrame(shuf(s1)), pd.DataFrame(shuf(s2)), pd.DataFrame(shuf(s3)), pd.DataFrame(labels))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/synthetic")
    ap.add_argument("--n-train", type=int, default=800)
    ap.add_argument("--n-test", type=int, default=400)
    ap.add_argument("--seed", type=int, default=7)
    args = ap.parse_args()
    for split, n, seed, off in (("train", args.n_train, args.seed, 0), ("test", args.n_test, args.seed + 1, 5_000_000)):
        s1, s2, s3, lab = generate(n, seed, off, test=split == "test")
        d = os.path.join(args.out, split)
        os.makedirs(d, exist_ok=True)
        for name, df in (("source1", s1), ("source2", s2), ("source3", s3)):
            df.to_csv(os.path.join(d, f"{name}.tsv"), sep="\t", index=False)
        if split == "train":
            lab.to_csv(os.path.join(d, "labels.tsv"), sep="\t", index=False)
        else:  # kept apart: never read by the pipeline, only by the demo scorer
            lab.to_csv(os.path.join(args.out, "test_labels_HIDDEN.tsv"), sep="\t", index=False)
        print(f"{split}: S1={len(s1)} S2={len(s2)} S3={len(s3)} singletons={(lab['matched_ids'] == '').sum()}")


if __name__ == "__main__":
    main()


In [ ]:
import importlib
importlib.invalidate_caches()
if "src" in sys.modules:
    print("NOTE: src was imported before - restart the kernel if you edited a module cell")
import src, src.training, src.prediction
from src.utils import mem_str
print("pipeline", src.__version__, "|", mem_str())
try:
    import psutil
    print(f"available RAM: {psutil.virtual_memory().available / 2**30:.1f} GB")
except ImportError:
    pass

In [ ]:
%%writefile METHODOLOGY.md
# Methodology — Business Entity Resolution (Amazon ML Challenge 2026)

This document describes the approach implemented in this repository. It is
included in the submission archive, as the challenge requires. It covers the
problem, each pipeline stage, how validation avoids leakage, the design
decisions, the assumptions, and a compliance audit.

References:
* **[T]** `ML_Challenge_2026_Video_Transcript.docx`, the official problem statement.
* **[A]** `Entity Resolution System Architecture.pdf`, the proposed architecture this code implements.

---

## 1. Problem (from [T])

* There are three independent sources. **Source 1** is the clean, de-duplicated reference list. **Sources 2 and 3** are noisy vendor fragments. No identifier is shared between them, so only the **business name and address** are available.
* For **every** Source-1 entity, find **all** matching records in Sources 2 and 3. An entity can have many matches, exactly one, or none.
* Ground truth has one row per Source-1 entity: its id maps to a comma-separated list of all matching ids, and the list is empty when there is no match. [T] says this "mirrors exactly what you'll submit".
* Every file is **tab-separated**.
* The deliverables are `matching_results.tsv` (one row per Source-1 entity, which is scored) and, at the end, one archive. The archive contains the final matches, the **candidate/unscored pairs** from blocking, the complete reproducible pipeline, and a methodology document.
* The metric is **Macro F0.5**, which weights precision twice as heavily as recall. A singleton scores 1 when its prediction is empty and 0 otherwise.
* Blocking sets the recall ceiling. [T] advises paying attention to region-specific patterns.
* External databases, APIs and lookups are **strictly prohibited**.

## 2. Pipeline

```
TSV ingestion ─► name normalisation + offline address parsing
      ─► hybrid blocking (11 syntactic methods [+ SupCon bi-encoder / FAISS]) ─► candidate_pairs.tsv
      ─► 103 pairwise features ─► LightGBM ─► isotonic calibration
      ─► [cascade ─► Ditto/DeBERTa-v3 cross-encoder ─► isotonic] ─► blend ─► final calibration
      ─► graph cleanup (off / exclusive / betweenness, chosen on validation)
      ─► decision rules tuned for Macro F0.5 (+ singleton gate) ─► matching_results.tsv
```

`mode: baseline` runs everything except the neural parts in brackets. `mode: full` runs the complete architecture from [A].

### 2.1 Ingestion (`src/data_io.py`, `src/schema.py`)
Every read uses `sep="\t"`, `dtype=str` and `keep_default_na=False`, so a business called "NA" is not turned into a missing value. Column names come from the config because [T] does not name them. The loader validates the schema and reports the columns it actually found. It counts empty names and addresses, and detects duplicate ids and duplicate name+address rows. Labels are parsed robustly (`a,b`, `[a, b]`, `['a','b']`, empty). Each label id is resolved to Source 2 or 3 by lookup, or by a per-column mapping when the labels use one column per source.

### 2.2 Name normalisation ([A] "Lexical Standardization and Legal Suffix Abstraction"; `src/normalization.py`)
1. NFKD decomposition. Combining marks are removed only after Latin letters, so Devanagari and similar scripts are not broken.
2. `str.casefold()` (for example, "Straße" becomes "strasse").
3. Dotted acronyms are collapsed ("S.A." → "sa"), letter/letter slashes are joined ("M/s" → "ms"), apostrophes are dropped, "&" becomes "and", and all other punctuation becomes a space.
4. **Legal suffixes are extracted by longest-first matching at the end of the name**, repeated so that compounds like "gmbh & co kg" or "co ltd" are removed whole. The table covers more than 45 legal forms (US, UK, India, DE, LatAm, EU, Asia). As [A] requires, the suffix is **kept**: its canonical form and family (corporation / llc / limited / gmbh / sa / …) become features (`suffix_eq`, `suffix_family_eq`, `suffix_conflict`). A suffix is never removed if that would leave the name empty.
5. Conservative canonicalisation of abbreviations ("manufacturing"/"mfg" → `mfg`, "holdings"/"hldgs" → `hldg`, …). Both the long and short forms map to the same token.
6. Derived keys: content tokens, a no-space string, Metaphone codes (whole name and first token), initials, and digits.

### 2.3 Address parsing ([A] "Statistical Address Parsing in an Offline Context"; `src/address_parser.py`)
* **libpostal**, when installed, is used offline through the `postal` bindings. It is a local C library and model, and it makes no network calls.
* An offline **rule-based parser** covering several regions is always available. It extracts the country, postcode (US ZIP/ZIP+4, Indian PIN including "560 038", UK, Canada, and 5-/4-digit codes), state (US, CA and AU codes and names; Indian state names), unit/suite/shop, landmarks ("Nr/Near/Opp …", the PDF's "house" component), house number (including "No 12", "Plot No 45", and European "Hauptstr 5"), road, city and suburb. It reproduces all three examples in the PDF's parsing table (see `tests/test_address_parser.py`).
* Address tokens are canonicalised (street→st, road→rd, suite→ste, near→nr, fifth→5th, "-strasse"→"-str").
* **No geocoder, map API or OSM query is used anywhere.**

### 2.4 Blocking ([A] "Algorithmic Paradigms for Candidate Generation"; `src/blocking.py`)
Candidates are generated separately for Source 2 and Source 3, for each Source-1 entity. The final set is the union of these methods:

| method | key / technique | handles |
|---|---|---|
| name_prefix | first 4 chars of the core name ([A]) | cheap exact-ish names |
| postcode | exact normalised postcode ([A]) | same area |
| postcode_phonetic | postcode + Metaphone of first token ([A]: "postal codes combined with phonetic hashing") | spelling variants locally |
| city_state | parsed city + state | missing postcodes |
| phonetic_name | Metaphone of the whole name | phonetic misspellings |
| address_key | house number + road token | **different names at the same address** ([T]) |
| tfidf_name_char / _word | TF-IDF cosine top-k ([A]) | typos, token reordering |
| tfidf_full_char | name + address char n-grams | joint evidence |
| tfidf_address | address-only char n-grams | shared address |
| minhash_lsh | MinHash (64 perms, 16 bands) over name shingles ([A]) | fault-tolerant Jaccard |
| semantic (full) | SupCon bi-encoder + FAISS top-k ([A] SC-Block) | abbreviations, paraphrases |

A key block larger than its `top_k` is ranked internally by TF-IDF cosine. Blocks larger than `max_block_size` are purged. Each (entity, source) then keeps at most `max_candidates_per_source` pairs. These are ranked by name+address TF-IDF cosine plus a bonus of 0.02 for each method that retrieved the pair.
*Why this ranking:* ranking by name similarity alone let many same-name branches of a common business push the true, address-matching record out of the cap. On a 20k-entity synthetic stress test the old ranking kept 95.7% of true pairs and the new one keeps 99.6%. The union before the cap held 99.7%.

The blocking report (`outputs/<run>/train/blocking_report.json`) contains pair completeness (overall, per source, per split, and before/after the cap), the reduction ratio, total possible and candidate pairs, the distribution of candidates per entity, the number of true pairs lost, per-method recall and unique contribution, and the **Macro-F0.5 ceiling that blocking imposes**. That ceiling is the score an oracle matcher would reach given these candidates. `candidate_pairs.tsv` is written **before** any model scores the pairs.

### 2.5 Labels, hard negatives and leakage control (`src/labels.py`, `src/splits.py`)
* **Positives** are the (S1, S2) and (S1, S3) pairs from the labels. True pairs that blocking missed are added to the LightGBM training rows. Their context features are left as NaN because the pair was never retrieved.
* **Negatives** are all other candidates. A candidate whose id appears in the entity's label list but cannot be resolved without ambiguity gets label −1 and is excluded, so a true match can **never** become a negative.
* **Hard negatives** come naturally from blocking: similar names, shared addresses, and same-brand branches. They are tagged (`similar_name`, `shared_address`, …) and reported. The cross-encoder trains on the highest-scoring LightGBM negatives of each entity plus a few random ones. The bi-encoder places entities with the same name prefix in the same batch.
* **Splitting unit:** one Source-1 entity together with all of its labelled Source-2/3 records. Because Source 1 is de-duplicated, a business belongs to exactly one split. The split is stratified on (#S2 matches, #S3 matches).
  * `fit` trains every model; `fit_val` (a subset of fit) is used only for early stopping.
  * `calib` fits the isotonic calibrators, the blend, the singleton model, the graph mode and the decision parameters.
  * `holdout` is never used for fitting and gives the reported estimate.
* **Guards against leakage:**
  * No test label exists or is read.
  * Thresholds are tuned only on `calib`.
  * The semantic-retrieval flag is **not** given to LightGBM as a feature, because on fit pairs it is in-sample.
  * Graph cleanup during evaluation only sees the pairs of the split being evaluated.
  * TF-IDF/IDF statistics are unsupervised statistics of the text in the dataset being processed. They use no labels, and exactly the same computation runs at test time.

### 2.6 Structural matcher: LightGBM ([A]; `src/features.py`, `src/train_lightgbm.py`)
The model uses 103 features:
* **Name:** exact (raw, clean, core, no-space), Jaro–Winkler (p = 0.1, prefix ≤ 4, as in [A]), Levenshtein, Indel, token sort/set, partial ratio, char/word TF-IDF cosine, **Soft-TFIDF** (secondary similarity Jaro–Winkler ≥ 0.90, per [A]; Levenshtein optional), token Jaccard/overlap, IDF-weighted overlap, highest shared IDF, IDF of unshared tokens, phonetic equality, prefix equality, common-prefix length, lengths, containment, acronym match, digits, and suffix agreement/conflict.
* **Address:** exact, Jaro–Winkler, token set, char TF-IDF, Soft-TFIDF, token Jaccard, and for each component (house number, road, city, state, postcode, unit) both-present / equality / conflict flags. Also postcode prefix, overlap of numeric tokens and a conflict flag, landmarks, suburb overlap, and missing-field indicators.
* **Combined:** name × address, strong-name/weak-address, weak-name/strong-address, number of agreeing and conflicting components, a total contradiction count, source id, and blocking provenance flags.
* **Label-free context:** the pair's rank, its gap to the best candidate and its margin over the best *other* candidate within the (entity, source) group, the same from the vendor's side, and a reciprocal-best flag.

Missing values are NaN, which LightGBM routes natively. Training uses early stopping on `fit_val`. Gain-based importance is saved to `models/<run>/feature_importance.tsv`, as [A] suggests.

### 2.7 Semantic blocking: SupCon bi-encoder ([A] SC-Block; `src/train_biencoder.py`)
Records are encoded by a shared MiniLM encoder (mean pooling, L2-normalised) trained with the batch-wise **Supervised Contrastive loss**. Each S1 entity and its labelled vendor records share one label. Singletons and extra vendor records get unique labels and act only as negatives. Batches are hard: similar keys go in the same batch, plus vendor negatives with the same key. The checkpoint is chosen by recall@k on `fit_val`. Every vendor record is encoded once and indexed per source in FAISS (`IndexFlatIP`, or HNSW for large data). The encoder and indices are saved.
**AdaFlood is not implemented.** [A] names it but gives no formula, and the per-sample flood levels need auxiliary models whose training procedure for SupCon is not specified. An optional *constant* Flooding term (Ishida et al., 2020) exists and is disabled by default. It is not AdaFlood.

### 2.8 Semantic matcher: Ditto cross-encoder ([A]; `src/train_crossencoder.py`)
* **Serialisation:** `COL name VAL … COL address VAL …` for each side. The tokenizer adds `[CLS]`/`[SEP]`, and a linear head on the `[CLS]` state outputs one logit. The default backbone is DeBERTa-v3.
* **Domain-knowledge injection:** postcodes are wrapped as `[PC] … [/PC]` and other digit-bearing tokens as `[NUM] … [/NUM]`. These tags are new special tokens.
* **MixDA:** each batch gets one of the operators del / swap / drop_col / token_del / attr_shuffle / entry_swap. The `[CLS]` encodings of the original and augmented inputs are interpolated with λ ~ Beta(α, α), and λ = max(λ, 1 − λ). This follows Ditto's MixDA.
* **TF-IDF summarisation** keeps only the highest-IDF words of any value longer than `tfidf_summarize_max_words`.
* **Focal loss** uses γ = 2, with optional α.
* **Cascade:** the cross-encoder only scores pairs where the calibrated LightGBM probability p is at least `lgb_low`, the pair is among the top `max_per_entity_source` of its (entity, source), and p is at most `lgb_high`. All other pairs keep their LightGBM probability.

### 2.9 Calibration, blending and decision ([A] "Isotonic Regression and Decision Threshold Calibration"; `src/calibration.py`, `src/scoring.py`, `src/decision.py`)
* Isotonic regression is fitted on `calib`, separately for each model. Platt scaling is also evaluated, and the cross-validated Brier score, log loss and ECE are reported. `method: auto` picks whichever is better.
* **Blending** can be weighted, rank-based or stacking. It is chosen on `calib` by Macro F0.5, and a blend must beat LightGBM alone by `min_improvement`. A final isotonic calibration is then applied.
* **Decision rules** (per entity):
  * a separate threshold for Source 2 and Source 3;
  * a margin to the entity's best score;
  * a singleton **gate** on the best score;
  * an optional **singleton model** that empties an entity when P(entity has a true match among its candidates) is below a threshold. This is an entity-level LightGBM on top-1/top-2 scores, the margin, counts, model agreement and the evidence of the top candidate, trained out-of-fold on `calib`;
  * an optional per-source cap;
  * a graph mode.
* The parameters are chosen by coordinate ascent that maximises the **exact entity-level Macro F0.5**. Every entity counts, including entities that have no candidates and true matches lost in blocking. For thresholds the search picks the **middle of the best plateau**, because calibrated scores are nearly binary and the first threshold of a tie sits on the edge. Any other rule must improve the score by `min_improvement` to be adopted. No threshold value is assumed in advance.

### 2.10 Graph cleanup ([A] GraLMatch/TransClean; `src/graph_cleanup.py`)
Analysis:
* The output is centred on Source 1, so no global clustering is required.
* Because Source 1 is de-duplicated, the only transitive error that can affect the output is a vendor record linked to **two** Source-1 entities. Transitivity would then declare two distinct reference businesses equal.
* The graph used is therefore bipartite, and each component may contain at most one S1 node.

The modes are:
* `exclusive`: each vendor keeps only its best S1 edge.
* `betweenness`: [A]'s procedure. The weakest separating bridge is cut first ("minimum edge cut"); otherwise the edge with the highest edge-betweenness × (1 − p) is cut. This repeats until no component holds two S1 nodes or exceeds the size limit.
* `off`.

The mode is chosen on `calib`, and cleanup is used only if it helps. The labels report counts vendor ids that appear under more than one S1 entity, which tests the at-most-one-S1-per-vendor assumption against the ground truth.

### 2.11 Output (`src/submission.py`)
`matching_results.tsv` has one row per Source-1 entity, in the original order. Predicted ids are comma-separated and sorted by score, and the field is empty for predicted singletons. The column names mirror the label file, or follow a configured sample submission. `candidate_pairs.tsv` has the columns `source1_id`, `candidate_id` and `candidate_source`. `scored_candidates.tsv` holds every score and decision for debugging. A local validator checks for tabs, the header, one row per entity, no duplicates, and that every id exists.

### 2.12 Engineering: bounded memory and resumable stages
The data size of the real challenge is unknown, and the pipeline has to run on ~12–16 GB notebook machines.
* **Blocking:** each vendor source is blocked, deduplicated and capped before the next source starts. The key-block ranking matrices are limited to about 2M cells, and MinHash-LSH processes Source-1 records in chunks with a running top-k.
* **Reports and labels:** the blocking report and label attachment use sorted integer pair ids, not Python sets of tuples.
* **Features:** pairwise features are computed in chunks by one reused worker pool. The parent calls `gc.freeze()` before forking, so the workers do not duplicate its heap. Results are streamed into a disk-backed float32 matrix (`features.npy`, memory-mapped), and LightGBM trains on the needed rows and predicts in blocks.
* **Stages:** training runs as seven stages and prediction as four. Each stage reads its inputs from checkpoints and writes its outputs back, so a crashed or restarted process resumes (`--resume`, or one notebook cell per stage) without recomputing earlier stages.
* **Memory release:** freed memory is handed back to the OS after every stage (`malloc_trim`).

## 3. Design decisions

1. **Normalisation.** Vendors write the same business differently ([T]: "Acme Robotics Incorporated" vs "Inc"). Without normalisation, string similarity and blocking keys fail on formatting alone. Suffixes are removed from the matching key but kept as evidence ([A]).
2. **Blocking.** Comparing every pair is |S1|·(|S2|+|S3|) comparisons, which is impossible at scale ([T], [A]).
3. **Several blocking methods.** Each key is brittle ([A]: "a single typographical error … will permanently separate a true match"). [T] shows that matches group through the name *or* through the address. A union of independent methods raises recall, and the per-method report shows what each one contributes.
4. **Candidate recall.** "You cannot match a record you never consider" ([T]). The report states the resulting Macro-F0.5 ceiling directly.
5. **LightGBM.** It is fast, handles NaN-rich tabular similarity features well, and is interpretable through gain importance ([A]). It scores every candidate, which makes it the base of the cascade.
6. **Semantic model.** Abbreviations, transliterations and paraphrases defeat character similarity ([A]: "Acme Mfg" vs "Acme Manufacturing").
7. **Bi-encoder for retrieval.** Records are encoded independently, so vendor embeddings are computed once and searched sub-linearly with FAISS ([A]). This is the only affordable way to apply a neural model to *all* records.
8. **Cross-encoder for hard cases.** Joint attention over both records catches token-level contradictions (for example, the house number differs) that a bi-encoder averages away. It costs far more per pair, so it only sees the uncertain part of the cascade.
9. **Calibration.** Raw LightGBM or transformer scores are not probabilities ([A]). Calibrated scores make blending meaningful and thresholds interpretable.
10. **Why Macro F0.5 moves the threshold.** A false merge costs roughly twice what a miss costs ([T]). The averaging is per entity, and a singleton goes from 1 to 0 with a single false id. So the best threshold is the empirical maximiser of *entity-level* F0.5 on validation, usually well above 0.5, and not the pairwise-F1 threshold. The value is measured, not assumed. [A]'s "0.85–0.95" is not taken as given.
11. **Singletons.** Each singleton is worth exactly as much as any other entity, and they are a sizable share of the data (see the label report). The gate and the singleton model target the rule "when in doubt, it is safer not to merge" ([T]).
12. **Graph cleanup.** It is not strictly necessary for a Source-1-centric output. The only meaningful case is one vendor record linked to two reference businesses. It is implemented as an option chosen on validation, and applying it blindly could remove correct links.
13. **A Source-1-centric output.** The task and the submission are defined per Source-1 entity ([T]). Source 2 and 3 records are never matched to each other, and the output always has exactly one row per Source-1 entity, even when the list is empty.

## 4. Assumptions and limitations
* **ASSUMPTION – schema.** [T] names no files or columns. All names come from the config, and `inspect_data.py` prints the real ones.
* **ASSUMPTION – label ids.** The ids in the combined list are resolved to a source by lookup. Ids present in both Source 2 and Source 3 are ambiguous: they are skipped for training and still count at string level in evaluation. One column per source is supported through `column_sources`.
* **ASSUMPTION – output format.** It mirrors the label file (per [T]), or a configured sample submission. The organisers' validation script must still be run.
* **ASSUMPTION – metric details.** Per-entity F0.5 on sets of ids; an empty prediction for an empty truth scores 1; an empty prediction for a non-empty truth scores 0; the average is unweighted over every Source-1 entity. This follows [T]. The official scorer's exact implementation is not public.
* **ASSUMPTION – candidate file format.** [T] only says "candidate/unscored pairs .tsv", so the columns used are (`source1_id`, `candidate_id`, `candidate_source`).
* **ASSUMPTION – calibration order.** [A] places isotonic calibration both on the raw model outputs (§ "Isotonic Regression") and after graph cleanup (§ "Synthesis"). The code calibrates each model, blends, calibrates again, and runs graph cleanup on those calibrated edge weights.
* **Not reproducible from [A]: AdaFlood.** See §2.7.
* **Pre-trained weights.** Full mode needs DeBERTa-v3 and MiniLM weights on local disk (`scripts/download_pretrained.py`). The build environment could not reach the HuggingFace hub, so full mode was verified end to end only with *tiny randomly initialised* local models (`scripts/build_local_tiny_transformer.py`). **No result from real pre-trained weights or real challenge data is claimed.**
* **Region coverage.** The rule-based parser has dictionaries for US/CA/AU/IN states and common postcode formats. Other regions work through the generic rules or libpostal, and the dictionaries can be extended in the config.
* **Refitting.** Models are trained on `fit` only (about 70% of the labelled entities). Refitting on all labels would change score distributions, which would invalidate the calibrators and thresholds, so it is not done by default.

## 5. Compliance audit against [T]

| Requirement | Status |
|---|---|
| No external database / API / lookup / geocoding / business lookup during matching | ✔ No network code in `src/`. Address parsing is local (rules or libpostal). Transformer weights are loaded with `local_files_only=True`. |
| Only the provided data is used for matching | ✔ Every feature, IDF statistic, tokenizer and model is built from the challenge TSVs. The one external input in full mode is generic pre-trained LM weights, downloaded once and explicitly (flagged below). |
| Output format | ✔ One row per Source-1 entity, TSV, comma-separated ids, empty for singletons. Mirrors the label file. Checked by a local validator. |
| candidate_pairs generated before matching | ✔ Written by `predict.py` right after blocking, before any feature or model runs. |
| Singletons can produce empty lists | ✔ Default for any entity with no accepted candidate, plus an explicit gate and singleton model. |
| Macro F0.5 is the optimisation target | ✔ The decision search maximises the exact entity-level Macro F0.5 on the calib split. |
| TSV with an explicit tab separator | ✔ `read_tsv` / `write_tsv` always use `\t`. |
| Reproducible pipeline + methodology in the archive | ✔ `package_submission.py` bundles the code, configs, this document, `candidate_pairs.tsv` and `matching_results.tsv`. |

**Flagged components.**
1. *libpostal* is trained on OpenStreetMap data, but it runs entirely offline, and [A] recommends it. It is optional, and the rule-based parser is the fallback.
2. *Pre-trained transformer weights* contain general language knowledge, not business records, and [A] requires them for full mode. If the organisers consider them out of bounds, the compliant alternative is `mode: baseline`, which uses only the challenge data. A second option is the tiny from-scratch models built by `scripts/build_local_tiny_transformer.py`, whose tokenizer is trained only on the training TSVs.


## 2. Configuration (same settings as the training notebook)

In [ ]:
import subprocess, yaml
from src.config import deep_merge, load_config

RUN_NAME = "baseline_v1"      # artefacts -> models/<RUN_NAME>/, outputs -> outputs/<RUN_NAME>/
MODE = "baseline"             # "full" adds the SupCon bi-encoder + DeBERTa-v3 cross-encoder (GPU, local weights)
LOW_MEMORY = True             # use configs/low_memory.yaml (recommended for ~12-16 GB RAM)
USE_SYNTHETIC_DEMO = False    # True -> generate a small fake dataset and run the whole notebook on it

# ---- the real challenge files (tab-separated) -----------------------------
DATA_DIR = "/kaggle/input/amazon-ml-challenge-2026"     # <- change to where your TSVs are
SETTINGS = {
    "run_name": RUN_NAME,
    "mode": MODE,
    "data": {
        "train": {"source1": f"{DATA_DIR}/train/source1.tsv", "source2": f"{DATA_DIR}/train/source2.tsv",
                  "source3": f"{DATA_DIR}/train/source3.tsv", "labels": f"{DATA_DIR}/train/labels.tsv"},
        "test": {"source1": f"{DATA_DIR}/test/source1.tsv", "source2": f"{DATA_DIR}/test/source2.tsv",
                 "source3": f"{DATA_DIR}/test/source3.tsv"},
    },
    # column names of the real files - check them with the "Inspect the data" cell
    "schema": {
        "source1": {"id": "id", "name": "name", "address": ["address"]},
        "source2": {"id": "id", "name": "name", "address": ["address"]},
        "source3": {"id": "id", "name": "name", "address": ["address"]},
        "labels": {"source1_id": "source1_id", "match_columns": ["matches"]},
    },
    # full mode only: local folders holding the pre-trained weights
    "biencoder": {"pretrained": "models/pretrained/all-MiniLM-L6-v2"},
    "crossencoder": {"pretrained": "models/pretrained/deberta-v3-base"},
}

if USE_SYNTHETIC_DEMO:
    if not os.path.exists("data/synthetic/train/labels.tsv"):
        subprocess.check_call([sys.executable, "scripts/make_synthetic_data.py", "--out", "data/synthetic",
                               "--n-train", "800", "--n-test", "400"])
    d = "data/synthetic"
    SETTINGS["data"] = {
        "train": {**{f"source{i}": f"{d}/train/source{i}.tsv" for i in (1, 2, 3)}, "labels": f"{d}/train/labels.tsv"},
        "test": {f"source{i}": f"{d}/test/source{i}.tsv" for i in (1, 2, 3)},
    }
    SETTINGS["schema"] = {**{f"source{i}": {"id": "record_id", "name": "business_name", "address": ["full_address"]}
                             for i in (1, 2, 3)},
                          "labels": {"source1_id": "source1_id", "match_columns": ["matched_ids"]}}

base_config = "configs/low_memory.yaml" if LOW_MEMORY else "config.yaml"
cfg = deep_merge(load_config(base_config), SETTINGS)
cfg["_config_path"] = os.path.abspath(base_config)

# Use the EXACT configuration the training notebook saved for this RUN_NAME
# (the settings above are only a fallback when it does not exist).
if os.path.exists(f"configs/{RUN_NAME}.yaml"):
    cfg = load_config(f"configs/{RUN_NAME}.yaml")
    print(f"loaded configs/{RUN_NAME}.yaml (saved by 01_train.ipynb)")
else:
    print(f"WARNING: configs/{RUN_NAME}.yaml not found - using the settings above; they must match training")
assert os.path.exists(f"models/{RUN_NAME}/pipeline.json"), "train first (01_train.ipynb) with the same RUN_NAME"
print(f"run={cfg['run_name']}  mode={cfg['mode']}  test source1={cfg['data']['test']['source1']}")

## 3. Prediction stages

After a restart: re-run sections 0–2, then continue with the first unfinished stage.

In [ ]:
from src.prediction import PREDICT_STAGES, PREDICT_FUNCS
from src.utils import release_memory
import json, pandas as pd
def run_predict_stage(name):
    out = PREDICT_FUNCS[name](cfg, 'test')
    release_memory()
    return out

### Stage: `prepare`

Load and normalise the test Source 1 / 2 / 3 files (no labels exist or are used).

In [ ]:
run_predict_stage("prepare")

### Stage: `blocking`

Hybrid blocking; writes `candidate_pairs.tsv` **before** any model scores a pair.

In [ ]:
run_predict_stage("blocking")

### Stage: `features`

Pairwise features streamed to the disk-backed matrix.

In [ ]:
run_predict_stage("features")

### Stage: `score`

LightGBM (block-wise) → calibration → [cascade → cross-encoder] → blend → graph cleanup → decision rules tuned on calib → `matching_results.tsv` (one row per Source-1 entity, empty for predicted singletons) → format validation.

In [ ]:
report = run_predict_stage("score")
print("validation:", report["validation"])

## 4. Outputs

In [ ]:
out_dir = f"outputs/{RUN_NAME}/test"
res = pd.read_csv(f"{out_dir}/{cfg['submission']['matching_results']}", sep="\t", dtype=str,
                  keep_default_na=False)
print(len(res), "Source-1 rows;", (res.iloc[:, 1:] == "").all(axis=1).sum(), "predicted singletons")
display(res.head(10))
display(pd.read_csv(f"{out_dir}/{cfg['submission']['candidate_pairs']}", sep="\t", nrows=5))

## 5. Submission archive

Contains `matching_results.tsv`, `candidate_pairs.tsv`, the methodology document, the reports and the pipeline code (`src/`, configs, scripts). Also run the organisers' own validation script before uploading.

In [ ]:
from src.submission import package_submission
files = {
    cfg["submission"]["matching_results"]: f"{out_dir}/{cfg['submission']['matching_results']}",
    cfg["submission"]["candidate_pairs"]: f"{out_dir}/{cfg['submission']['candidate_pairs']}",
    "METHODOLOGY.md": "METHODOLOGY.md",
}
for name, path in [("reports/train_training_report.json", f"outputs/{RUN_NAME}/train/training_report.json"),
                   ("reports/train_blocking_report.json", f"outputs/{RUN_NAME}/train/blocking_report.json"),
                   ("reports/test_prediction_report.json", f"{out_dir}/prediction_report.json")]:
    if os.path.exists(path):
        files[name] = path
archive = f"outputs/{RUN_NAME}/{cfg['submission']['archive_name']}"
res_zip = package_submission(".", files, archive)
print(archive, len(res_zip["files"]), "files")
# Colab: from google.colab import files as colab_files; colab_files.download(archive)